# Generative-AI utilities for cross-domain enrichment & authority linking

**Deliverable D4.1 - Provision of software code (University of Jena)**

This notebook bundles the generative-AI scripts used to enrich, classify,
geolocalise and link cultural-heritage 3D-object metadata (Objaverse / Europeana
records) and to map them to external authority data (Wikidata, OSM, GeoNames).

Pipeline stages:

1. **Description text modifications** - language detection (fastText), translation
   to English (NLLB-200), summarisation (DistilBART), short-description generation (Qwen).
2. **Classification** - by text (Qwen + Europeana vocabulary) and by image
   (YOLOv5, CLIP, Qwen-VL).
3. **Geolocalisation** - from images (OrienterNet, PLONK, GeoCLIP) and from text
   (DeepSeek-R1, Qwen, Llama 3.2) with resolution via GeoPy / Google Geocoding.
4. **Authority mapping** - Wikidata, OpenStreetMap and GeoNames id retrieval.
5. **Reports** - CSV/JSON exports, distance/quality evaluation, processing-time stats.

### Configuration & secrets

All credentials are read from environment variables (see `.env.example`).
**No API keys are stored in this notebook.** Run the *Configuration* cell below
once per kernel session before executing any stage.

> The keys previously hard-coded in this notebook are exposed in version history
> and must be rotated/revoked.

### Prerequisites

Requires an Objaverse SQLite database with Zenodo records. Build it first via
`objaverse1_metadata.py` then `objaverse1_zenodo.py`.


In [ ]:
# === Configuration ===========================================================
# Central configuration. Credentials come from environment variables so that no
# secret is committed to source control. Copy .env.example to .env and fill it
# in, or export the variables in your shell before launching Jupyter.
import os
from pathlib import Path

# Root of the local working tree (databases, reports, data). Override via env.
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", str(Path.cwd()))
# Directory holding local Keras models used by some image classifiers.
KERAS_DIR = os.environ.get("KERAS_DIR", os.path.join(PROJECT_ROOT, "models", "keras"))
# Default SQLite database (per-stage cells may override DATABASE locally).
DATABASE = os.environ.get(
    "OBJAVERSE_DB", os.path.join(PROJECT_ROOT, "Tutorials", "objaverse.sqlite3.db")
)

# Required credentials (read lazily by the cells that need them):
#   DEEPSEEK_API_KEY     - DeepSeek inference API
#   GOOGLE_MAPS_API_KEY  - Google Geocoding / Maps
#   HUGGINGFACE_TOKEN    - Hugging Face Hub (gated models)
#   GEONAMES_USERNAME    - GeoNames web service username
for _required in ("DEEPSEEK_API_KEY", "GOOGLE_MAPS_API_KEY",
                  "HUGGINGFACE_TOKEN", "GEONAMES_USERNAME"):
    if not os.environ.get(_required):
        print(f"[config] warning: environment variable {_required} is not set")



# This is a set of utilities to classify and generate image / texts

### NER
* Spacy for NER: https://spacy.io

### Image description
* GPT4All as CLM: https://gpt4all.io/index.html
* LLAMA 3.0 as CLM: https://huggingface.co/meta-llama/Meta-Llama-3-8B

### Image Classification
* CLIP / ViT-B/32 for image classification: https://docs.vultr.com/zero-shot-image-classification-using-openai-clip; 
    pip install git+https://github.com/openai/CLIP.git scikit-image matplotlib

### Zeroshot version uses the CIFAR-100 dataset of 100 pretrained classes: https://www.cs.toronto.edu/~kriz/cifar.html    
* pix2struct for image description: https://huggingface.co/docs/transformers/main/en/model_doc/pix2struct
* Mistral7B for image description: https://huggingface.co/docs/transformers/main/en/model_doc/pix2struct

### Translator
* Translation via Transformers + https://huggingface.co/facebook/nllb-200-distilled-1.3B

## Usage
Requirements: Needs a objaverse1.0 Sqlite database with Zenodo records created:
    Run: 
    1. objaverse1_metadata.py
    2. objaverse1_zenodo.py

### Data processing
* clip_query(zenodo_image_url:str)
* clip_zeroshot_query(zenodo_image_url:str)
* pix2struct_query(zenodo_image_url): zenodo_image_url:str #Loads image for the Zenodo record in "/files/thumb0.jpeg/content"
* llama_query (name:str, description:str)
* gpt4all_query (name:str, description:str)
* ner_query (name:str, description:str) name: string with the item name; description: description of the item
* nllb_translation("eng_Latn", 'deu_Latn', description, 'nllb-distilled-1.3B') #input language, #output language, #string to be translated, # # LLM model
* openllama_query(input: str, max_new_tokens:int)
* tts(text:str, speaker_wav=str, language=str, file_path=str) #text=output['result'], speaker_wav="test.wav", language="en", file_path="egypt_2.wav"
### Limitations
* Outputs are currently only written to logs


In [ ]:
# =============================================================================
# STEP: Setup: dependencies & shared utilities
# Installs required Python packages and imports the project helper module (generative_ai), which exposes the NER, translation, image-captioning, classification and text-to-speech functions reused below.
# =============================================================================

%pip install protobuf
%pip install sentencepiece
%pip install matplotlib
%pip install TTS
%pip install fasttext

#%pip install -U spacy
#%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.0.0/en_core_web_sm-3.0.0-py3-none-any.whl


# 1 Description text modifications
## 1.1 Detect the language
https://huggingface.co/facebook/fasttext-language-identification

In [ ]:
# =============================================================================
# STEP: 1.1 Detect description language
# Runs fastText language identification on each object description and stores the detected language code and confidence.
# reads description -> writes aiDescriptionLang | Model: facebook/fasttext-language-identification | DB: objaverse.sqlite3.db
# =============================================================================

import os
import json
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from generative_ai import nllb_translation
import fasttext
from huggingface_hub import hf_hub_download
#%pip install 

dbvalueread="description"#"description_en"
dbvaluewrite="aiDescriptionLang"

import time
import re
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

def main():
    DATABASE = 'Tutorials/objaverse.sqlite3.db'
    retrievalquery = (
        "SELECT uid, \"" + dbvalueread + "\"  "
        "FROM object "
        "WHERE \"" + dbvalueread + "\" IS NOT NULL AND \"" + dbvaluewrite + "\" IS NOT NULL"
    )
    
    db = sqlite3.connect(DATABASE, timeout=10)
    cur = db.cursor()
    cura = db.cursor()
    updatequery = str()

    #Counts the number of processed items 
    cur.execute("select count(*) from object WHERE "+dbvaluewrite+" IS NOT NULL;")
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute("select count(*) from object WHERE "+dbvaluewrite+" IS NULL AND "+dbvalueread+" IS NOT NULL;")
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid, description in cur.execute(retrievalquery):
        description = description.replace('\n', ' ').replace('\r', '') #remove breaks
        checkquery = 'SELECT * FROM object WHERE "uid" = "'+uid+'"'
        cura.execute(checkquery)

        datajson=[]
        
        start_time = time.time()
        model_path = hf_hub_download(repo_id="facebook/fasttext-language-identification", filename="model.bin")
        model = fasttext.load_model(model_path)
        end_time = time.time()
        # Process the result and encode it as JSON
        result = model.predict(description)
        data_entry = {
            "processor": "NLLB",
            "model": "facebook/fasttext-language-identification",
            "inferencetime": end_time - start_time,
            "result": {
                "label": result[0][0],  # Extract the label
                "confidence": result[1][0].tolist()  # Extract confidence and convert to list
            }
        }

        datajson.append(data_entry)

        # Convert the entire datajson list to a JSON string
        datajson_str = json.dumps(datajson, indent=4)
        print(datajson_str)
        
        updatequery='UPDATE object SET "mode" = 1, "'+dbvaluewrite+'"="'+str(datajson)+'" WHERE "uid" = "'+uid+'"'
        cura.execute(str(updatequery))
        checkquery = 'SELECT * FROM object WHERE "uid" = "'+uid+'"'
        cura.execute(checkquery)
        print(cura.fetchall())
        db.commit()
        
        continue
    db.close()
    

main()
        


## 1.2 Translate object descriptions
For numerous datasets title and descriptions are available only in other languages than English. For further text processing a first step is required to translate all texts in other languages into English. The second script translates the descriptions from any language into English. The translation utilizes Huggingfaces transformers and the 'nllb-distilled-1.3B’ model. 

Translation via Transformers + https://huggingface.co/facebook/nllb-200-distilled-1.3B
Usage:
* nllb_translation("eng_Latn", 'deu_Latn', description, 'nllb-distilled-1.3B') #input language, #output language, #string to be translated, # # LLM model


In [ ]:
# =============================================================================
# STEP: 1.2 Translate place labels to English
# Translates non-English edmPlaceLabel values into English with NLLB-200 so downstream geocoding runs on English text.
# reads edmPlaceLabel -> writes edmPlaceLabelEn | Model: facebook/nllb-200-distilled-1.3B | DB: objaverse.sqlite3.db
# =============================================================================

import os
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from generative_ai import nllb_translation

DATABASE = 'Tutorials/objaverse.sqlite3.db'
dbvalueread='edmPlaceLabel'#"description"
dbvaluewrite='edmPlaceLabelEn' #"aiDescriptionEn"
dbvaluecheck='edmPlaceLabel' #"aiDescriptionLang"
dbvaluecheckstring='$' #"label__eng_Latn"
uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' IS NULL AND '+dbvaluecheck+' NOT LIKE "%'+dbvaluecheckstring+'%"') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' IS NOT NULL AND '+dbvaluecheck+';')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' IS NULL AND '+dbvaluecheck+' NOT LIKE "%'+dbvaluecheckstring+'%";')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for i in uidlist:
        description = i[1].replace('\n', ' ').replace('\r', '') #remove breaks
        j=j+1
        try:
            datajson='['
            if description != "":
                datajson+=str(nllb_translation("", 'eng_Latn', description, 'nllb-distilled-1.3B')) 
            datajson+=']'

            updatequery='UPDATE object SET '+dbvaluewrite+'="'+datajson.replace('"','')+'" WHERE "uid" = "'+i[0]+'"'
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
            cura.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()       
        except:
            print(f"Processing {i[0]} loading error occurred")
        
        continue
    

main()


## 1.3 Text summarization
Current descriptions highly varies for most models in length, structure and content. For summarizing texts we do use a BART based model (distilbart-cnn-12-6). 

In [ ]:
# =============================================================================
# STEP: 1.3 Summarise into a short description (DistilBART) - current version
# Generates a concise short description from the object name/description using DistilBART.
# reads name -> writes aiDescriptionShort | Model: sshleifer/distilbart-cnn-12-6 | DB: objaverse.sqlite32.db
# =============================================================================

# Text SUmmarization with DistilBART - NEW VERSION
import os
import sqlite3
from PIL import Image
import requests
import json
import re

import time
import numpy as np
from urllib.parse import urlparse
#from generative_ai import ner_query
from transformers import pipeline

from pathlib import Path

#from mlx_lm import load, generate
modelname = "sshleifer/distilbart-cnn-12-6"
#model, tokenizer = load("Qwen/Qwen3-32B-MLX-4bit")   
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="name"
dbvaluewrite="aiDescriptionShort"
dbvaluecheck='source LIKE "%Europeana%"'

DATABASE = 'Tutorials/objaverse.sqlite32.db'

idlist=[]

def main():
    retrievalquery = ('SELECT id, '+dbvalueread+', '+dbvaluewrite + ', aiDescriptionEn '
             'FROM object '
             'WHERE '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND source LIKE "%Europeana%"') #NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
 
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for id in cur.execute(retrievalquery):
        idlist.append(id)
    db.close()

    for i in idlist:
        j=j+1
        if j>-1:
                
            target_id = i[0]
            previousdata=i[2]    
            name=i[1]
            try:
                description_data = json.loads(i[3]) if i[3] else {}
            except json.JSONDecodeError:
                description_data = {}
            description = str({
                'result': description_data.get('result', '')
            }).replace('{\'result\': ', '').replace('}', '')
            description=name+" . "+description
            start_time = time.time()
            classifier = pipeline(
            "summarization",
            model="sshleifer/distilbart-cnn-12-6",
            device="mps",
            min_length=48,
            max_length=224
            )
            print(description)
            output=classifier(description)
            result = re.sub(r'[@#$%^&*\'\\"\{\}]', '', str(output[0])) #to eliminate special chars
            print(result)
            end_time = time.time()
            datajson= str({"processor": "distilbert_summarizer-100",
                    "inferencetime": str(end_time - start_time),
                    "result": result}) #re.sub(r'@#$%^&*\'\"]', '',) to eliminate special chars
     
            print(datajson)

            try:        
                updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(datajson).replace('"',"'")+str(previousdata).replace('"',"'")+'" WHERE "id" = "'+i[0]+'"' #datajson.replace('"','')+
                print(updatequery)
                db = sqlite3.connect(DATABASE)
                cura = db.cursor()
                cura.execute(str(updatequery))
                db.commit()
                checkquery = 'SELECT * FROM object WHERE "id" = "'+i[0]+'"'
                cura.execute(checkquery)
                print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
                db.close()
            except:
                print("Error.")    
            continue
main()    


### Write short description with QWEN 32B

In [ ]:
# =============================================================================
# STEP: 1.4 Short description via Qwen3-32B
# Alternative short-description generator using the Qwen3-32B LLM (Apple MLX) for higher-quality summaries.
# reads name -> writes aiDescriptionShort | Model: Qwen/Qwen3-32B-MLX-4bit | DB: objaverse.sqlite32.db
# =============================================================================

# Text Classification via QWEN+Europeana
import os
import sqlite3
from PIL import Image
import requests
import json

import time
import numpy as np
from urllib.parse import urlparse

from pathlib import Path

from mlx_lm import load, generate
modelname = "Qwen/Qwen3-32B-MLX-4bit"
model, tokenizer = load("Qwen/Qwen3-32B-MLX-4bit")   
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="name"
dbvaluewrite="aiDescriptionShort"
dbvaluecheck=''#'AND source LIKE "%Europeana%"'

DATABASE = 'Tutorials/objaverse.sqlite32.db'

idlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ', aiDescriptionEn '
             'FROM object '
             'WHERE '+dbvaluewrite +' IS NULL AND source LIKE "%%"') #NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
 
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvaluewrite+' LIKE "%'+modelname+'%" '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for id in cur.execute(retrievalquery):
        idlist.append(id)
    db.close()

    for i in idlist:
        j=j+1
        if j>-1:
            print(i)
            target_id = i[0]
            previousdata=i[2]    
            name=i[1]
            try:
                description_data = json.loads(i[3]) if i[3] else {}
            except json.JSONDecodeError:
                description_data = {}
            description = {
                "result": description_data.get("result", "")
            }
            start_time = time.time()
            try:
                prompt = f"Create an English description of 200 characters of the described item from the given name and decription text. Do not invent anything. Input string: {name}; Description: {description}"
                messages = [{"role": "user", "content": prompt}]
                prompt = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, enable_thinking=False 
                )
                output = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=224)
                output = ((output.replace("json", "")).replace("```", "")).replace("...", "")
                print(output)
            
            
            except:
                output=""
            datajson=str( {"processor": "Name_Classification",
                    "model": modelname,
                    "inferencetime": time.time() - start_time,
                    "result":output})
        
            try:  
                print(datajson)      
                if previousdata is None:
                    previousdata = ""
                else:
                    previousdata = str(previousdata)
                print (i[0])
                updatequery = 'UPDATE object SET ' + dbvaluewrite + '="' + str(datajson).replace('"', "'") + previousdata.replace('"', "'") + '" WHERE "uid" = "' + i[0] + '"'
                print(previousdata)
                print(updatequery)
                db = sqlite3.connect(DATABASE)
                cura = db.cursor()
                cura.execute(str(updatequery))
                db.commit()
                checkquery = 'SELECT * FROM object WHERE "id" = "'+i[0]+'"'
                cura.execute(checkquery)
                print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
                db.close()
            except:
                print("Error.")    
            continue

main()


# 2. Classification

## 2.1 Classification via texts

In [ ]:
# =============================================================================
# STEP: 2.1 Zero-shot text classification (DeBERTa)
# Classifies objects into Europeana concept categories from their English description using zero-shot DeBERTa / mDeBERTa NLI models.
# reads aiDescriptionEn -> writes aiConceptPrefLabel | Model: MoritzLaurer DeBERTa zero-shot | DB: objaverse.sqlite3.db
# =============================================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import os
import sqlite3
#from PIL import Image
import requests

#import clip
import time
import numpy as np
import matplotlib.pyplot as plt
import re
from operator import itemgetter
from pathlib import Path

modelname="MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
#modelname = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="aiDescriptionEn"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck='source LIKE "%Europeana%"'

DATABASE = 'Tutorials/objaverse.sqlite3.db'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    tokenizer = AutoTokenizer.from_pretrained(modelname)
    model = AutoModelForSequenceClassification.from_pretrained(modelname)
    for i in uidlist:
       
        try:
            premise = (i[1].split("result':")[1]) 
            previousdata = i[2]
            j=j+1
            datajson='['        
            start_time = time.time()

            list=[]
            for k in europeanacategories:
            
                input = tokenizer(premise, k, truncation=False, return_tensors="pt")
                output = model(input["input_ids"].to(device))  # device = "cuda:0" or "cpu"
                prediction = torch.softmax(output["logits"][0], -1).tolist()
                item=(k,(round(float(prediction[0]*100),1)))
                list.append(item)
            sorted_list=sorted(list, reverse=True, key=itemgetter(1))
          
            datajson+=str( {"processor": "Description_Classification",
                    "model": modelname,
                    "inferencetime": time.time() - start_time,
                    "result":sorted_list[:5]})
            datajson+=']'

            updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(previousdata)+datajson.replace('"','')+'" WHERE "uid" = "'+i[0]+'"'
            
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
            cura.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()
        except:
            print("Error.")    
        continue
main()


## 2.2 Categorization from keywords

In [ ]:
# =============================================================================
# STEP: 2.2 Categorise from existing keywords
# Maps existing language-aware concept keywords to Europeana categories with the same zero-shot DeBERTa classifier.
# reads edmConceptPrefLabelLangAware -> writes aiConceptPrefLabel | Model: MoritzLaurer DeBERTa zero-shot | DB: objaverse.sqlite3.db
# =============================================================================

# Categorization from keywords
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import os
import sqlite3
#from PIL import Image
import requests

#import clip
import time
import numpy as np
import matplotlib.pyplot as plt
import re
from operator import itemgetter
from pathlib import Path

modelname="MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
#modelname = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="edmConceptPrefLabelLangAware"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck='source LIKE "%Europeana%"'

DATABASE = 'Tutorials/objaverse.sqlite3.db'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%Concept_Classification%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    tokenizer = AutoTokenizer.from_pretrained(modelname)
    model = AutoModelForSequenceClassification.from_pretrained(modelname)
    for i in uidlist:
       
        try:
            premise = (i[1]) 
            previousdata = i[2]
            j=j+1
            datajson='['        
            start_time = time.time()

            list=[]
            for k in europeanacategories:
            
                input = tokenizer(premise, k, truncation=False, return_tensors="pt")
                output = model(input["input_ids"].to(device))  # device = "cuda:0" or "cpu"
                prediction = torch.softmax(output["logits"][0], -1).tolist()
                item=(k,(round(float(prediction[0]*100),1)))
                list.append(item)
            sorted_list=sorted(list, reverse=True, key=itemgetter(1))
            
            datajson+=str( {"processor": "Concept_Classification",
                    "model": modelname,
                    "inferencetime": time.time() - start_time,
                    "result":sorted_list[:5]})
            datajson+=']'

            updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(previousdata)+datajson.replace('"','')+'" WHERE "uid" = "'+i[0]+'"'
            
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
            cura.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()
        except:
            print("Error.")    
        continue
main()


### Text Classification via QWEN 3.0-32B + Europeana

In [ ]:
# =============================================================================
# STEP: 2.1b Text classification via Qwen3-32B + Europeana
# LLM-based subject classification constrained to the Europeana vocabulary using Qwen3-32B.
# reads name -> writes aiConceptPrefLabel | Model: Qwen/Qwen3-32B-MLX-4bit | DB: objaverse.sqlite32.db
# =============================================================================

# Text Classification via QWEN+Europeana
import os
import sqlite3
from PIL import Image
import requests
import json

import time
import numpy as np
from urllib.parse import urlparse

from pathlib import Path

from mlx_lm import load, generate
modelname = "Qwen/Qwen3-32B-MLX-4bit"
model, tokenizer = load("Qwen/Qwen3-32B-MLX-4bit")   
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="name"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck='source LIKE "%Europeana%"'

DATABASE = 'Tutorials/objaverse.sqlite32.db'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]

idlist=[]

def main():
    retrievalquery = ('SELECT id, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
 
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for id in cur.execute(retrievalquery):
        idlist.append(id)
    db.close()

    for i in idlist:
        j=j+1
        if j>-1:
            try:
                
                target_id = i[0]
                previousdata=i[2]    
                name=i[1]
                start_time = time.time()
                try:
                    prompt = f"Select exactly five labels from the provided category list that best describe the quote. Use only labels from the category list. Return exactly five labels. Separate labels using commas only. Do not add explanations, numbering, or extra text. Input string: {name}; Category list:"+str(europeanacategories)
                    messages = [{"role": "user", "content": prompt}]
                    prompt = tokenizer.apply_chat_template(
                        messages, add_generation_prompt=True, enable_thinking=False 
                    )
                    output = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=128)
                    output = ((output.replace("json", "")).replace("```", "")).replace("...", "")
                    print(output)
                
                #{'processor': 'Name_Classification', 'model': 'mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit', 'inferencetime': 5.1980979442596436, 'result': 'Tool, Machinery, Woodwork, Mechanical, Historical'}

                except:
                    output=""
                datajson=str( {"processor": "Name_Classification",
                        "model": modelname,
                        "inferencetime": time.time() - start_time,
                        "result":output})
            
                    
                updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(datajson)+str(previousdata).replace('"',"'")+'" WHERE "id" = "'+i[0]+'"' #datajson.replace('"','')+
                print(updatequery)
                db = sqlite3.connect(DATABASE)
                cura = db.cursor()
                cura.execute(str(updatequery))
                db.commit()
                checkquery = 'SELECT * FROM object WHERE "id" = "'+i[0]+'"'
                cura.execute(checkquery)
                print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
                db.close()
            except:
                print("Error.")    
            continue

main()


### Text Classification via QWEN 14B +Objaverse

In [ ]:
# =============================================================================
# STEP: 2.1c Text classification via Qwen3-14B (Objaverse)
# LLM subject classification for Objaverse-sourced records using the Qwen3-14B description-tuned model.
# reads aiDescriptionEn -> writes aiConceptPrefLabel | Model: Qwen/Qwen3-14B-MLX-4bit-description | DB: objaverse.sqlite32.db
# =============================================================================

# Text Classification via QWEN+Europeana
import os
import sqlite3
from PIL import Image
import requests
import json

import time
import numpy as np
from urllib.parse import urlparse

from pathlib import Path

from mlx_lm import load, generate
modelname = "Qwen/Qwen3-14B-MLX-4bit-description"
model, tokenizer = load("Qwen/Qwen3-14B-MLX-4bit")   
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="aiDescriptionEn"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck='source LIKE "%Objaverse%"' #source LIKE "%Europeana%"'

DATABASE = 'Tutorials/objaverse.sqlite32.db'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]

idlist=[]

def main():
    retrievalquery = ('SELECT id, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ' )#"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 10 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
 
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for id in cur.execute(retrievalquery):
        idlist.append(id)
    db.close()

    for i in idlist:
        j=j+1
        if j>-1:
                
            target_id = i[0]
            previousdata=i[2]    
            name=i[1]
            start_time = time.time()
            try:
                prompt = f"Start your answer with the most likely five categories within [] brackets. Do not invent additional categories. For string '{name}' name the top five categories from the following categories: {str(europeanacategories)}"
                messages = [{"role": "user", "content": prompt}]
                prompt = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, enable_thinking=False
                )
                output = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=64)
                output = ((output.replace("json", "")).replace("```", "")).replace("...", "")
                output_list = output.split(", ")

                # Nur die Kategorien aus europeanacategories beibehalten
                filtered_output = [category.strip() for category in output_list if category.strip() in europeanacategories]

                #if len(filtered_output) < 5:
                #    remaining_categories = [cat for cat in europeanacategories if cat not in filtered_output]
                #    prompt = f"Select exactly {str(5 - len(filtered_output))} from the provided category list that best describe the quote. Do not use the terms {filtered_output}. Input string: {name}; Category list: {str(remaining_categories)}"
                #    messages = [{"role": "user", "content": prompt}]
                #    prompt = tokenizer.apply_chat_template(
                #        messages, add_generation_prompt=True, enable_thinking=False
                #    )
                #    output = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=64)
                #    found_categories = [
                #        category.strip()
                #        for category in europeanacategories
                #        if category.strip() in output
                #    ]
                    # Gefundene Kategorien als durch Kommas getrennte Werte
                    #filtered_categories2 = ", ".join(found_categories)
                #    filtered_output.extend(found_categories)

                # Ergebnis als kommaseparierten String formatieren
                output = ", ".join(filtered_output)
                print('Output is: ' + output)
            except:
                output=""
            datajson=str( {"processor": "Description_Classification",
                    "model": modelname,
                    "inferencetime": time.time() - start_time,
                    "result":output})
            datajson = json.dumps(datajson, indent=4)
                
            updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(datajson).replace('"',"'")+str(previousdata).replace('"',"'")+'" WHERE "id" = "'+i[0]+'"' #datajson.replace('"','')+
            print(updatequery)
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            
            db.commit()
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()

main()


## 2.3 Classification via images
This script classifies objects via images.

In [ ]:
# =============================================================================
# STEP: 2.3 Image classification via CLIP
# Zero-shot image classification of object thumbnails with OpenAI CLIP (ViT-B/32).
# Model: CLIP ViT-B/32 | DB: objaverse.sqlite3.db
# =============================================================================

import os
import sqlite3
from PIL import Image
import requests

import clip
import time
import numpy as np
import matplotlib.pyplot as plt
import re

from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from generative_ai import clip_query,clip_zeroshot_query_CIFAR10,clip_zeroshot_query_CIFAR100,llavanext_keyword_query

def main():
    DATABASE = 'Tutorials/objaverse.sqlite3.db'
    retrievalquery = ("SELECT uid, uri, name, description, zenodo_url, edmPreview "
             "FROM object "
             "WHERE zenodo_url IS NULL AND mode IS NOT 1 ") #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    cura = db.cursor()
    updatequery = str()

    #Counts the number of processed items 
    cur.execute("select count(*) from object WHERE mode IS 1;")
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute("select count(*) from object WHERE mode IS NOT 1;")
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid, uri, name, description, zenodo_url, edmPreview in cur.execute(retrievalquery):
        #This is to compile and clean strings for images and descriptions
        zenodo_image_url = edmPreview
        #zenodo_image_url = (zenodo_url.replace('deposit/depositions','records'))+'/files/thumb0.jpeg/content'
        description = description.replace('\n', ' ').replace('\r', '') #remove breaks

        checkquery = 'SELECT * FROM object WHERE "uid" = "'+uid+'"'
        cura.execute(checkquery)
        #print ('Description input string: '+description)

        try:
            print('Image input string: '+zenodo_image_url)
            #plt.figure(figsize=(5, 5))
            #image = Image.open(requests.get(zenodo_image_url, stream=True).raw).convert("RGB")
            #plt.imshow(image)
            #plt.axis("off")

            datajson='['
            
            datajson+=str(llavanext_keyword_query (zenodo_image_url))+','
            datajson+=']'
        
            updatequery='UPDATE object SET "mode" = 1, "categories_ki"="'+datajson.replace('"','')+'" WHERE "uid" = "'+uid+'"'
            cura.execute(str(updatequery))
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+uid+'"'
            cura.execute(checkquery)
            print(cura.fetchall())
            db.commit()
        except:
            print(f"preview image {zenodo_image_url} loading error occurred")
        
        continue
    db.close()

main()


In [ ]:
# =============================================================================
# STEP: 2.3b Object-detection demo via YOLOv5
# Exploratory single-image object detection with YOLOv5 (ultralytics) and a visual preview.
# Model: YOLOv5
# =============================================================================

# Image classifier via Yolo V5
# Installiere die YOLOv5-Bibliothek (falls noch nicht installiert)
# !pip install ultralytics

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# Lade ein vortrainiertes YOLOv5-Modell (z. B. YOLOv5s - small version)
model = YOLO('yolov5s.pt')  # Du kannst auch 'yolov5m.pt', 'yolov5l.pt', etc. verwenden

# Lade ein Bild
image_path = 'path_to_your_image.jpg'  # Pfad zu deinem Bild
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Führe die Objekterkennung durch
results = model(image)

# Zeige die Ergebnisse an
results.show()

# Optional: Speichere das Bild mit den Ergebnissen
results.save(save_dir='runs/detect/exp')  # Speichert die Ergebnisse im Ordner 'runs/detect/exp'

# Visualisiere das Bild mit matplotlib
plt.imshow(image)
plt.axis('off')
plt.show()


In [ ]:
# =============================================================================
# STEP: 2.3c Image classification via CLIP + Europeana
# Classifies object preview images into Europeana categories with CLIP.
# reads edmPreview -> writes aiConceptPrefLabel | Model: CLIP ViT-B/32 | DB: objaverse.sqlite3.db
# =============================================================================

# Image Classification via CLIP+Europeana
import os
import sqlite3
from PIL import Image
import requests

import clip
import time
import numpy as np
import matplotlib.pyplot as plt
import re

from pathlib import Path
import torch

modelname = "CLIP-ViT-B/32"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="edmPreview"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck='source LIKE "%Europeana%"'

DATABASE = 'Tutorials/objaverse.sqlite3.db'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' IS NULL AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    #from torchvision.datasets import CIFAR100
    #cifar100 = CIFAR100(root=os.path.expanduser("~/.cache"), download=True, train=False)
    device = "cuda" if torch.cuda.is_available() else "mps"
    model, preprocess = clip.load('ViT-B/32', device)

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for i in uidlist:
        try:
            image_url = i[1].replace("['", "").replace("']", "") #remove breaks
            previousdata = ""#i[2]
            j=j+1
  
            start_time = time.time()

            image = Image.open(requests.get(image_url, stream=True).raw)
            image_input = preprocess(image).unsqueeze(0).to(device)
            text_inputs = torch.cat([clip.tokenize(f"a photo of a {c}") for c in europeanacategories]).to(device)

            with torch.no_grad():
                image_features = model.encode_image(image_input)
                text_features = model.encode_text(text_inputs)

            # Pick the top 5 most similar labels for the image
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            values, indices = similarity[0].topk(5)

            output= "\nTop predictions:\n"
            for value, index in zip(values, indices):
                output +=(f"{europeanacategories[index]:>16s}: {100 * value.item():.2f}% ")
            print(output)

            datajson=str( {"processor": "Image_Classification",
                    "model": modelname,
                    "inferencetime": time.time() - start_time,
                    "result":output})
   
                
            updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(previousdata)+datajson.replace('"','')+'" WHERE "uid" = "'+i[0]+'"'
            
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
            cura.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()
        except:
            print("Error.")    
        continue

main()


### Image Classification via YOLO v5

In [ ]:
# =============================================================================
# STEP: 2.3d Batch object detection via YOLOv5
# Runs YOLOv5 over the local thumbnail directory for batch detection.
# Model: yolov5s | Input: Data/Images/thumbnail
# =============================================================================

import os
from ultralytics import YOLO

# Lade das YOLO-Modell
model = YOLO('yolov5s.pt')  # Verwende ein vortrainiertes YOLO-Modell

# Verzeichnis mit Bildern
image_dir = os.path.join(PROJECT_ROOT, "Data/Images/thumbnail")

# Iteriere durch alle Dateien im Verzeichnis
for filename in os.listdir(image_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):  # Prüfe, ob es sich um ein Bild handelt
        image_path = os.path.join(image_dir, filename)
        print(f"Processing image: {image_path}")
        
        # Führe YOLO-Erkennung auf dem Bild aus
        results = model(image_path)
        
        # Zeige die Ergebnisse an
        print(results)
        #Speichere die Klassen in einer Datei
europeanacategories = [
    'Building',
    'Archaeological Site',
    'Cartoon',
    'Ceramics',
    'Clothing',
    'Costume Accessories',
    'Drawing',
    'Map',
    'Furniture',
    'Textile',
    'Food',
    'Glassware',
    'Inscription',
    'Jewellery',
    'Metalwork',
    'Machinery',
    'Medal',
    'Memorabilia',
    'Mineral',
    'Musical Instrument',
    'Painting',
    'Photograph',
    'Postcard',
    'Poster',
    'Print',
    'Sculpture',
    'Specimen',
    'Tableware',
    'Tool',
    'Tapestry',
    'Toy',
    'Weaponry',
    'Woodwork',
    'Stamp'
]

# Speichere die Klassen in einer Datei
with open('europeanacategories.txt', 'w') as f:
    for category in europeanacategories:
        f.write(category.strip() + '\n')  # Entferne führende/trailing Leerzeichen und schreibe in die Datei


### Image Classification via QWEN VIS+Europeana

In [ ]:
# =============================================================================
# STEP: 2.3e Image classification via Qwen-VL + Europeana
# Vision-language classification of preview images into Europeana categories using Qwen3-VL.
# reads edmPreview -> writes aiConceptPrefLabel | Model: Qwen3-VL-30B | DB: objaverse.sqlite32.db
# =============================================================================

# Image Classification via QWEN VIS+Europeana
import os
import sqlite3
from PIL import Image
import requests
import json

import time
import numpy as np
from mlx_vlm import load, generate
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.utils import load_config
from urllib.parse import urlparse

from pathlib import Path

modelname = "mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="edmPreview"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck='source LIKE "%Europeana%"'#

DATABASE = 'Tutorials/objaverse.sqlite32.db'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]
model, processor = load("mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit")
config = load_config("mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit")

idlist=[]

def main():
    retrievalquery = ('SELECT id, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for id in cur.execute(retrievalquery):
        idlist.append(id)
    db.close()

    for i in idlist:
        j=j+1
        if j>1578:
            previousdata2=""
            json_file_path = os.path.join(PROJECT_ROOT, "Data/Database/object.json")
            target_id = i[0]

            # Lade die JSON-Daten
            with open(json_file_path, 'r', encoding='utf-8') as file:
                data = json.load(file)
            previousdata2 = previousdata2 or ""
            
            # Suche nach der entsprechenden ID und extrahiere das Label
            previousdata = ""
            for item in data:
                if item.get("id") == target_id:  # Ersetze "ID" durch den tatsächlichen Schlüssel für die ID
                    previousdata2 = item.get("aiConceptPrefLabel")
                    break

            # Überprüfe das Ergebnis
            if previousdata2:
                print(f"Gefundenes aiConceptPrefLabel: {previousdata2}")
            else:
                print(f"Keine Daten für ID {target_id} gefunden.")

            previousdata = i[2]
            previousdata = previousdata or ""
            print(previousdata)
            datajson=''        
            start_time = time.time()
            try:
                # Bereinige und überprüfe die URL
                #image_url = i[1].strip().replace("['", "").replace("']", "")  # Entferne unerwünschte Zeichen
                #parsed_url = urlparse(image_url)
                image_url=os.path.join(PROJECT_ROOT, "Data/Images/thumbnail/")+i[0]+'.jpg'
                image=Image.open(image_url).convert("RGB")
                #image = Image.open(requests.get(parsed_url, stream=True).raw)
                prompt = "Start your answer with the most likely five categories within [] brackets. Name the top five categories from the following categories which describe this image:"+str(europeanacategories)

                # Apply chat template
                formatted_prompt = apply_chat_template(
                    processor, config, prompt, num_images=1
                )

                # Generate output
                output = str(generate(model, processor, formatted_prompt, image))
                start = output.find('[') + 1
                end = output.find(']', start)
                if start > 0 and end > 0:
                    output = output[start:end]
                else:
                    print("Keine eckigen Klammern gefunden.")
                print(output)
            except:
                output=""
            datajson+=str( {"processor": "Image_Classification",
                    "model": modelname,
                    "inferencetime": time.time() - start_time,
                    "result":output})
            
            datajson+=''
                
            updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(previousdata).replace('"',"'")+str(previousdata2).replace('"',"'")+'" WHERE "id" = "'+i[0]+'"' #datajson.replace('"','')+
            print(updatequery)
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "id" = "'+i[0]+'"'
            cura.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()

main()


### Image Classification via QWEN VIS + CLIP for Objaverse

In [ ]:
# =============================================================================
# STEP: 2.3f Image classification via Qwen-VL + CLIP (Objaverse)
# Combined Qwen-VL and CLIP classifier for Objaverse records; defines qwen_vis_classifier / clip_vis_classifier helpers.
# reads uid -> writes aiConceptPrefLabel | Model: Qwen3-VL-30B + CLIP | DB: objaverse.sqlite32.db
# =============================================================================

# Image Classification via QWEN VIS+Objaverse
import os
import sqlite3
from PIL import Image
import requests
import json
import clip

import matplotlib.pyplot as plt
import re

from pathlib import Path
import torch
import time
import numpy as np
#from mlx_vlm import load, generate
#from mlx_vlm.prompt_utils import apply_chat_template
#from mlx_vlm.utils import load_config
from urllib.parse import urlparse

#modelname = "mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit"
modelname = "CLIP-ViT-B/32"

os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="uid"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck='source LIKE "%Objaverse%" AND id is "ac21f76a0e324955b72c66b77b87cdd3"'#source LIKE "%Europeana%"

DATABASE = 'Tutorials/objaverse.sqlite32.db'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]

device = "cuda" if torch.cuda.is_available() else "mps"
model, preprocess = clip.load('ViT-B/32', device)
idlist=[]

def format_previousdata2(data):
    if data:
        parts = data.split(";")
        if len(parts) == 3:
            return {
                "model": parts[0].split("=")[1] if "=" in parts[0] else parts[0],
                "result": parts[1],
                "value": parts[2]
            }
    return {}

def qwen_vis_classifier(model, processor, config, image, europeanacategories):
    """
    Extrahiert die fünf wahrscheinlichsten Kategorien aus einer Liste von Kategorien,
    die ein Bild beschreiben, und gibt sie als String zurück.

    Args:
        model: Das Modell, das für die Generierung verwendet wird.
        processor: Der Prozessor, der für die Verarbeitung verwendet wird.
        config: Die Konfiguration für das Modell.
        image: Das Bild, das analysiert werden soll.
        europeanacategories: Die Liste der verfügbaren Kategorien.

    Returns:
        Ein String mit den extrahierten Kategorien, getrennt durch Kommas.
    """
    try:
        # Prompt erstellen
        prompt = "Start your answer with the most likely five categories within [] brackets. Name the top five categories from the following categories which describe this image:" + str(europeanacategories)

        # Chat-Template anwenden
        formatted_prompt = apply_chat_template(
            processor, config, prompt, num_images=1
        )

        # Ausgabe generieren
        output = str(generate(model, processor, formatted_prompt, image))

        # Kategorien aus den eckigen Klammern extrahieren
        start = output.find('[') + 1
        end = output.find(']', start)
        if start > 0 and end > 0:
            output = output[start:end]
        else:
            print("Keine eckigen Klammern gefunden.")
            output = ""

        return output
    except Exception as e:
        print(f"Ein Fehler ist aufgetreten: {e}")
        return ""

def clip_vis_classifier(model, processor, config, image, europeanacategories):
    """
    Extrahiert die fünf wahrscheinlichsten Kategorien aus einer Liste von Kategorien,
    die ein Bild beschreiben, und gibt sie als String zurück.

    Args:
        model: Das Modell, das für die Generierung verwendet wird.
        processor: Der Prozessor, der für die Verarbeitung verwendet wird.
        config: Die Konfiguration für das Modell.
        image: Das Bild, das analysiert werden soll.
        europeanacategories: Die Liste der verfügbaren Kategorien.

    Returns:
        Ein String mit den extrahierten Kategorien, getrennt durch Kommas.
    """
    try:
        # Prompt erstellen
        prompt = "Start your answer with the most likely five categories within [] brackets. Name the top five categories from the following categories which describe this image:" + str(europeanacategories)

        # Chat-Template anwenden
        formatted_prompt = apply_chat_template(
            processor, config, prompt, num_images=1
        )

        # Ausgabe generieren
        output = str(generate(model, processor, formatted_prompt, image))

        # Kategorien aus den eckigen Klammern extrahieren
        start = output.find('[') + 1
        end = output.find(']', start)
        if start > 0 and end > 0:
            output = output[start:end]
        else:
            print("Keine eckigen Klammern gefunden.")
            output = ""

        return output
    except Exception as e:
        print(f"Ein Fehler ist aufgetreten: {e}")
        return ""

def main():
    retrievalquery = ('SELECT id, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for id in cur.execute(retrievalquery):
        idlist.append(id)
    db.close()

    for i in idlist:
        j=j+1
        if j>0:
            previousdata2=""
            json_file_path = os.path.join(PROJECT_ROOT, "Data/Database/object.json")
            target_id = i[0]

            # Lade die JSON-Daten
            with open(json_file_path, 'r', encoding='utf-8') as file:
                data = json.load(file)
            previousdata2 = previousdata2 or ""
            
            # Suche nach der entsprechenden ID und extrahiere das Label
            previousdata = ""
            for item in data:
                if item.get("id") == target_id:  # Ersetze "ID" durch den tatsächlichen Schlüssel für die ID
                    previousdata2 = item.get("aiConceptPrefLabel")
                    try:
                        formatted_data = format_previousdata2(previousdata2)
                        print(json.dumps(formatted_data, indent=4))
                    except Exception as e:
                        print(f"Fehler beim Verarbeiten von previousdata2: {e}")

                        # Überprüfe das Ergebnis
                        if previousdata2:
                            print(f"Gefundenes aiConceptPrefLabel: {previousdata2}")
                        else:
                            print(f"Keine Daten für ID {target_id} gefunden.")

            previousdata = i[2]
            previousdata = previousdata or ""
          
            start_time = time.time()

            
            #CLIP
            try:
                image_url=os.path.join(PROJECT_ROOT, "Data/Images/thumbnail/")+i[0]+'.jpg'
                image=Image.open(image_url).convert("RGB")

                image_input = preprocess(image).unsqueeze(0).to(device)
                text_inputs = torch.cat([clip.tokenize(f"a photo of a {c}") for c in europeanacategories]).to(device)

                with torch.no_grad():
                    image_features = model.encode_image(image_input)
                    text_features = model.encode_text(text_inputs)

                # Pick the top 5 most similar labels for the image
                image_features /= image_features.norm(dim=-1, keepdim=True)
                text_features /= text_features.norm(dim=-1, keepdim=True)
                similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
                values, indices = similarity[0].topk(5)

                output= ""
                for value, index in zip(values, indices):
                    output+= ','+(f"{europeanacategories[index]:>16s}")
                output=output.replace("  ", " ").replace(' ',' ').replace('  ',' ').replace('  ',' ').replace('  ',' ').lstrip(',').lstrip(' ')#: {100 * value.item():.2f}% )

                print(output)
            except:
                output=""
            
            # modelname = "mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit"
            #model, processor = load("mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit")
            #config = load_config("mlx-community/Qwen3-VL-30B-A3B-Instruct-4bit")
            #qwen_vis_classifier(model, processor, config, image, europeanacategories)
            
            datajson=str( {"processor": "Image_Classification",
                    "model": modelname,
                    "inferencetime": time.time() - start_time,
                    "result":output})

                
            updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(previousdata).replace('"',"'")+str(previousdata2).replace('"',"'")+str(datajson).replace('"',"'")+'" WHERE "id" = "'+i[0]+'"' #datajson.replace('"','')+
            print(updatequery)
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "id" = "'+i[0]+'"'
            cura.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()

main()


# 3. Geolocalization

## 3.1 Geolocalization via images
### 3.1.1 OrienterNet
Image orientation via https://github.com/facebookresearch/OrienterNet

In [ ]:
# =============================================================================
# STEP: 3.1.1 Image geolocalisation (OrienterNet / PLONK / GeoCLIP)
# Estimates object location from its image using OrienterNet, PLONK and GeoCLIP, then reverse-geocodes the coordinates.
# reads aiPlaceLabel -> writes aiPlaceLabel | Model: OrienterNet, PLONK_YFCC, GeoCLIP | DB: objaverse.sqlite3.db
# =============================================================================

from openai import OpenAI
from geopy.geocoders import GoogleV3
import os
import sqlite3
import requests
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from geo_utilities import *
import matplotlib.pyplot as plt
from maploc.demo import Demo
from maploc.osm.viz import GeoPlotter
from maploc.osm.tiling import TileManager
from maploc.osm.viz import Colormap, plot_nodes
from maploc.utils.viz_2d import plot_images
from PIL import Image

import torch
from geoclip import GeoCLIP
from plonk import PlonkPipeline

dbvalueread="aiPlaceLabel"
dbvaluewrite="aiPlaceLabel"
dbvaluecheck='source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL AND aiConceptPrefLabel LIKE "%BUILD%"'
#dbvaluecheck= 'name like "%Vianden%"'
modelnamea="OrienterNet"
modelnameb = "nicolas-dufour/PLONK_YFCC"
modelnamec="GeoCLIP"
DATABASE = 'Tutorials/objaverse.sqlite3.db'

deepseek_key = os.environ["DEEPSEEK_API_KEY"]
google_key = os.environ["GOOGLE_MAPS_API_KEY"]
huggingfaces_key = os.environ["HUGGINGFACE_TOKEN"]

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, edmPreview, zenodo_url, '+dbvalueread+', '+dbvaluewrite + ',name '
             'FROM object '
             'WHERE '+dbvaluecheck+' AND '+dbvaluewrite+' NOT LIKE "%'+modelnameb+'%"') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelnameb+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelnameb+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for i in uidlist:
        print(i)
        uid=i[0]
        print(uid)
        print((i[4].split(':')[4]))
        try:
            
            addressinput = ((i[4].split(':')[3])[:200])
        except:
            addressinput = i[5]
            print(i[5])
        geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
        try:
            address = geolocator.geocode(addressinput).address
        except:
            try:
                addressinput = ((i[4].split(':')[4])[:200])
                address = geolocator.geocode(addressinput).address
            except:
                address = "Northpole"
        print(address)
        if i[1]:
            image_url = i[1].replace("['", "").replace("']", "") #remove breaks
        else:
            image_url = i[2]
        description = i[3].replace('\n', ' ').replace('\r', '') #remove breaks
        previousdata = i[4]
        #url = "https://zenodo.org/api/records/10290891/files/thumb0.jpeg/content"
        response = requests.get(image_url)
        if response.status_code == 200:
            with open('requests.jpg', 'wb') as file:
                file.write(response.content)
        j=j+1

#Plonk
        
        datajson='['        
        start_time = time.time()
        #%export PYTORCH_ENABLE_MPS_FALLBACK=1 
        image = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')
        pipeline = PlonkPipeline(model_path=modelnameb, device="cpu")
        gps_coords = pipeline(image, batch_size=1024)
        

        stringtemp=''
        geolocator = Nominatim(user_agent="4DS")

        coords=(str(gps_coords[0]).replace("[", "").replace("]", "")).replace("   ", "  ").replace("  ",",")
        segment = (geolocator.reverse(coords, language="en").address).split(',')
        length = (len(segment))
        try:
            city= (segment[length-5])
        except:
            city= ("")
        try:
            state= (segment[length-3])
        except:
            state= ("")
        try:
            region= (segment[length-4])
        except:
            region= ("")
        try:
            country= (segment[length-1])
        except:
            country= ("")

        datajson+=str( {"processor": "Image_Based_Location_Retrieval",
                "model": modelnameb+" & geopy.geocoders.GoogleV3",
                "inferencetime": time.time() - start_time,
                "result":str(gps_coords[0]),
                "location":country+';'+state+';'+region+';'+city
                })
        
        datajson+=']'

    #GeoCLIP
    
        model = GeoCLIP()
        image_path = "requests.jpg"
        start_time = time.time()
        top_pred_gps, top_pred_prob = model.predict(image_path, top_k=5)

        for i in range(1):
            lat, lon = top_pred_gps[i]
            print(f"Prediction {i+1}: ({lat:.6f}, {lon:.6f})")
            print(f"Probability: {top_pred_prob[i]:.6f}")
            print("")

            geolocator = Nominatim(user_agent="4DS")

            coords=(top_pred_gps[i])
            segment = (geolocator.reverse(coords, language="en").address).split(',')
            length = (len(segment))
            try:
                city= (segment[length-5])
            except:
                city= ("")
            try:
                state= (segment[length-3])
            except:
                state= ("")
            try:
                region= (segment[length-4])
            except:
                region= ("")
            try:
                country= (segment[length-1])
            except:
                country= ("")
            datajson+='['
            datajson+=str( {"processor": "Image_Based_Location_Retrieval",
                    "model": modelnamec+" & geopy.geocoders.GoogleV3",
                    "inferencetime": time.time() - start_time,
                    "result":str(gps_coords[0]),
                    "location":country+';'+state+';'+region+';'+city
                    })
            
            datajson+=']'
    #Run OrienterNet
        try:
            starttime=time.time()
            demo = Demo(num_rotations=128, device='cpu') # change to "cuda" if you have a GPU.
            tile_size_meters = 128 #@param [64, 128, 256, 512]
            image_path="requests.jpg"
            print(f"Using image {image_path} with location prior '{address}'")

            image, camera, gravity, proj, bbox = demo.read_input_image(
                image_path,
                prior_address=address,
                tile_size_meters=tile_size_meters,
            )

            # Show the query area in an interactive map
            #from maploc.osm.viz import GeoPlotter
            #plot = GeoPlotter(zoom=16)
            #plot.points(proj.latlonalt[:2], "red", name="location prior", size=10)
            #plot.bbox(proj.unproject(bbox), "blue", name="map tile")
            #plot.fig.show()
        

            # Query OpenStreetMap for this area
            tiler = TileManager.from_bbox(proj, bbox + 10, demo.config.data.pixel_per_meter)
            canvas = tiler.query(bbox)

            # Show the inputs to the model: image and raster map
            #from maploc.osm.viz import Colormap, plot_nodes
            #from maploc.utils.viz_2d import plot_images
            map_viz = Colormap.apply(canvas.raster)
            #plot_images([image, map_viz], titles=["input image", "OpenStreetMap raster"])
            #plot_nodes(1, canvas.raster[2], fontsize=6, size=10)

            # @title Localize!
            from maploc.utils.viz_localization import (
                likelihood_overlay,
                plot_dense_rotations,
                add_circle_inset,
            )
            from maploc.utils.viz_2d import features_to_RGB

            # Run the inference
            uv, yaw, prob, neural_map, image_rectified = demo.localize(
                image, camera, canvas, gravity=gravity
            )
        
            coords=(proj.unproject(canvas.to_xy(uv)))
            # Plot as interactive figure
            bbox_ll = proj.unproject(canvas.bbox)
            plot = GeoPlotter(zoom=16.5)
            plot.raster(map_viz, proj.unproject(bbox), opacity=0.5)
            plot.raster(likelihood_overlay(prob.numpy().max(-1)), proj.unproject(bbox))
            plot.points(proj.latlonalt[:2], "red", name="location prior", size=10)
            plot.points(proj.unproject(canvas.to_xy(uv)), "black", name="argmax", size=10)
            plot.bbox(proj.unproject(bbox), "blue", name="map tile")

            print(yaw)
            #plot.fig.show()

            #coords=str(coords)[1:((len(address))-2)].split('    ')
            print(coords)
            stringtemp=''
            geolocator = Nominatim(user_agent="4DS")
            coords=(float(coords[0]),float(coords[1]))
            print(coords)
            segment = (geolocator.reverse(coords, language="en").address).split(',')
            length = (len(segment))
            print(segment)
            try:
                city= (segment[length-5])
            except:
                city= ("")
            try:
                state= (segment[length-3])
            except:
                state= ("")
            try:
                region= (segment[length-4])
            except:
                region= ("")
            try:
                country= (segment[length-1])
            except:
                country= ("")
            datajson+='['    
            datajson+=str({"processor": "Image_Based_Location_Retrieval",
                    "model": modelnamea+" & geopy.geocoders.GoogleV3",
                    "inferencetime": time.time() - starttime,
                    "result":str(coords),
                    "location":country+';'+state+';'+region+';'+city,
                    "yaw":yaw.to(torch.float)
                    })
            
            datajson+=']'
        except:
            print("Orienternet not run")    
            
        print(datajson)  
        print(previousdata)
        
        updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(previousdata)+datajson.replace('"','')+'" WHERE "uid" = "'+str(uid)+'"'
        print(updatequery)
        db = sqlite3.connect(DATABASE)
        cura = db.cursor()
        cura.execute(str(updatequery))
        db.commit()
        checkquery = 'SELECT * FROM object WHERE "uid" = "'+str(uid)+'"'
        cura.execute(checkquery)
        print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
        db.close()
main()


In [ ]:
# =============================================================================
# STEP: 3.1.1b Image geocoding for 4D-browser images
# Variant image-geolocalisation routine for 4D-browser imagery using an LLM plus Google V3 geocoding.
# Model: OrienterNet/PLONK/GeoCLIP + GoogleV3
# =============================================================================

# Geocoding of 4Dbrowser images
from openai import OpenAI
from geopy.geocoders import GoogleV3
import os
import sqlite3
import requests
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from geo_utilities import *
import matplotlib.pyplot as plt
from maploc.demo import Demo
from maploc.osm.viz import GeoPlotter
from maploc.osm.tiling import TileManager
from maploc.osm.viz import Colormap, plot_nodes
from maploc.utils.viz_2d import plot_images
from PIL import Image

import torch
from geoclip import GeoCLIP
from plonk import PlonkPipeline

dbvalueread="aiPlaceLabel"
dbvaluewrite="aiPlaceLabel"
dbvaluecheck='source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL AND aiConceptPrefLabel LIKE "%BUILD%"'
#dbvaluecheck= 'name like "%Vianden%"'
modelnamea="OrienterNet"
modelnameb = "nicolas-dufour/PLONK_YFCC"
modelnamec="GeoCLIP"

DATABASE = 'Tutorials/Image_Datasets/images.db'
DBTABLE = "4dBrowserDataset"

deepseek_key = os.environ["DEEPSEEK_API_KEY"]
google_key = os.environ["GOOGLE_MAPS_API_KEY"]
huggingfaces_key = os.environ["HUGGINGFACE_TOKEN"]

uidlist=[]

# API URL
url = "https://4dbrowser.urbanhistory4d.org/api/images"
params = {
    "lat": 51.049326,#50.927996,
    "lon": 13.738146,#11.587980,
    "r": 1000  # radius in meters
}

# Send request
response = requests.get(url, params=params)

# Raise error if request failed
response.raise_for_status()

# Parse JSON
data = response.json()
j=0

# Extract and print lat, lon, image url
for item in data:
    id=item["id"]
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    cur.execute('select count(*) from "'+DBTABLE+'";')
    numberOfRows = cur.fetchone()[0]
    #print ("Number of datasets already processed: "+str(numberOfRows))
    checkquery = 'SELECT count(*) FROM "'+DBTABLE+'" WHERE "id" = "'+str(id)+'"'
    #print (checkquery)
    cur.execute(checkquery)
    numberOfRows = cur.fetchone()[0]
    db.close()
    j=+j
    if (numberOfRows==0):
        lat = item["camera"]["latitude"]
        lon = item["camera"]["longitude"]
        image_url = "https://4dbrowser.urbanhistory4d.org/data/"+item["file"]["path"]+item["file"]["original"]
        print(image_url)
        geolocator = Nominatim(user_agent="4DS")
        coords=[params["lat"], params["lon"]]
        print(coords)
        addressinput = (geolocator.reverse(coords, language="en").address).split(",")
        address=addressinput [5]
        
        print(f"Latitude: {lat}, Longitude: {lon}, Image URL: {image_url}")

        #image_url = "https://zenodo.org/api/records/10290891/files/thumb0.jpeg/content"
        response = requests.get(image_url)
        if response.status_code == 200:
            with open('requests.jpg', 'wb') as file:
                file.write(response.content)

        print ("##### Run Plonk")
        
        datajson='['        
        start_time = time.time()
        #%export PYTORCH_ENABLE_MPS_FALLBACK=1 
        image = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')
        pipeline = PlonkPipeline(model_path=modelnameb, device="cpu")
        gps_coords = pipeline(image, batch_size=1024)
        

        stringtemp=''
        geolocator = Nominatim(user_agent="4DS")

        coords=(str(gps_coords[0]).replace("[", "").replace("]", "")).replace("   ", "  ").replace("  ",",")
        t=0
        while (t==0):
            try: 
                segment = (geolocator.reverse(coords, language="en").address).split(',')
                t=1
            except:
                time.sleep(1)
                t=0    
        length = (len(segment))
        try:
            city= (segment[length-5])
        except:
            city= ("")
        try:
            state= (segment[length-3])
        except:
            state= ("")
        try:
            region= (segment[length-4])
        except:
            region= ("")
        try:
            country= (segment[length-1])
        except:
            country= ("")

        datajson+=str( {"processor": "Image_Based_Location_Retrieval",
                "model": modelnameb+" & geopy.geocoders.GoogleV3",
                "inferencetime": time.time() - start_time,
                "result":str(gps_coords[0]),
                "location":country+';'+state+';'+region+';'+city
                })
        
        datajson+=']'

        print ("##### Run GeoCLIP")
        
        model = GeoCLIP()
        image_path = "requests.jpg"
        start_time = time.time()
        top_pred_gps, top_pred_prob = model.predict(image_path, top_k=5)

        for i in range(1):
            lat, lon = top_pred_gps[i]
            print(f"Prediction {i+1}: ({lat:.6f}, {lon:.6f})")
            print(f"Probability: {top_pred_prob[i]:.6f}")
            print("")

            geolocator = Nominatim(user_agent="4DS")

            coords=(top_pred_gps[i])
            t=0
            while (t==0):
                try: 
                    segment = (geolocator.reverse(coords, language="en").address).split(',')
                    t=1
                except:
                    time.sleep(1)
                    t=0
            length = (len(segment))
            try:
                city= (segment[length-5])
            except:
                city= ("")
            try:
                state= (segment[length-3])
            except:
                state= ("")
            try:
                region= (segment[length-4])
            except:
                region= ("")
            try:
                country= (segment[length-1])
            except:
                country= ("")
            datajson+='['
            datajson+=str( {"processor": "Image_Based_Location_Retrieval",
                    "model": modelnamec+" & geopy.geocoders.GoogleV3",
                    "inferencetime": time.time() - start_time,
                    "result":str(gps_coords[0]),
                    "location":country+';'+state+';'+region+';'+city
                    })
            
            datajson+=']'
        print ("##### Run OrienterNet")

        starttime=time.time()
        demo = Demo(num_rotations=128, device='cpu') # change to "cuda" if you have a GPU.
        tile_size_meters = 128 #@param [64, 128, 256, 512]
        image_path="requests.jpg"
        print(f"Using image {image_path} with location prior '{address}'")

        image, camera, gravity, proj, bbox = demo.read_input_image(
            image_path,
            prior_address=address,
            tile_size_meters=tile_size_meters,
        )

        # Show the query area in an interactive map
        #from maploc.osm.viz import GeoPlotter
        #plot = GeoPlotter(zoom=16)
        #plot.points(proj.latlonalt[:2], "red", name="location prior", size=10)
        #plot.bbox(proj.unproject(bbox), "blue", name="map tile")
        #plot.fig.show()

        # Query OpenStreetMap for this area
        tiler = TileManager.from_bbox(proj, bbox + 10, demo.config.data.pixel_per_meter)
        canvas = tiler.query(bbox)

        # Show the inputs to the model: image and raster map
        #from maploc.osm.viz import Colormap, plot_nodes
        #from maploc.utils.viz_2d import plot_images
        map_viz = Colormap.apply(canvas.raster)
        #plot_images([image, map_viz], titles=["input image", "OpenStreetMap raster"])
        #plot_nodes(1, canvas.raster[2], fontsize=6, size=10)

        # @title Localize!
        from maploc.utils.viz_localization import (
            likelihood_overlay,
            plot_dense_rotations,
            add_circle_inset,
        )
        from maploc.utils.viz_2d import features_to_RGB

        # Run the inference
        uv, yaw, prob, neural_map, image_rectified = demo.localize(
            image, camera, canvas, gravity=gravity
        )

        coords=(proj.unproject(canvas.to_xy(uv)))
        # Plot as interactive figure
        bbox_ll = proj.unproject(canvas.bbox)
        plot = GeoPlotter(zoom=16.5)
        plot.raster(map_viz, proj.unproject(bbox), opacity=0.5)
        plot.raster(likelihood_overlay(prob.numpy().max(-1)), proj.unproject(bbox))
        plot.points(proj.latlonalt[:2], "red", name="location prior", size=10)
        plot.points(proj.unproject(canvas.to_xy(uv)), "black", name="argmax", size=10)
        plot.bbox(proj.unproject(bbox), "blue", name="map tile")

        print(yaw)
        #plot.fig.show()

        #coords=str(coords)[1:((len(address))-2)].split('    ')
        stringtemp=''
        geolocator = Nominatim(user_agent="4DS")
        t=0
        while (t==0):
            try: 
                segment = (geolocator.reverse(coords, language="en").address).split(',')
                t=1
            except:
                time.sleep(1)
                t=0
        length = (len(segment))
        print(segment)
        try:
            city= (segment[length-5])
        except:
            city= ("")
        try:
            state= (segment[length-3])
        except:
            state= ("")
        try:
            region= (segment[length-4])
        except:
            region= ("")
        try:
            country= (segment[length-1])
        except:
            country= ("")
        datajson+='['    
        datajson+=str({"processor": "Image_Based_Location_Retrieval",
                "model": modelnamea+" & geopy.geocoders.GoogleV3",
                "inferencetime": time.time() - starttime,
                "result":str(coords),
                "location":country+';'+state+';'+region+';'+city,
                "yaw":yaw.to(torch.float)
                })
        
        datajson+=']'

        model="original" 
        print(datajson)

        try:
            city= (segment[length-5])
        except:
            city= ("")
        try:
            state= (segment[length-3])
        except:
            state= ("")
        try:
            region= (segment[length-4])
        except:
            region= ("")
        try:
            country= (segment[length-1])
        except:
            country= ("")
        stringtemp+=(';'+((model+';'+str(params["lat"])+';'+str(params["lon"])+';'+country+';'+state+';'+region+';'+city)))
        
        stringtest=datajson.split('][')
        for i in (stringtest):
            
            #    stringtemp+=(';'+(model+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[2].split(','))[0]))))+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[3].split(','))[0]))))+';;;'))
            
            j+=1
            stringproxy=i.split('Image_Based_Location_Retrieval')
            for k in stringproxy:
                if(len(k)>20):
                    print(k)
                    l=(k.split("'model': ")[1]).split(",")[0]
                    m=(k.split("'location': ")[1]).split(",")[0]
                    m=m.split("}")[0]
                    stringtemp+="#;;;"+l+";"+m
        print(stringtemp)
        
        
        
        updatequery='INSERT INTO "'+DBTABLE+'" (id, aiPlaceLabel) VALUES ("'+id+'","'+str(stringtemp)+'");'
        print(updatequery)
        db = sqlite3.connect(DATABASE)
        cura = db.cursor()
        cura.execute(str(updatequery))
        db.commit()
        checkquery = 'SELECT * FROM "'+DBTABLE+'" WHERE "id" = "'+str(id)+'"'
        cura.execute(checkquery)
        print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
        db.close()
else:
    print('##### '+j)
        #with open("geocoding_image_hist-images.csv", "a") as f:
        #    f.write(str(j)+": "+stringtemp+"\n")
        


### 3.1.2 OrienterNet Image Classification via PLONK
https://github.com/nicolas-dufour/plonk

In [ ]:
# =============================================================================
# STEP: 3.1.2 Image geolocalisation via PLONK
# Standalone PLONK image-to-GPS geolocalisation.
# reads name -> writes aiConceptPrefLabel | Model: nicolas-dufour/PLONK_YFCC | DB: objaverse.sqlite3.db
# =============================================================================

from plonk import PlonkPipeline

import os
import sqlite3
from PIL import Image
import requests

import clip
import time
import numpy as np
import matplotlib.pyplot as plt
import re
from openai import OpenAI
from geopy.geocoders import *
from pathlib import Path
import torch

modelname = "nicolas-dufour/PLONK_YFCC"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

dbvalueread="name" #"edmPreview"
dbvaluewrite="aiConceptPrefLabel"
dbvaluecheck= 'source LIKE "%Europeana%"' #'NAME like "%Vianden%"'

google_key = os.environ["GOOGLE_MAPS_API_KEY"]
DATABASE = 'Tutorials/objaverse.sqlite3.db'

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluecheck+' LIMIT 10') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()
    print(uidlist)

    for i in uidlist:
        print(i)
        image_url = i[1].replace("['", "").replace("']", "") #remove breaks
        #image_url = "https://zenodo.org/api/records/10290891/files/thumb0.jpeg/content"
        previousdata = ""#i[2]
        j=j+1
        datajson='['        
        start_time = time.time()
        #%export PYTORCH_ENABLE_MPS_FALLBACK=1 
        image = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')
        pipeline = PlonkPipeline(model_path=modelname, device="cpu")
        gps_coords = pipeline(image, batch_size=1024)
        

        stringtemp=''
        geolocator = Nominatim(user_agent="4DS")

        coords=(str(gps_coords[0]).replace("[", "").replace("]", "")).replace("   ", "  ").replace("  ",",")
        segment = (geolocator.reverse(coords, language="en").address).split(',')
        length = (len(segment))
        try:
            city= (segment[length-5])
        except:
            city= ("")
        try:
            state= (segment[length-3])
        except:
            state= ("")
        try:
            region= (segment[length-4])
        except:
            region= ("")
        try:
            country= (segment[length-1])
        except:
            country= ("")

        datajson+=str( {"processor": "Image_Based_Location_Retrieval",
                "model": modelname,
                "inferencetime": time.time() - start_time,
                "result":str(gps_coords[0]),
                "location":country+';'+state+';'+region+';'+city
                })
        
        datajson+=']'
        print(datajson)           
        updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(previousdata)+datajson.replace('"','')+'" WHERE "uid" = "'+i[0]+'"'
            
        db = sqlite3.connect(DATABASE)
        cura = db.cursor()
        checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
        cura.execute(checkquery)
        print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
        db.close()

main()


## 3.1 Geolocalization via Deepseek R1 + resolving via Geopy / Google Geocoding API
Europeana CHO Types Vocabulary basic section for 3D content [74]





In [ ]:
# =============================================================================
# STEP: 3.1 Text geolocalisation via DeepSeek-R1
# Infers place from the description with DeepSeek-R1, then resolves coordinates via GeoPy / Google Geocoding.
# reads description -> writes aiPlaceLabel | Model: DeepSeek-R1 | DB: objaverse.sqlite3.db
# =============================================================================

from openai import OpenAI
from geopy.geocoders import GoogleV3
import os
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from geo_utilities import *

dbvalueread="description"
dbvaluewrite="aiPlaceLabel"
dbvaluewrite2="aiPlaceLatitude"
dbvaluewrite3="aiPlaceLongitude"
dbvaluecheck='source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL'

deepseek_key = os.environ["DEEPSEEK_API_KEY"]
google_key = os.environ["GOOGLE_MAPS_API_KEY"]

modelname = "DeepSeek-R1"
client = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com")
DATABASE = 'Tutorials/objaverse.sqlite3.db'

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for i in uidlist:
        description = i[1].replace('\n', ' ').replace('\r', '') #remove breaks
        previousdata = i[2]
        j=j+1
        try:
            description = description[(description.find("result':")+8):]
            query= "Please begin your answer with the name of the place and be as exact as possible. The geographic location of the object described in this text: " + str(description)
            start_time = time.time()
            response = client.chat.completions.create(
            model="deepseek-reasoner",
            messages = [{"role": "user", "content": query}]
            )
            reasoning_content = response.choices[0].message.reasoning_content
            content = response.choices[0].message.content
            end_time = time.time()
            datajson='['  
            datajson+=str( {"processor": "NER",
                    "model": modelname+" & geopy.geocoders.GoogleV3",
                    "inferencetime": end_time - start_time,
                    "result": content})
            datajson+=']'

            geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
            location = geolocator.geocode(content[:200])
            
            try:
                datajson+='['  
                datajson+=str( {"processor": "GEOCODING",
                        "model": modelname+" & geopy.geocoders.GoogleV3",
                        "address": location.address,
                        "latitude": location.latitude,
                        "longitude": location.longitude,
                        "address": location.raw})
                datajson+=']'

        
            except:
                print("Location could not be geocoded.")
            
            updatequery='UPDATE object SET '+dbvaluewrite+'="'+datajson.replace('"','')+'",'+dbvaluewrite2+'="'+str(location.latitude)+'",'+dbvaluewrite3+'="'+str(location.longitude)+'" WHERE "uid" = "'+i[0]+'"'
            
            db = sqlite3.connect(DATABASE)
            cura = db.cursor()
            cura.execute(str(updatequery))
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
            cura.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
            db.close()
        except:
            print(f"Processing {i[0]} loading error occurred")
        
        continue

main()


## 3.2 Geolocalization via Distilled Deepseek R1 & QWEN + resolving via Geopy / Google Geocoding API

In [ ]:
# =============================================================================
# STEP: 3.2 Text geolocalisation via distilled DeepSeek-R1 + Qwen
# Lighter text-geolocalisation using a distilled DeepSeek-R1 / Qwen model with the same coordinate resolution.
# reads description -> writes aiPlaceLabel | Model: DeepSeek-R1-Distill-Qwen-1.5B | DB: objaverse.sqlite3.db
# =============================================================================

import os
import time
import spacy
import re
import sqlite3
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from huggingface_hub import InferenceClient

modelname = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  # alternatives: Qwen/Qwen2.5-Math-1.5B" #DeepSeek-R1-Distill-Qwen-7B" #DeepSeek-R1-Distill-Qwen-14B" #DeepSeek-R1-Distill-Qwen-32B" #deepseek-llm-67b-base" #model_name = "Qwen/Qwen2.5-Math-1.5B"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

from geopy.geocoders import GoogleV3

dbvalueread="description"
dbvaluewrite="aiPlaceLabel"
dbvaluewrite2="aiPlaceLatitude"
dbvaluewrite3="aiPlaceLongitude"
dbvaluecheck='source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL AND aiPlaceLabel LIKE "%DeepSeek-R1%"'

deepseek_key = os.environ["DEEPSEEK_API_KEY"]
google_key = os.environ["GOOGLE_MAPS_API_KEY"]

DATABASE = 'Tutorials/objaverse.sqlite3.db'

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))

    tokenizer = AutoTokenizer.from_pretrained(modelname)
    model = AutoModelForCausalLM.from_pretrained(modelname, device_map="auto")
    model.generation_config = GenerationConfig.from_pretrained(modelname)
    model.generation_config.pad_token_id = model.generation_config.eos_token_id

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for i in uidlist:
        description = i[1].replace('\n', ' ').replace('\r', '') #remove breaks
        previousdata = i[2]
        j=j+1
        try:
            description = description[(description.find("result':")+2):]
            query= "Please begin your answer with the name of the place and be as exact as possible. The geographic location of the object described in this text: " + str(description)
            
            datajson='['        
            start_time = time.time()

            #query= "Is the object named in the following text located in Europe? Please start your answer with yes or not : "
            #query= "Please begin your answer with the name of the location. The geographic location of the object described in this text: "
            text = query + description[78:]
            inputs = tokenizer(text, return_tensors="pt")
            outputs = model.generate(**inputs.to(model.device), max_new_tokens=100)
            end_time = time.time()
            print (description)

            datajson+=str( {"processor": "NER",
                    "model": modelname,
                    "inferencetime": end_time - start_time,
                    "result":tokenizer.decode(outputs[0], skip_special_tokens=True)})
            
            datajson+=']'
                
            geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
            location = geolocator.geocode(tokenizer.decode(outputs[0], skip_special_tokens=True))
            
            try:
                datajson+='['  
                datajson+=str( {"processor": "GEOCODING",
                        "model": modelname+" & geopy.geocoders.GoogleV3",
                        "address": location.address,
                        "latitude": location.latitude,
                        "longitude": location.longitude})
                datajson+=']'
            except:
                print("Location could not be geocoded.")

        except:
            print(f"Processing {i[0]} loading error occurred")    
            datajson+=str( {"processor": "NER",
                "model": modelname,
                "result": "No result"})
            datajson+=']'
            
        updatequery='UPDATE object SET '+dbvaluewrite+'="'+previousdata+datajson.replace('"','')+'" WHERE "uid" = "'+i[0]+'"'
        
        db = sqlite3.connect(DATABASE)
        cura = db.cursor()
        cura.execute(str(updatequery))
        db.commit()
        checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
        cura.execute(checkquery)
        print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
        db.close()
        
        continue

main()


## Geolocalization of Megascenes via Google GeoPy

In [ ]:
# =============================================================================
# STEP: Geolocalisation of MegaScenes via Qwen + Google GeoPy
# Derives place names for MegaScenes records with Qwen3-32B and resolves them via Google GeoPy.
# reads id -> writes name | Model: Qwen/Qwen3-32B-MLX-4bit | DB: objaverse.sqlite32.db
# =============================================================================

import os
import time
import re
import sqlite3
from pathlib import Path
from geopy.geocoders import GoogleV3
import urllib.request, json, requests

#modelname = "Qwen/Qwen3-32B-MLX-4bit" #Qwen/Qwen2.5-Math-1.5B" #DeepSeek-R1-Distill-Qwen-7B" #DeepSeek-R1-Distill-Qwen-14B" #DeepSeek-R1-Distill-Qwen-32B" #deepseek-llm-67b-base" #model_name = "Qwen/Qwen2.5-Math-1.5B"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

#!pip install geopy

dbvalueread="id"
dbvaluewrite="name"

dbvaluecheck='source LIKE "%Megascenes%" AND edmPlaceLatitude IS "" LIMIT 10'
google_key = os.environ["GOOGLE_MAPS_API_KEY"]

DATABASE = 'Tutorials/objaverse.sqlite32.db'

uidlist=[]

def main():
     conn = sqlite3.connect(DATABASE)  # Replace with your database file
     cursor = conn.cursor()

       #Counts the number of processed items 
     cursor.execute('select count(*) from object WHERE source LIKE "%Megascenes%" AND edmPlaceLatitude IS NOT "" ;')
     numberOfRows = cursor.fetchone()[0]
     print ("Number of datasets already processed: "+str(numberOfRows))

     #Counts the number of remaining items
     cursor.execute('select count(*) from object WHERE source LIKE "%Megascenes%" AND edmPlaceLatitude IS "";')
     numberOfRows = cursor.fetchone()[0]
     print ("Number of datasets to process is: "+str(numberOfRows))

     # Query rows where source is 'Megascenes' and edmPlaceLabel is empty
     cursor.execute("SELECT id,name FROM object WHERE source = 'Megascenes' AND edmPlaceLabel = ''")
     rows = cursor.fetchall()

     # Iterate through the rows and update the database
     for row in rows:
          object_id = row[0]  # Assuming the first column is the object ID
          name = row[1].replace('"','')  # Assuming the second column is the "name" field

    
          print(name)
          geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
          try:
               location = geolocator.geocode(name)
               print(location.latitude, location.longitude)
          #     location=addressjson
          except:
               print("Location could not be geocoded.")

          if location:
               edmPlaceLabel = location.address
               edmPlaceLatitude = location.latitude
               edmPlaceLongitude = location.longitude

               # Update the row in the database
               cursor.execute("""
                    UPDATE object
                    SET edmPlaceLabel = ?, edmPlaceLatitude = ?, edmPlaceLongitude = ?
                    WHERE id = ?
               """, (edmPlaceLabel, edmPlaceLatitude, edmPlaceLongitude, object_id))
               print(f"Updated object ID {object_id} with place: {edmPlaceLabel}")
               conn.commit()
          else:
               print(f"No location found for object ID {object_id} with name: {name}")

     # Commit changes and close the connection
   
   
     conn.close()
     

main()


## 3.3 Geolocalization via LLama 3.2 + resolving via Geopy / Google Geocoding API

In [ ]:
# =============================================================================
# STEP: 3.3 Text geolocalisation via Llama 3.2
# Place inference from description using Llama-3.2-3B, resolved via GeoPy / Google.
# reads description -> writes aiPlaceLabel | Model: meta-llama/Llama-3.2-3B | DB: objaverse.sqlite3.db
# =============================================================================

import os
import time
import spacy
import re
import sqlite3
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from huggingface_hub import InferenceClient, login

modelname = "meta-llama/Llama-3.2-3B" #Qwen/Qwen2.5-Math-1.5B" #DeepSeek-R1-Distill-Qwen-7B" #DeepSeek-R1-Distill-Qwen-14B" #DeepSeek-R1-Distill-Qwen-32B" #deepseek-llm-67b-base" #model_name = "Qwen/Qwen2.5-Math-1.5B"
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

from geopy.geocoders import GoogleV3

dbvalueread="description"
dbvaluewrite="aiPlaceLabel"
dbvaluewrite2="aiPlaceLatitude"
dbvaluewrite3="aiPlaceLongitude"

dbvaluecheck='source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL AND aiPlaceLabel LIKE "%deepseek-ai%"'

deepseek_key = os.environ["DEEPSEEK_API_KEY"]
google_key = os.environ["GOOGLE_MAPS_API_KEY"]
huggingfaces_key = os.environ["HUGGINGFACE_TOKEN"]

DATABASE = 'Tutorials/objaverse.sqlite3.db'

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, '+dbvalueread+', '+dbvaluewrite + ' '
             'FROM object '
             'WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    cur.execute('select count(*) from object WHERE '+dbvalueread+' IS NOT NULL AND '+dbvaluewrite+' NOT LIKE "%'+modelname+'%" AND '+dbvaluecheck+' ;')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets to process is: "+str(numberOfRows))
    login(huggingfaces_key)
    tokenizer = AutoTokenizer.from_pretrained(modelname)
    model = AutoModelForCausalLM.from_pretrained(modelname, device_map="auto")
    model.generation_config = GenerationConfig.from_pretrained(modelname)
    model.generation_config.pad_token_id = model.generation_config.eos_token_id

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for i in uidlist:
        description = i[1].replace('\n', ' ').replace('\r', '') #remove breaks
        previousdata = i[2]
        j=j+1
        try:
            description = description[(description.find("result':")+2):]
            query= "Please begin your answer with the name of the place and be as exact as possible. The geographic location of the object described in this text: " + str(description)
            
            datajson='['        
            start_time = time.time()

            #query= "Is the object named in the following text located in Europe? Please start your answer with yes or not : "
            #query= "Please begin your answer with the name of the location. The geographic location of the object described in this text: "
            text = query + description[78:]
            inputs = tokenizer(text, return_tensors="pt")
            outputs = model.generate(**inputs.to(model.device), max_new_tokens=100)
            end_time = time.time()
            #print (description)

            datajson+=str( {"processor": "NER",
                    "model": modelname,
                    "inferencetime": end_time - start_time,
                    "result":tokenizer.decode(outputs[0], skip_special_tokens=True)})
            
            datajson+=']'
                
            geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
            location = geolocator.geocode(tokenizer.decode(outputs[0], skip_special_tokens=True))
            
            try:
                datajson+='['  
                datajson+=str( {"processor": "GEOCODING",
                        "model": modelname+" & geopy.geocoders.GoogleV3",
                        "address": location.address,
                        "latitude": location.latitude,
                        "longitude": location.longitude})
                datajson+=']'
            except:
                print("Location could not be geocoded.")

        except:
            print(f"Processing {i[0]} loading error occurred")    
            datajson+=str( {"processor": "NER",
                "model": modelname,
                "result": "No result"})
            datajson+=']'
            
        updatequery='UPDATE object SET '+dbvaluewrite+'="'+previousdata+datajson.replace('"','')+'" WHERE "uid" = "'+i[0]+'"'

        db = sqlite3.connect(DATABASE)
        cura = db.cursor()
        cura.execute(str(updatequery))
        db.commit()
        checkquery = 'SELECT * FROM object WHERE "uid" = "'+i[0]+'"'
        cura.execute(checkquery)
        print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
        db.close()        
        continue

main()


## 3.4 Geolocalization via Qwen3-32B-MLX-4bit + resolving via Geopy / Google Geocoding API

In [ ]:
# =============================================================================
# STEP: 3.4 Text geolocalisation via Qwen3-32B + Nominatim
# Place inference with Qwen3-32B resolved through OSM / Nominatim.
# reads name -> writes aiPlaceLabel | Model: Qwen3-32B-MLX-4bit | DB: objaverse.sqlite32.db
# =============================================================================

import os
import time
import sqlite3
from mlx_lm import load, generate
from pathlib import Path
from geopy.geocoders import GoogleV3
from geopy.geocoders import Nominatim

# Model and API keys
modelname = "Qwen/Qwen3-32B-MLX-4bit-Nominatim"
dbvalueread = "name"
dbvaluewrite = "aiPlaceLabel"
dbvaluecheck = 'source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL'

google_key = os.environ["GOOGLE_MAPS_API_KEY"]
DATABASE = 'Tutorials/objaverse.sqlite32.db'
uidlist = []

# Load the model and tokenizer
model, tokenizer = load("Qwen/Qwen3-32B-MLX-4bit")

# Change directory if necessary
if Path().absolute().name == "Tutorials":
    os.chdir(Path().absolute().parent)

def main():
    retrievalquery = (
        f'SELECT uid, {dbvalueread}, {dbvaluewrite}, aiDescriptionEn '
        f'FROM object '
        f'WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} NOT LIKE "%{modelname}%" AND {dbvaluecheck}'
    )
    print(retrievalquery)

    db = sqlite3.connect(DATABASE)
    cur = db.cursor()

    # Count processed items
    cur.execute(
        f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} LIKE "%{modelname}%" AND {dbvaluecheck};'
    )
    print(f"Number of datasets already processed: {cur.fetchone()[0]}")

    # Count remaining items
    cur.execute(
        f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} NOT LIKE "%{modelname}%" AND {dbvaluecheck};'
    )
    print(f"Number of datasets to process: {cur.fetchone()[0]}")

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for j, i in enumerate(uidlist, start=1):
        name = i[1].replace('\n', ' ').replace('\r', '')
        previousdata = i[2]
        if i[3]:
            if 'result": "' in i[3]:
                description=i[3].split('result": "')[1]
            else:
                description = i[3].replace('\n', ' ').replace('\r', '')
        else:
            description=""
        datajson = ''
        try:
            query = (
                "Please begin your answer with the name of the place and be as exact as possible. Please begin your answer with NONE if no location is mentioned. "
                f"The geographic location of the object described in this text: {name} +' - '+ {description}"
            )
       
            start_time = time.time()

            # Prepare messages for the model
            messages = [{"role": "user", "content": query}]

            # Generate response
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=False
            )
            result = generate(
                model, tokenizer, prompt=prompt, verbose=True, max_tokens=128
            )
            end_time = time.time()

            datajson += str({
                "processor": "NER",
                "model": modelname,
                "inferencetime": end_time - start_time,
                "result": result
            })
            print(result)

            # Geocoding
            if not "NONE" in result.upper():
                geolocator = Nominatim(user_agent="4DS")
                #location = (geolocator.reverse(result, language="en").address).split(',')
                location = geolocator.geocode(result)

                #geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
                #location = geolocator.geocode(result)
                if location:
                    datajson += str({
                        "processor": "GEOCODING",
                        "model": f"{modelname} & geopy.geocoders.Nominatim", #geopy.geocoders.GoogleV3
                        "address": location.address,
                        "latitude": location.latitude,
                        "longitude": location.longitude
                    })
            else:
                print("Location could not be geocoded.")

        except Exception as e:
            print(f"Processing {i[0]} loading error occurred: {e}")
            datajson += str({
                "processor": "NER",
                "model": modelname,
                "result": "No result"
            })
            datajson += ''

        # Update database
        print('DATAJSON: '+datajson)
        updatequery = (
            f'UPDATE object SET {dbvaluewrite}="{previousdata.replace('"', '\'') + datajson.replace('"', '\'')}" ' #edmPlaceLatitude="{location.latitude if location else ""}", edmPlaceLongitude="{location.longitude if location else ""}", edmPlaceLabel="{location.address.replace('\"', '\'') if location else ""}"
            f'WHERE "uid" = "{i[0]}"'
        )
        print(updatequery)
        db = sqlite3.connect(DATABASE)
        cura = db.cursor()
        cura.execute(updatequery)
        db.commit()

        # Verify update
        checkquery = f'SELECT * FROM object WHERE "uid" = "{i[0]}"'
        cura.execute(checkquery)
        print(f"#{j}: Successfully updated: {cura.fetchall()}")
        db.close()

if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# STEP: 3.4b Text geolocalisation via Qwen3-14B
# Same geolocalisation step using the smaller Qwen3-14B model.
# reads name -> writes aiPlaceLabel | Model: Qwen3-14B-MLX-4bit | DB: objaverse.sqlite32.db
# =============================================================================

import os
import time
import sqlite3
from mlx_lm import load, generate
from pathlib import Path
from geopy.geocoders import GoogleV3
geolocator = Nominatim(user_agent="4DS")
# Model and API keys
modelname = "Qwen/Qwen3-14B-MLX-4bit"
dbvalueread = "name"
dbvaluewrite = "aiPlaceLabel"
dbvaluecheck = 'source LIKE "%Objaverse%" AND edmPlaceLatitude IS NULL'

google_key = os.environ["GOOGLE_MAPS_API_KEY"]
DATABASE = 'Tutorials/objaverse.sqlite32.db'
uidlist = []

# Load the model and tokenizer
model, tokenizer = load("Qwen/Qwen3-14B-MLX-4bit")

# Change directory if necessary
if Path().absolute().name == "Tutorials":
    os.chdir(Path().absolute().parent)

def main():
    retrievalquery = (
        f'SELECT uid, {dbvalueread}, {dbvaluewrite}, aiDescriptionEn '
        f'FROM object '
        f'WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} NOT LIKE "%{modelname}%" AND {dbvaluecheck}'
    )
    print(retrievalquery)

    db = sqlite3.connect(DATABASE)
    cur = db.cursor()

    # Count processed items
    cur.execute(
        f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} LIKE "%{modelname}%" AND {dbvaluecheck};'
    )
    print(f"Number of datasets already processed: {cur.fetchone()[0]}")

    # Count remaining items
    cur.execute(
        f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} NOT LIKE "%{modelname}%" AND {dbvaluecheck};'
    )
    print(f"Number of datasets to process: {cur.fetchone()[0]}")

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()

    for j, i in enumerate(uidlist, start=1):
        name = i[1].replace('\n', ' ').replace('\r', '')
        previousdata = i[2]
        if i[3]:
            if 'result": "' in i[3]:
                description=i[3].split('result": "')[1]
            else:
                description = i[3].replace('\n', ' ').replace('\r', '')
        else:
            description=""
        datajson = ''
        try:
            query = (
                "Please begin your answer with the name of the place and be as exact as possible. Please begin your answer with NONE if no location is mentioned. "
                f"The geographic location of the object described in this text: {name} +' - '+ {description}"
            )
       
            start_time = time.time()

            # Prepare messages for the model
            messages = [{"role": "user", "content": query}]

            # Generate response
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=False
            )
            result = generate(
                model, tokenizer, prompt=prompt, verbose=True, max_tokens=128
            )
            end_time = time.time()

            datajson += str({
                "processor": "NER",
                "model": modelname,
                "inferencetime": end_time - start_time,
                "result": result
            })
            print(result)

            # Geocoding
            if not "NONE" in result.upper():
                geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
                location = geolocator.geocode(result)
                if location:
                    datajson += str({
                        "processor": "GEOCODING",
                        "model": f"{modelname} & geopy.geocoders.GoogleV3",
                        "address": location.address,
                        "latitude": location.latitude,
                        "longitude": location.longitude
                    })
            else:
                print("Location could not be geocoded.")

        except Exception as e:
            print(f"Processing {i[0]} loading error occurred: {e}")
            datajson += str({
                "processor": "NER",
                "model": modelname,
                "result": "No result"
            })
            datajson += ''

        # Update database
        updatequery = (
            f'UPDATE object SET {dbvaluewrite}="{previousdata.replace('"', '\'') + datajson.replace('"', '\'')}", edmPlaceLatitude="{location.latitude if location else ""}", edmPlaceLongitude="{location.longitude if location else ""}", edmPlaceLabel="{location.address.replace('\"', '\'') if location else ""}" '
            f'WHERE "uid" = "{i[0]}"'
        )
        print(updatequery)
        db = sqlite3.connect(DATABASE)
        cura = db.cursor()
        cura.execute(updatequery)
        db.commit()

        # Verify update
        checkquery = f'SELECT * FROM object WHERE "uid" = "{i[0]}"'
        cura.execute(checkquery)
        print(f"#{j}: Successfully updated: {cura.fetchall()}")
        db.close()

if __name__ == "__main__":
    main()


### 3.4 SV Export of results
The comparison uses the geopy + OSM / Nominatim Geoservice to resolve coordinates from EDMPlace and AIPlace for comparison.
https://geopy.readthedocs.io/en/stable/#data

In [ ]:
# =============================================================================
# STEP: Helper: load Qwen3-8B text-generation pipeline
# Loads a Qwen3-8B tokenizer/model for ad-hoc text generation used by the export helpers below.
# Model: Qwen/Qwen3-8B
# =============================================================================

# Use a pipeline as a high-level helper
#%pip install -update transformers
#%pip install update pytorch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-8B")

#query= "Please begin your answer with the name of the place and be as exact as possible. The geographic location of the object described in this text: " + str(description)
    
datajson='['        
start_time = time.time()

#query= "Is the object named in the following text located in Europe? Please start your answer with yes or not : "
query= "Please begin your answer with the name of the location. The geographic location of the object described in this text: "
text = query
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs.to(model.device), max_new_tokens=100)
end_time = time.time()
#print (description)

datajson+=str( {"processor": "NER",
        "model": modelname,
        "inferencetime": end_time - start_time,
        "result":tokenizer.decode(outputs[0], skip_special_tokens=True)})

datajson+=']'


Export of Geocodings for Image-based locations

In [ ]:
# =============================================================================
# STEP: Export image-based geocodings
# Resolves and exports geocodings for image-based locations (GoogleV3 / Nominatim).
# DB: objaverse.sqlite3.db
# =============================================================================

from geopy.geocoders import Nominatim
import os
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from geo_utilities import *

dbvaluecheck='source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL AND aiPlaceLatitude IS NOT NULL AND aiPlaceLabel LIKE "%nicolas-dufour/PLONK_YFCC%"'# limit 2'
DATABASE = 'Tutorials/objaverse.sqlite3.db'
uidlist=[]

def main():
    retrievalquery = ('SELECT uid, aiPlaceLabel, edmPlaceLatitude, edmPlaceLongitude '
             'FROM object '
             'WHERE '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    #Counts the number of processed items 
    #numberOfRows = cur.fetchone()[0]
    #print ("Number of datasets already processed: "+str(numberOfRows))

    #Counts the number of remaining items
    #numberOfRows = cur.fetchone()[0]
    #print ("Number of datasets to process is: "+str(numberOfRows))

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()
    
    #print (uidlist)
    with open("geocoding_image.csv", "w") as f:
        f.write("")
    
    for i in uidlist:
        print("##### "+str(j))
        stringtemp=i[0].split(";")[0]
        geolocator = Nominatim(user_agent="4DS")

      
        model="Original"
        coordinates= (str(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[2].split(','))[0]))))+','+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[3].split(','))[0])))))
        coordinates=coordinates.replace("'","")
        segment = (geolocator.reverse(coordinates, language="en").address).split(',')
        length = (len(segment))
        try:
            city= (segment[length-5])
        except:
            city= ("")
        try:
            state= (segment[length-3])
        except:
            state= ("")
        try:
            region= (segment[length-4])
        except:
            region= ("")
        try:
            country= (segment[length-1])
        except:
            country= ("")
        stringtemp+=(';'+((model+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[2].split(','))[0]))))+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[3].split(','))[0]))))+';'+country+';'+state+';'+region+';'+city)))
        #    stringtemp+=(';'+(model+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[2].split(','))[0]))))+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[3].split(','))[0]))))+';;;'))
        
        j+=1
        stringproxy=i[1].split('Image_Based_Location_Retrieval')
        for k in stringproxy:
            l=(k.split("'model': ")[1]).split(",")[0]
            m=(k.split("'location': ")[1]).split(",")[0]
            m=m.split("}")[0]
            stringtemp+="#;;;"+l+";"+m
        print(stringtemp)
        
        with open("geocoding_image.csv", "a") as f:
          f.write(str(j)+": "+stringtemp+"\n")
        
main()     


### Export of Geocodings from text

In [ ]:
# =============================================================================
# STEP: Export text-based geocodings
# Resolves and exports geocodings derived from text-based place inference.
# DB: objaverse.sqlite3.db
# =============================================================================

from geopy.geocoders import Nominatim
import os
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from geo_utilities import *

dbvaluecheck='source LIKE "%Europeana%" AND edmPlaceLatitude IS NOT NULL AND aiPlaceLatitude IS NOT NULL '# limit 2'

DATABASE = 'Tutorials/objaverse.sqlite3.db'

uidlist=[]

def main():
    retrievalquery = ('SELECT uid, aiPlaceLabel, edmPlaceLatitude, edmPlaceLongitude '
             'FROM object '
             'WHERE '+dbvaluecheck+' ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")
             #"LIMIT 500 ") #limit 3 to be removed
    
    print(retrievalquery)
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    updatequery = str()
    j=0

    for uid in cur.execute(retrievalquery):
        uidlist.append(uid)
    db.close()
    
    #print (uidlist)
    with open("geocoding.csv", "w") as f:
        f.write("")
    
    for i in uidlist:
        stringtemp=''
        geolocator = Nominatim(user_agent="4DS")

      
        model="Original"
        coordinates= (str(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[2].split(','))[0]))))+','+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[3].split(','))[0])))))
        coordinates=coordinates.replace("'","")
        segment = (geolocator.reverse(coordinates, language="en").address).split(',')
        length = (len(segment))
        try:
            city= (segment[length-5])
        except:
            city= ("")
        try:
            state= (segment[length-3])
        except:
            state= ("")
        try:
            region= (segment[length-4])
        except:
            region= ("")
        try:
            country= (segment[length-1])
        except:
            country= ("")
        stringtemp+=(';'+((model+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[2].split(','))[0]))))+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[3].split(','))[0]))))+';'+country+';'+state+';'+region+';'+city)))
        #    stringtemp+=(';'+(model+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[2].split(','))[0]))))+';'+(re.sub(r"\'\]", "", (re.sub(r"\[\'", "", (i[3].split(','))[0]))))+';;;'))
        
        
        #stringtemp+=('Original: '+str(location))
        try: 
            try:
                locationtemp = str(i[1]).split("geopy.geocoders.GoogleV3', 'address")                     
                for i in range (0,len(locationtemp)-1):
                
                    model = ((((locationtemp[i].split("'model': "))[1]).split(','))[0])
                    try:
                        time = ((((locationtemp[i].split("'inferencetime': "))[1]).split(','))[0])
                    except:
                        time=""
                    latitude=((((locationtemp[(i+1)].split("'latitude': "))[1]).split(','))[0])
                    longitude=((((locationtemp[(i+1)].split("'longitude': "))[1]).split('}'))[0])

                    segment=(geolocator.reverse(latitude+','+longitude, language="en").address).split(',')
                    length = (len(segment))
                    try:
                        city= (segment[length-5])
                    except:
                        city= ("")
                    try:
                        state= (segment[length-3])
                    except:
                        state= ("")
                    try:
                        region= (segment[length-4])
                    except:
                        region= ("")
                    try:
                        country= (segment[length-1])
                    except:
                        country= ("")
                    stringtemp+=(';'+(model+';'+latitude+';'+longitude+';'+time+';'+country+';'+state+';'+region))
                
                    #stringtemp+=(' #'+(model+': '+str(geolocator.reverse(latitude+','+longitude).address)))
                    
            except:
                print("No location")
        except:
            print("No location")
        print(str(j)+": "+stringtemp)

        with open("geocoding.csv", "a") as f:
          f.write(str(j)+": "+stringtemp+"\n")
        j=j+1

main()


# Helper Code: Format as JSON

In [ ]:
# =============================================================================
# STEP: Helper: normalise stored JSON fields
# Utility routines to repair / normalise malformed JSON stored in the database and to export records to JSON.
# DB: objaverse.sqlite3.db
# =============================================================================

import sqlite3
import json
import os
from pathlib import Path
import numpy as np

if Path().absolute().name == "Tutorials":
    os.chdir(Path().absolute().parent)

DATABASE = 'Tutorials/objaverse.sqlite3.db'
OUTPUT_JSON = 'database_export.json'

# Convert to JSON-serializable format
def aiDescriptionLang_format_data_for_json(data):
    print(data)
    for item in data:
        if 'result' in item:
            result = item['result']
            if isinstance(result, tuple) and len(result) == 2:
                # Convert tuple and NumPy array to JSON-serializable types
                label = result[0][0] if isinstance(result[0], tuple) and len(result[0]) > 0 else result[0]
                confidence = result[1].tolist() if hasattr(result[1], "tolist") else result[1]
                item['result'] = {'label': label, 'confidence': confidence}
        print(data)
    return data

# Correct malformed JSON strings
def correct_malformed_json(json_string):
    if not isinstance(json_string, str):
        return json_string
    try:
        # Replace single quotes with double quotes and ensure valid JSON
        json_string = json_string.replace("'", '"')
        json.loads(json_string)  # Validate JSON
        return json_string
    except json.JSONDecodeError:
        # Attempt to fix common issues with JSON formatting
        if json_string.count('{') > json_string.count('}'):
            json_string += '}' * (json_string.count('{') - json_string.count('}'))
        elif json_string.count('}') > json_string.count('{'):
            json_string = json_string.rstrip('}')
        if json_string.count('[') > json_string.count(']'):
            json_string += ']' * (json_string.count('[') - json_string.count(']'))
        elif json_string.count(']') > json_string.count('['):
            json_string = json_string.rstrip(']')
        try:
            json.loads(json_string)  # Validate again after fixing
            return json_string
        except json.JSONDecodeError:
            pass
    print(f"Malformed JSON: {json_string}")
    return json_string

# Export database to JSON
def export_database_to_json():
    try:
        db = sqlite3.connect(DATABASE)
        cur = db.cursor()
        cur.execute('SELECT * FROM object LIMIT 1')
        rows = cur.fetchall()
        column_names = [description[0] for description in cur.description]
        results = []
        for row in rows:
            row_dict = {}
            for col_name, col_value in zip(column_names, row):
                if col_name == 'aiDescriptionLang':
                    try:
                        # Debugging: Print the raw value
                        print(f"Debug: aiDescriptionLang raw value: {col_value}")
                        
                        # Parse and format the aiDescriptionLang field
                        if col_value:
                            print(col_value)
                            col_value = aiDescriptionLang_format_data_for_json(str(col_value))
                            #col_value = correct_malformed_json(col_value)
                            col_value = json.dumps(col_value, indent=4)
                        row_dict[col_name] = col_value
                    except Exception as e:
                        print(f"Error processing aiDescriptionLang: {e}")
                        row_dict[col_name] = col_value
                    continue
                elif col_name == 'aiCreationProcess':
                    try:
                        # Parse and format the aiCreationProcess field
                        if col_value:
                            col_value = correct_malformed_json(col_value)
                            col_value = json.loads(col_value)  # Parse as JSON
                        row_dict[col_name] = col_value
                    except Exception as e:
                        print(f"Error processing aiCreationProcess: {e}")
                        row_dict[col_name] = col_value
                    continue
                elif col_name == 'edmPlaceLabelEn':
                    try:
                        # Parse and format the edmPlaceLabelEn field
                        if col_value:
              
                            col_value = correct_malformed_json(col_value)
                            col_value = json.loads(col_value)  # Parse as JSON
                            # Ensure the 'result' field is properly formatted
                            if isinstance(col_value, list):
                                for item in col_value:
                                    if 'result' in item and isinstance(item['result'], str):
                                        item['result'] = item['result'].strip()
                            col_value = json.dumps(col_value, indent=4)
                        row_dict[col_name] = col_value
                    except Exception as e:
                        print(f"Error processing edmPlaceLabelEn: {e}")
                        row_dict[col_name] = col_value
                    continue
                if col_value and isinstance(col_value, str):
                    col_value = correct_malformed_json(col_value)
                row_dict[col_name] = col_value
            results.append(row_dict)
        with open(OUTPUT_JSON, mode='w', encoding='utf-8') as jsonfile:
            json.dump(results, jsonfile, indent=4, ensure_ascii=False)
        print(f"Database exported to {OUTPUT_JSON}")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        if 'db' in locals():
            db.close()

if __name__ == "__main__":
    export_database_to_json()


In [ ]:
# =============================================================================
# STEP: Helper: export database to JSON
# Dumps the object table to a JSON file for downstream reporting.
# =============================================================================

def export_database_to_json():
    try:
        db = sqlite3.connect(DATABASE)
        cur = db.cursor()
        cur.execute('SELECT * FROM object LIMIT 1')  # Fetch all rows
        rows = cur.fetchall()
        column_names = [description[0] for description in cur.description]
        results = []
        
        for row in rows:
            row_dict = {}
            for col_name, col_value in zip(column_names, row):
                if col_name == 'aiDescriptionLanguage':
                    try:
                        # Debugging: Print the raw value
                        print(f"Debug: aiDescriptionLanguage raw value: {col_value}")
                        
                        # Correct malformed JSON
                        if col_value:
                            col_value = correct_malformed_json(col_value)
                            col_value = json.loads(col_value)  # Parse as JSON
                            
                            # Process each item in the list
                            if isinstance(col_value, list):
                                col_value = aiDescriptionLang_format_data_for_json(col_value)
                            
                            # Convert back to JSON string
                            col_value = json.dumps(col_value, indent=4)
                        row_dict[col_name] = col_value
                    except Exception as e:
                        print(f"Error processing aiDescriptionLanguage: {e}")
                        row_dict[col_name] = col_value
                    continue
                if col_value and isinstance(col_value, str):
                    col_value = correct_malformed_json(col_value)
                row_dict[col_name] = col_value
            results.append(row_dict)
        
        # Write the corrected data to the output JSON file
        with open(OUTPUT_JSON, mode='w', encoding='utf-8') as jsonfile:
            json.dump(results, jsonfile, indent=4, ensure_ascii=False)
        print(f"Database exported to {OUTPUT_JSON}")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        if 'db' in locals():
            db.close()

if __name__ == "__main__":
    export_database_to_json()


### 3.1 Translation

In [ ]:
# =============================================================================
# STEP: Translation report + energy export
# Corrects translation JSON and exports translation processing-time / energy statistics.
# -> REPORT-energy_translating-data.csv | DB: objaverse.sqlite3.db
# =============================================================================

import sqlite3
import json
import statistics
import re

DATABASE = 'objaverse.sqlite3.db'
output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-energy_translating-data.csv")

def correct_json(raw_data):
    """
    Versucht, fehlerhafte JSON-Daten zu reparieren.
    """
    try:
        # Fehlende Anführungszeichen um den Wert von "result" hinzufügen und sicherstellen, dass es mit } endet
        raw_data = re.sub(r'("result":\s*)([^"\[\{].*?)(,|})', r'\1"\2"\3', raw_data)

        # Falsche Anführungszeichen innerhalb von Strings korrigieren
        raw_data = raw_data.replace('\"', '\\"').replace('"s ', "'s ")

        # Reparierte JSON-Daten zurückgeben
        return raw_data
    except Exception as e:
        print(f"Fehler beim Reparieren von JSON: {e}")
        return None

def main():
    # Verbindung zur SQLite-Datenbank herstellen
    conn = sqlite3.connect(DATABASE)
    cursor = conn.cursor()

    # Alle Einträge aus der Tabelle `object` abrufen
    query = 'SELECT id, aiDescriptionEn FROM object LIMIT 400000;'
    cursor.execute(query)
    results = cursor.fetchall()

    # Dictionary für die Gruppierung von Zeiten nach Prozessor und Modell
    processor_model_times = {}

    # Durch alle Einträge iterieren
    for record in results:
        record_id, raw_data = record

        if raw_data:  # Nur Einträge mit Daten verarbeiten
            raw_data = raw_data.strip()  # Entfernen von überflüssigen Leerzeichen
            raw_data = raw_data.replace('[', '').replace(']', '')  # Entfernen von eckigen Klammern
            raw_data = raw_data.replace('\'', '"')  # Ersetzen von einfachen Anführungszeichen durch doppelte
            raw_dataa = raw_data.split('"result":')
            raw_dataa[1] = str(raw_dataa[1]).replace('"', '').replace('}', '').replace('\\', '').replace("\t", "")
            raw_dataa[1] = '"result":"'+raw_dataa[1]+'"'
            raw_data = ''.join(raw_dataa)
            if not raw_data.endswith('}'):
                raw_data += '}'
            print(raw_data)
            json_data = json.loads(raw_data)

            # Überprüfen, ob 'result' vorhanden ist
            if "result" in json_data:
                result = json_data['result']
                if isinstance(result, str):
                    model = json_data.get('model', 'N/A')
                    processor = json_data.get('processor', 'N/A')
                    json_data['result'] = result.strip()

            # Überprüfen, ob 'inferencetime' vorhanden ist
            if 'inferencetime' in json_data:
                time = json_data['inferencetime']
                processor = json_data.get('processor', 'N/A')
                model = json_data.get('model', 'N/A')

                # Zeiten nach Prozessor und Modell gruppieren
                if processor not in processor_model_times:
                    processor_model_times[processor] = {}
                if model not in processor_model_times[processor]:
                    processor_model_times[processor][model] = []
                processor_model_times[processor][model].append(time)

    # Ergebnisse in die Ausgabedatei schreiben
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("Processor; Model; Min Time; Mean Time; Median Time; Max Time\n")
        for processor, models in processor_model_times.items():
            for model, times in models.items():
                min_time = str(min(times)).replace('.', ',')
                max_time = str(max(times)).replace('.', ',')
                mean_time = str(statistics.mean(times)).replace('.', ',')
                median_time = str(statistics.median(times)).replace('.', ',')
                f.write(f"{processor}; {model}; {min_time}; {mean_time}; {median_time}; {max_time}\n")

    print(f"Korrigierte Daten wurden in {output_file} gespeichert.")
    conn.close()

if __name__ == "__main__":
    main()


### Geocoding: This is code to correct wrong json entries in the aiPlaceLabel field from Europeana source in the Objaverse database.

In [ ]:
# =============================================================================
# STEP: Geocoding correction + distance report
# Repairs aiPlaceLabel JSON and computes haversine distances against ground truth.
# -> REPORT-corrected_data.json | DB: objaverse.sqlite32.db
# =============================================================================

#This is code to correct wrong json entries in the aiPlaceLabel field from Europeana source in the Objaverse database.
import sqlite3
import json
import re
import math
from geopy.geocoders import GoogleV3

google_key = os.environ["GOOGLE_MAPS_API_KEY"]
DATABASE = 'objaverse.sqlite32.db'

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Berechnet die Entfernung zwischen zwei Punkten auf der Erde (in Kilometern) basierend auf ihren Breitengraden und Längengraden.
    
    :param lat1: Breitengrad des ersten Punkts
    :param lon1: Längengrad des ersten Punkts
    :param lat2: Breitengrad des zweiten Punkts
    :param lon2: Längengrad des zweiten Punkts
    :return: Entfernung in Kilometern
    """
    # Radius der Erde in Kilometern
    R = 6371.0

    # Koordinaten in Bogenmaß umwandeln
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Unterschiede berechnen
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # Haversine-Formel
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Entfernung berechnen
    distance = R * c
    return distance

# Beispielaufruf
lat1, lon1 = 52.5200, 13.4050  # Berlin
lat2, lon2 = 48.8566, 2.3522   # Paris

def main():
    # Datenbankpfad
    
    geolocator = GoogleV3(api_key=google_key, domain='maps.googleapis.com')
    # Verbindung zur SQLite-Datenbank herstellen
    conn = sqlite3.connect(DATABASE)
    cursor = conn.cursor()

    # Alle Einträge aus der Tabelle `object` abrufen
    query = 'SELECT id, aiPlaceLabel, edmPlaceLabel, edmPlaceLatitude, edmPlaceLongitude FROM object WHERE "source" like "Europeana" LIMIT 40000;'
    cursor.execute(query)
    results = cursor.fetchall()

    # Liste für korrigierte Daten
    corrected_records = []

    # Durch alle Einträge iterieren
    for record in results:
        record_id, raw_data, edmPlaceLabel, edmPlaceLatitude, edmPlaceLongitude = record

        if edmPlaceLabel and edmPlaceLatitude and edmPlaceLongitude:
            original_json_data = {
                "location": edmPlaceLabel.replace('"','').strip(),
                "latitude": float(edmPlaceLatitude.replace('"','').split(',')[0]),
                "longitude": float(edmPlaceLongitude.replace('"','').split(',')[0])
            }
        elif  edmPlaceLabel:
            try:
                address = geolocator.geocode(edmPlaceLabel)
                original_json_data = {    
                    "location": edmPlaceLabel.replace('"','').strip(),
                    "latitude": address.latitude,
                    "longitude": address.longitude
                }
            except:
                original_json_data = {    
                    "location": edmPlaceLabel.replace('"','').strip(),
                    "latitude": "",
                    "longitude": ""
                }
        if(edmPlaceLatitude and edmPlaceLongitude):
            lat1=float(edmPlaceLatitude.replace('"','').split(',')[0])
            lon1=float(edmPlaceLongitude.replace('"','').split(',')[0])
        if raw_data:  # Nur Einträge mit Daten verarbeiten
            corrected_data = []
            raw_data = raw_data.strip()  # Entfernen von überflüssigen Leerzeichen
            raw_data = str(raw_data).replace('[', '').replace(']', '')  # Entfernen von Zeilenumbrüchen
            # JSON-Daten korrigieren
            if not raw_data.startswith("["):  # Prüfen, ob es kein Array ist
                raw_data = raw_data.replace("}{", "},{")  # In ein Array umwandeln

            try:
                corrected_data = json.loads(raw_data)  # JSON validieren und laden
            except json.JSONDecodeError:
                # Wenn das JSON ungültig ist, versuchen, es zu reparieren
                corrected_items = []
                for item in raw_data.split('},{'):
                    item = item.strip()
                    item = item.replace('\'', '"')

                    textitem=""
                    geoitem=""
                    lat2=0
                    lat2=0
                    if '"result":' in item:
                        item=item.split('"result":')
                        try:
                        # Bereinigung der `result`-Zeichenkette
                            item[1] = item[1].replace('\n', '').replace('"', '#').replace('{', '').replace('}', '').replace(']', '').replace('Please begin your answer with the name of the place and be as exact as possible.','') 
                            item[1] = re.sub(r'[^a-zA-Z0-9,.\s]', '', item[1])  # Entfernt Sonderzeichen
                            item[1] = item[1].strip()
                            if ', location' in item[1]:
                                itemtempe = item[1].split(', location')
                                while '  ' in itemtempe[0]:
                                    itemtempe[0] = itemtempe[0].replace('  ', ' ')
                                itemtempf=itemtempe[0].split(' ')
                                if 'GeoCLIP' in item[0]:
                                    address = geolocator.geocode(itemtempe[1].strip())
                                    try:        
                                        lat2=address.latitude
                                        lon2=address.longitude
                                        if(edmPlaceLatitude and edmPlaceLongitude):
                                            distance = haversine_distance(lat1, lon1, lat2, lon2)
                                        textitem=(item[0]+'"latitude":'+str(address.latitude)+', "longitude":'+str(address.longitude)+',"distance_(km)":'+str(distance)+', "location":"'+itemtempe[1].strip()+'"}').replace(',,',',')
                                    except:
                                        textitem=(item[0]+'"latitude":"", "longitude":"", "location":"'+itemtempe[1].strip()+'"}').replace(',,',',')
                                else:
                                    lat2=float((itemtempf[0]).replace(',','').strip())
                                    lon2=float((itemtempf[1]).replace(',','').strip())
                                    if(edmPlaceLatitude and edmPlaceLongitude):
                                        distance = haversine_distance(lat1, lon1, lat2, lon2)
                                    textitem=(item[0]+'"latitude":'+itemtempf[0]+', "longitude":'+itemtempf[1]+',"distance_(km)":'+str(distance)+', "location":"'+itemtempe[1].strip()+'"}').replace(',,',',')
                            else:
                                textitem = item[0] + '"result":"' + item[1][:200] + '"}'
                        except Exception as e:
                            print(f"Fehler bei der Verarbeitung von `result`: {e}")
                            textitem = item[0]
                    if '"address":' in item:
                        try:
                            itemtempa=item.split('"address":')
                            #itemtempa[1]=itemtempa[1].replace('"','\'')
                            if '"lat":' in itemtempa[1]:
                                itemtempb = itemtempa[1].split('"lat":')
                            elif '"latitude":' in itemtempa[1]:
                                itemtempb = itemtempa[1].split('"latitude":')
                        
                            # Extrahiere den Wert für "lng" oder "longitude"
                            if '"lng":' in itemtempb[1]:
                                itemtempc = itemtempb[1].split('"lng":')
                            elif '"longitude":' in itemtempb[1]:
                                itemtempc = itemtempb[1].split('"longitude":')
                            itemtempd=itemtempc[1].split(',')
                            lat2=float(itemtempc[0].replace(',','').strip())
                            lon2=float(itemtempd[0].replace('}','').strip())
                            if(edmPlaceLatitude and edmPlaceLongitude):
                                distance = haversine_distance(lat1, lon1, lat2, lon2)
                            geoitem=itemtempa[0]+'"latitude":'+itemtempc[0].replace(',','').strip()+',"longitude":'+itemtempd[0].replace('}','').strip()+',"distance_(km)":'+str(distance)+"}"
                            
                        except:
                            geoitem=item[0]
        
                    if len(geoitem + textitem) > 10:
                        item = geoitem + textitem
                    else: 
                        item=item
                    if not item.startswith("{"):
                        item = "{" + item
                    if not item.endswith("}"):
                        item = item + "}"
                    
                    try:
                        corrected_items.append(json.loads(item))
                    except json.JSONDecodeError:
                        print(f"Fehler beim Parsen des Elements bei ID {record_id}: {item}")
                        continue
                corrected_data = corrected_items
                print(f"Corrected Item {record_id}: "+corrected_data.__str__())

                # Korrigierte Daten speichern
                corrected_records.append({
                    "id": record_id,
                    "location_data": original_json_data,
                    "geocoded_data": corrected_data
                })
            # Korrigierte Daten zurückschreiben
            corrected_json = json.dumps(corrected_data, ensure_ascii=False)
            #update_query = "UPDATE object SET aiPlaceLabel = ? WHERE id = ?;"

    # Änderungen speichern und Verbindung schließen
    output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_data.json")
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(corrected_records, f, ensure_ascii=False, indent=4)

    print(f"Korrigierte Daten wurden in {output_file} gespeichert.")
    conn.close()

main()


### Cascading Geolocalization for all

In [ ]:
# =============================================================================
# STEP: Cascading geolocalisation (all methods)
# Runs the full cascade of image- and text-based geolocalisation across the dataset and records the results.
# writes aiConceptPrefClassifier | -> REPORT-geocoding-cascading_data.json | DB: objaverse.sqlite3.db / sqlite32.db
# =============================================================================

#This is code to correct wrong json entries in the aiPlaceLabel field from Europeana source in the Objaverse database.
import sqlite3
import json
import re
import math
from geopy.geocoders import GoogleV3
#from geo_utilities import *
import matplotlib.pyplot as plt
from maploc.demo import Demo
from maploc.osm.viz import GeoPlotter
from maploc.osm.tiling import TileManager
from maploc.osm.viz import Colormap, plot_nodes
from maploc.utils.viz_2d import plot_images
from PIL import Image

from maploc.utils.viz_localization import (
    likelihood_overlay,
    plot_dense_rotations,
    add_circle_inset,
)
from maploc.utils.viz_2d import features_to_RGB
from geopy.geocoders import Nominatim
import requests
from openai import OpenAI
import os
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from geo_utilities import *
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import urllib.request, json, requests
from keras.preprocessing import image
from keras.preprocessing.image import *
from keras.models import load_model

google_key = os.environ["GOOGLE_MAPS_API_KEY"]
DATABASE = 'objaverse.sqlite3.db'
geolocator = Nominatim(user_agent="geo_resolver", timeout=10)
#Predictor for the 3D object datset

dbvaluewrite='aiConceptPrefClassifier'

DBTABLE = "object"
DATABASE = 'Tutorials/objaverse.sqlite32.db'
#modelname=os.path.join(KERAS_DIR, "object/3D-architecture-exterior-vgg16")

uidlist=[]
j=0

def localize_image(address, image_path="requests.jpg", tile_size_meters=512, device='cpu', num_rotations=128):
    """
    Führt die Lokalisierung eines Bildes durch und gibt die Koordinaten und Adresse zurück.

    Args:
        address (str): Die Adresse als Standortprior.
        image_path (str): Pfad zum Eingabebild.
        tile_size_meters (int): Größe der Kachel in Metern. Standard ist 128.
        device (str): Gerät für die Berechnung ('cpu' oder 'cuda'). Standard ist 'cpu'.
        num_rotations (int): Anzahl der Rotationen für die Demo. Standard ist 128.

    Returns:
        tuple: Koordinaten (Latitude, Longitude) und die Adresse des lokalisierten Punktes.
    """
    # Initialisiere Demo
    demo = Demo(num_rotations=num_rotations, device=device)
    print(f"Using image {image_path} with location prior '{address}'")

    # Lese Eingabebild und Metadaten
    image, camera, gravity, proj, bbox = demo.read_input_image(
        image_path,
        prior_address=address,
        tile_size_meters=tile_size_meters,
    )

    # Abfrage von OpenStreetMap für diesen Bereich
    tiler = TileManager.from_bbox(proj, bbox + 10, demo.config.data.pixel_per_meter)
    canvas = tiler.query(bbox)

    # Erstelle Kartenvisualisierung
    map_viz = Colormap.apply(canvas.raster)

    # Führe die Lokalisierung durch
    uv, yaw, prob, neural_map, image_rectified = demo.localize(
        image, camera, canvas, gravity=gravity
    )

    # Berechne Koordinaten
    coords = proj.unproject(canvas.to_xy(uv))

    # Interaktive Karte erstellen
    bbox_ll = proj.unproject(canvas.bbox)
    plot = GeoPlotter(zoom=16.5)
    plot.raster(map_viz, proj.unproject(bbox), opacity=0.5)
    plot.raster(likelihood_overlay(prob.numpy().max(-1)), proj.unproject(bbox))
    plot.points(proj.latlonalt[:2], "red", name="location prior", size=10)
    plot.points(proj.unproject(canvas.to_xy(uv)), "black", name="argmax", size=10)
    plot.bbox(proj.unproject(bbox), "blue", name="map tile")

    # Ausgabe der Ergebnisse
    print(yaw)
    print(coords)

    # Adresse mit Geopy abrufen
    geolocator = Nominatim(user_agent="4DS")
    coords = (float(coords[0]), float(coords[1]))
    address_segment = (geolocator.reverse(coords, language="en").address).split(',')

    return coords, address_segment

def reverse_geocode(lat, lon):
    """
    Resolve coordinates into location details using Nominatim.
    """
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True, language="en")
        if location and location.raw:
            address = location.raw.get("address", {})
            return {
                "country": address.get("country"),
                "region": address.get("state"),
                "city": address.get("city") or address.get("town") or address.get("village"),
                "street": address.get("road")
            }
    except GeocoderTimedOut:
        print(f"Geocoding timed out for coordinates: ({lat}, {lon})")
    return {"country": None, "region": None, "city": None, "street": None}

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Berechnet die Entfernung zwischen zwei Punkten auf der Erde (in Kilometern) basierend auf ihren Breitengraden und Längengraden.
    
    :param lat1: Breitengrad des ersten Punkts
    :param lon1: Längengrad des ersten Punkts
    :param lat2: Breitengrad des zweiten Punkts
    :param lon2: Längengrad des zweiten Punkts
    :return: Entfernung in Kilometern
    """
    # Radius der Erde in Kilometern
    R = 6371.0

    # Koordinaten in Bogenmaß umwandeln
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Unterschiede berechnen
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # Haversine-Formel
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Entfernung berechnen
    distance = R * c
    return distance

def main():
    conn = sqlite3.connect(DATABASE)
    cursor = conn.cursor()

    # Alle Einträge aus der Tabelle `object` abrufen
    output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-geocoding-cascading_data.json")
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump('Header', f, ensure_ascii=False, indent=4)

    query = 'SELECT id, aiPlaceLabel, edmPlaceLabel, edmPlaceLatitude, edmPlaceLongitude FROM object WHERE "source" like "%Europeana%" LIMIT 10000;'
    cursor.execute(query)
    results = cursor.fetchall()

    # Liste für korrigierte Daten
    corrected_records = []

    # Durch alle Einträge iterieren
    for record in results:
        flag=0
        location = ""
        modeltemp=["","",""]
        lat=["","",""]
        lon=["","",""]
        modeltemp2=""
        lat2=0
        lon2=0
        record_id, raw_data, edmPlaceLabel, edmPlaceLatitude, edmPlaceLongitude = record

        if edmPlaceLabel and edmPlaceLatitude and edmPlaceLongitude:
            original_json_data = {
                "location": edmPlaceLabel.replace('"','').strip(),
                "latitude": float(edmPlaceLatitude.replace('"','').split(',')[0]),
                "longitude": float(edmPlaceLongitude.replace('"','').split(',')[0])
            }
        elif  edmPlaceLabel:
            try:
                address = geolocator.geocode(edmPlaceLabel)
                original_json_data = {    
                    "location": edmPlaceLabel.replace('"','').strip(),
                    "latitude": address.latitude,
                    "longitude": address.longitude
                }
            except:
                original_json_data = {    
                    "location": edmPlaceLabel.replace('"','').strip(),
                    "latitude": "",
                    "longitude": ""
                }
        if(edmPlaceLatitude and edmPlaceLongitude):
            lat1=float(edmPlaceLatitude.replace('"','').split(',')[0])
            lon1=float(edmPlaceLongitude.replace('"','').split(',')[0])
        if raw_data:  # Nur Einträge mit Daten verarbeiten
            corrected_data = []
            raw_data = raw_data.strip()  # Entfernen von überflüssigen Leerzeichen
            raw_data = str(raw_data).replace('[', '').replace(']', '')  # Entfernen von Zeilenumbrüchen
            # JSON-Daten korrigieren
            if not raw_data.startswith("["):  # Prüfen, ob es kein Array ist
                raw_data = raw_data.replace("}{", "},{")  # In ein Array umwandeln

            try:
                corrected_data = json.loads(raw_data)  # JSON validieren und laden
            except json.JSONDecodeError:
                # Wenn das JSON ungültig ist, versuchen, es zu reparieren
                corrected_items = []
                for item in raw_data.split('},{'):
                    item = item.strip()
                    item = item.replace('\'', '"')

                    textitem=""
                    geoitem=""
           
                    if '"result":' in item:
                        item=item.split('"result":')
                        try:
                        # Bereinigung der `result`-Zeichenkette
                            item[1] = item[1].replace('\n', '').replace('"', '#').replace('{', '').replace('}', '').replace(']', '').replace('Please begin your answer with the name of the place and be as exact as possible.','') 
                            item[1] = re.sub(r'[^a-zA-Z0-9,.\s]', '', item[1])  # Entfernt Sonderzeichen
                            item[1] = item[1].strip()
                            
                            if ', location' in item[1]:
                                itemtempe = item[1].split(', location')
                                while '  ' in itemtempe[0]:
                                    itemtempe[0] = itemtempe[0].replace('  ', ' ')
                                itemtempf=itemtempe[0].split(' ')
                                if 'GeoCLIP' in item[0]:
                                    address = ""
                                    
                                else:
                                    lat2=float((itemtempf[0]).replace(',','').strip())
                                    lon2=float((itemtempf[1]).replace(',','').strip())
                                    if(edmPlaceLatitude and edmPlaceLongitude):
                                        distance = haversine_distance(lat1, lon1, lat2, lon2)
                                    textitem=(item[0]+'"latitude":'+itemtempf[0]+', "longitude":'+itemtempf[1]+',"distance_(km)":'+str(distance)+', "location":"'+itemtempe[1].strip()+'"}').replace(',,',',')
                            else:
                                textitem = item[0] + '"result":"' + item[1][:200] + '"}'
                        except Exception as e:
                            print(f"Fehler bei der Verarbeitung von `result`: {e}")
                            textitem = item[0]
                    if '"address":' in item:
                        try:
                            itemtempa=item.split('"address":')
                            #itemtempa[1]=itemtempa[1].replace('"','\'')
                            if '"lat":' in itemtempa[1]:
                                itemtempb = itemtempa[1].split('"lat":')
                            elif '"latitude":' in itemtempa[1]:
                                itemtempb = itemtempa[1].split('"latitude":')
                        
                            # Extrahiere den Wert für "lng" oder "longitude"
                            if '"lng":' in itemtempb[1]:
                                itemtempc = itemtempb[1].split('"lng":')
                            elif '"longitude":' in itemtempb[1]:
                                itemtempc = itemtempb[1].split('"longitude":')
                            itemtempd=itemtempc[1].split(',')
                            lat2=float(itemtempc[0].replace(',','').strip())
                            lon2=float(itemtempd[0].replace('}','').strip())
                            if(edmPlaceLatitude and edmPlaceLongitude):
                                distance = haversine_distance(lat1, lon1, lat2, lon2)
                            geoitem=itemtempa[0]+'"latitude":'+itemtempc[0].replace(',','').strip()+',"longitude":'+itemtempd[0].replace('}','').strip()+',"distance_(km)":'+str(distance)+"}"
                            
                        except:
                            geoitem=item[0]
        
                    if len(geoitem + textitem) > 10:
                        item = geoitem + textitem
                    else: 
                        item=str(item)
                    if not item.startswith("{"):
                        item = "{" + item
                    if not item.endswith("}"):
                        item = item + "}"
                    
                    try:
                        
                        corrected_items=(json.loads(item))
                        
                   

                        if "processor" in corrected_items and "GEOCODING"in corrected_items['processor'] and flag==0:
                            
                            if "model" in corrected_items and "Qwen/Qwen3-32B-MLX-4bit "in corrected_items['model']: 
                                #if "result" in corrected_items and 'None' not in corrected_items['result'] and 'NONE' not in corrected_items['result'] and 'No result' not in corrected_items['result']:
                                
                                if "latitude" in corrected_items and "longitude" in corrected_items:
                                    lat[0]=corrected_items['latitude']
                                    lon[0]=corrected_items['longitude']
                                    modeltemp[0]=corrected_items['model']  
                            
                            elif "model" in corrected_items and "Qwen3-14B-MLX"in corrected_items['model'] and flag==0:
                                #if "result" in corrected_items and 'None' not in corrected_items['result'] and 'NONE' not in corrected_items['result'] and 'No result' not in corrected_items['result']:
                                if "latitude" in corrected_items and "longitude" in corrected_items:
                                    lat[1]=corrected_items['latitude']
                                    lon[1]=corrected_items['longitude']
                                    modeltemp[1]=corrected_items['model']
                             
                            elif "model" in corrected_items and "Deepseek"in corrected_items['model'] and flag==0:
                                #if "result" in corrected_items and 'None' not in corrected_items['result'] and 'NONE' not in corrected_items['result'] and 'No result' not in corrected_items['result']:
                                if "latitude" in corrected_items and "longitude" in corrected_items:
                                    lat[2]=corrected_items['latitude']
                                    lon[2]=corrected_items['longitude']
                                    modeltemp[2]=corrected_items['model']
                    except json.JSONDecodeError:
                        print(f"Fehler beim Parsen des Elements bei ID {record_id}: {item}")
                        continue

                    if isinstance(modeltemp, (list, tuple)) and modeltemp[2] != "":
                        lat2=lat[2]
                        lon2=lon[2]
                        modeltemp2=modeltemp[2]
          
                    elif isinstance(modeltemp, (list, tuple)) and modeltemp[1] != "":
                        lat2=lat[1]
                        lon2=lon[1]
                        modeltemp2=modeltemp[1]
                    elif isinstance(modeltemp, (list, tuple)) and modeltemp[0] != "":
                        lat2=lat[0]
                        lon2=lon[0]
                        modeltemp2=modeltemp[0]
                    
            if modeltemp2 != "" and lat2 != 0 and lon2 != 0:
                print(f"Finale Koordinaten: Latitude: {lat2}, Longitude: {lon2}, Modell: {modeltemp2}")

            
        
            if(lat2 and lon2 and lat1 and lon1):
                distance = haversine_distance(lat1, lon1, lat2, lon2)
                print(str(distance)+" km") 

                if location and location.raw:
                    address_data = location.raw.get("address", {})
                    address = {
                        "street": address_data.get("road"),
                        "city": address_data.get("city") or address_data.get("town") or address_data.get("village"),
                        "region": address_data.get("state"),
                        "country": address_data.get("country")
                    }
            
                    # Erstelle eine lesbare Adresse als String
                    formatted_address = f"{address.get('street', '')}, {address.get('city', '')}, {address.get('country', '')}"
                    print(formatted_address)
            
                
                corrected_datatemp = {
                    "model": modeltemp,
                    "distance_(km)": distance
                }

                corrected_data = json.loads(json.dumps(corrected_datatemp))

                    
                # Füge die Daten zu corrected_records hinzu
                corrected_records.append({
                    "id": record_id,
                    "location_data": original_json_data,
                    "geocoded_data": corrected_data
                })
                
                # Adresse und Bildpfad definieren
                image_path = 'https://data.3drepo.eu/thumbnail/'+str(record_id)+'.jpg'
                print(image_path)

                #geolocator = Nominatim(user_agent="4DBrowser.eu", timeout=10)
                #location = geolocator.reverse((lat1, lon1), language="en")
                # response = requests.get(image_path)

                # Überprüfen, ob der Download erfolgreich war
                
                        
            corrected_json = json.dumps(corrected_data, ensure_ascii=False)
                    #update_query = "UPDATE object SET aiPlaceLabel = ? WHERE id = ?;"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(corrected_records, f, ensure_ascii=False, indent=4)
                    
            # Änderungen speichern und Verbindung schließen

    print(f"Korrigierte Daten wurden in {output_file} gespeichert.")
    conn.close()

main()

        


### Export Geocodings to CSV data

In [ ]:
# =============================================================================
# STEP: Export geocoding distances to CSV
# Evaluates cascaded geocoding distances against ground truth and writes summary CSVs.
# -> REPORT-cascading-distance_summary-europeana.csv, REPORT-distance_summary-all.csv
# =============================================================================

# Export Geocodings to CSV data
import json
import csv

def evaluate_distances(json_file, output_csv):
    # Define intervals
    intervals = {
        "0-20 km": (0, 20),   
        "21-50 km": (21, 50),   
        "51-100 km": (51, 100),
        "101-200 km": (101, 200),
        "201-1000 km": (201, 1000),
        "1001-2000 km": (1001, 2000),
        "2001-3000 km": (2001, 3000),
        "3000+ km": (3001, float('inf'))
    }

    # Initialize results dictionary
    results = {model: {interval: 0 for interval in intervals} for model in set()}

    # Load JSON data
    with open(json_file, 'r') as file:
        data = json.load(file)

    # Process data
    for entry in data:
        if "geocoded_data" in entry:
            for geocode in entry["geocoded_data"]:
                model=(entry["geocoded_data"]["model"])
                distance=(entry["geocoded_data"]["distance_(km)"])
            
                #model = geocode.get("model", "Unknown")
                #distance = geocode.get("distance_(km)", None)
                if distance is not None:
                    # Determine the interval
                    for interval, (low, high) in intervals.items():
                        if low <= distance <= high:
                            if model not in results:
                                results[model] = {key: 0 for key in intervals}
                            results[model][interval] += 1
                            break

    # Write results to CSV
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # Write header
        header = ["Model"] + list(intervals.keys())
        writer.writerow(header)
        # Write data
        for model, interval_data in results.items():
            row = [model] + [interval_data[interval] for interval in intervals]
            writer.writerow(row)

# Example usage
evaluate_distances(os.path.join(PROJECT_ROOT, "Reports/REPORT-geocoding-cascading_data.json"), os.path.join(PROJECT_ROOT, "Reports/REPORT-cascading-distance_summary-europeana.csv"))
#evaluate_distances(os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_data.json"), os.path.join(PROJECT_ROOT, "Reports/REPORT-distance_summary-all.csv"))


### Export Geocodings to CSV data / 5 km steps

In [ ]:
# =============================================================================
# STEP: Export geocoding distances (5 km bins)
# Same distance evaluation aggregated into 5 km buckets.
# -> REPORT-cascading-distance_summary-europeana.csv
# =============================================================================

import json
import csv

def evaluate_distances(json_file, output_csv):
    # Define intervals in 5 km steps up to 1000 km
    intervals = {}
    step = 5
    max_distance = 1000
    for i in range(0, max_distance, step):
        start = i + 1 if i > 0 else 0
        end = i + step
        intervals[f"{start}-{end} km"] = (start, end)
    intervals[f"{max_distance + 1}+ km"] = (max_distance + 1, float('inf'))

    # Initialize results dictionary
    results = {model: {interval: 0 for interval in intervals} for model in set()}

    # Load JSON data
    with open(json_file, 'r') as file:
        data = json.load(file)

    # Process data
    for entry in data:
        if "geocoded_data" in entry:
            for geocode in entry["geocoded_data"]:
                model = entry["geocoded_data"]["model"]
                distance = entry["geocoded_data"]["distance_(km)"]
                if isinstance(model, list):  # Wenn model eine Liste ist
                    model = tuple(model)  # In ein Tupel umwandeln
                if distance is not None:
                    # Determine the interval
                    for interval, (low, high) in intervals.items():
                        if low <= distance <= high:
                            if model not in results:
                                results[model] = {key: 0 for key in intervals}
                            results[model][interval] += 1
                            break

    # Write results to CSV
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # Write header
        header = ["Model"] + list(intervals.keys())
        writer.writerow(header)
        # Write data
        for model, interval_data in results.items():
            row = [model] + [interval_data[interval] for interval in intervals]
            writer.writerow(row)

# Example usage
evaluate_distances(os.path.join(PROJECT_ROOT, "Reports/REPORT-geocoding-cascading_data.json"), os.path.join(PROJECT_ROOT, "Reports/REPORT-cascading-distance_summary-europeana.csv"))


### Geocode location and compare with groundtruth data by country, region, city, name

In [ ]:
# =============================================================================
# STEP: Geocode & compare to ground truth by place
# Reverse-geocodes results and compares country / region / city / name against ground-truth data.
# -> REPORT-geocoding-by-name-comparison_results.csv/.json
# =============================================================================

import csv
import json
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# Initialize the geocoder with English language preference
geolocator = Nominatim(user_agent="geo_resolver", timeout=10)

def reverse_geocode(lat, lon):
    """
    Resolve coordinates into location details using Nominatim.
    """
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True, language="en")
        if location and location.raw:
            address = location.raw.get("address", {})
            return {
                "country": address.get("country"),
                "region": address.get("state"),
                "city": address.get("city") or address.get("town") or address.get("village"),
                "street": address.get("road")
            }
    except GeocoderTimedOut:
        print(f"Geocoding timed out for coordinates: ({lat}, {lon})")
    return {"country": None, "region": None, "city": None, "street": None}

def process_json(input_file, output_file, csv_file):
    """
    Process a JSON file, reverse geocode coordinates, compare results, and save the updated data and comparison results.
    """
    # Load JSON data
    with open(input_file, 'r') as file:
        data = json.load(file)

    comparison_results = []
    j=0
    # Process each entry in the JSON
    for entry in data:
        j+=1
        print(f"Processing entry {j}")
        # Process location_data
        location_data = entry.get("location_data", {})
        lat = location_data.get("latitude")
        lon = location_data.get("longitude")

        resolved_location = {}
        if lat is not None and lon is not None:
            resolved_location = reverse_geocode(lat, lon)
            location_data.update(resolved_location)

        # Process geocoded_data
        geocoded_data = entry.get("geocoded_data", [])
        for geocode in geocoded_data:
            lat = geocode.get("latitude")
            lon = geocode.get("longitude")

            if lat is not None and lon is not None:
                resolved_geocode = reverse_geocode(lat, lon)
                geocode.update(resolved_geocode)

                # Compare resolved geocode with resolved location
                comparison = {
                    "model": entry.get("model", "Unknown"),
                    "location_lat": location_data.get("latitude"),
                    "location_lon": location_data.get("longitude"),
                    "geocode_lat": geocode.get("latitude"),
                    "geocode_lon": geocode.get("longitude"),
                    "location_country": resolved_location.get("country"),
                    "geocode_country": resolved_geocode.get("country"),
                    "country_match": resolved_geocode.get("country") == resolved_location.get("country"),
                    "location_region": resolved_location.get("region"),
                    "geocode_region": resolved_geocode.get("region"),
                    "region_match": resolved_geocode.get("region") == resolved_location.get("region"),
                    "location_city": resolved_location.get("city"),
                    "geocode_city": resolved_geocode.get("city"),
                    "city_match": resolved_geocode.get("city") == resolved_location.get("city"),
                    "location_street": resolved_location.get("street"),
                    "geocode_street": resolved_geocode.get("street"),
                    "street_match": resolved_geocode.get("street") == resolved_location.get("street"),
                }
                comparison_results.append(comparison)

    # Save the updated JSON to a new file
    with open(output_file, 'w') as file:
        json.dump(data, file, indent=4)

    # Save comparison results to a CSV file
    with open(csv_file, 'w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=comparison_results[0].keys())
        writer.writeheader()
        writer.writerows(comparison_results)

    print(f"Comparison results saved to {csv_file}")

# Example usage
process_json(
    os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_data.json"),
    os.path.join(PROJECT_ROOT, "Reports/REPORT-geocoding-by-name-comparison_results.json"),
    os.path.join(PROJECT_ROOT, "Reports/REPORT-geocoding-by-name-comparison_results.csv")
)


In [ ]:
# =============================================================================
# STEP: Geocode & compare to ground truth (variant)
# Variant of the place-level ground-truth comparison.
# -> REPORT-geocoding-by-name-comparison_results.csv/.json
# =============================================================================

import csv
import json
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

input_file=os.path.join(PROJECT_ROOT, "Reports/REPORT-geocoding-by-name-comparison_results.json")
csv_file=os.path.join(PROJECT_ROOT, "Reports/REPORT-geocoding-by-name-comparison_results.csv")

def process_json():
    """
    Process a JSON file, reverse geocode coordinates, compare results, and save the updated data and comparison results.
    """
    # Load JSON data
    with open(input_file, 'r') as file:
        data = json.load(file)

    comparison_results = []
    j=0
    # Process each entry in the JSON
    for entry in data:
        j+=1
        print(f"Processing entry {j}")
        # Process location_data
        location_data = entry.get("location_data", {})
        lat = location_data.get("latitude")
        lon = location_data.get("longitude")

        # Process geocoded_data
        geocoded_data = entry.get("geocoded_data", [])
        for geocode in geocoded_data:
            if lat is not None and lon is not None:
                if geocode.get("latitude") is not None or geocode.get("longitude") is not None:
                    # Compare resolved geocode with resolved location
                    comparison = {
                        "model": geocode.get("model"),
                        "processor": geocode.get("processor"),
                        "location_lat": location_data.get("latitude"),
                        "location_lon": location_data.get("longitude"),
                        "geocode_lat": geocode.get("latitude"),
                        "geocode_lon": geocode.get("longitude"),
                        "location_country": location_data.get("country"),
                        "geocode_country": geocode.get("country"),
                        "location_region": location_data.get("region"),
                        "geocode_region": geocode.get("region"),
                        "location_city": location_data.get("city"),
                        "geocode_city": geocode.get("city"),
                        "location_street": location_data.get("street"),
                        "geocode_street": geocode.get("street"),
                    }

                    # Handle country match
                    if geocode.get("country") is not None:
                        comparison["country_match"] = geocode.get("country") == location_data.get("country")
                    else:
                        comparison["country_match"] = None

                    # Handle region match
                    if geocode.get("region") is not None:
                        comparison["region_match"] = geocode.get("region") == location_data.get("region")
                    else:
                        comparison["region_match"] = None

                    # Handle city match
                    if geocode.get("city") is not None:
                        comparison["city_match"] = geocode.get("city") == location_data.get("city")
                    else:
                        comparison["city_match"] = None

                    # Handle street match
                    if geocode.get("street") is not None:
                        comparison["street_match"] = geocode.get("street") == location_data.get("street")
                    else:
                        comparison["street_match"] = None

                    comparison_results.append(comparison)
                    print(comparison)
    
    # Save comparison results to a CSV file
    with open(csv_file, 'w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=comparison_results[0].keys())
        writer.writeheader()
        writer.writerows(comparison_results)

    print(f"Comparison results saved to {csv_file}")

process_json()


### Export processing times for geocodings

In [ ]:
# =============================================================================
# STEP: Export geocoding processing times
# Aggregates per-model geocoding inference times for the energy report.
# -> REPORT-energy_geocoding-data.csv
# =============================================================================

import json
from statistics import mean, median

file_path = os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_data.json")
output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-energy_geocoding-data.csv")

# JSON-Daten laden
with open(file_path, "r") as file:
    data = json.load(file)

# Dictionary to store times grouped by processor and model
processor_model_times = {}

# Process data
for entry in data:
    for item in entry.get("geocoded_data", []):
        if 'inferencetime' in item:
            time = item['inferencetime']
            processor = item.get('processor', 'N/A')
            model = item.get('model', 'N/A')

            # Group times by processor and model
            if processor not in processor_model_times:
                processor_model_times[processor] = {}
            if model not in processor_model_times[processor]:
                processor_model_times[processor][model] = []
            processor_model_times[processor][model].append(time)

# Write results to the output file
with open(output_file, "w", encoding="utf-8") as f:
    f.write("Processor; Model; Min Time; Mean Time; Median Time; Max Time \n")
    for processor, models in processor_model_times.items():
        for model, times in models.items():
            min_time = str(min(times)).replace('.',',')
            max_time = str(max(times)).replace('.',',')
            mean_time = str(mean(times)).replace('.',',')
            median_time = str(median(times)).replace('.',',')
            f.write(f"{processor}; {model}; {min_time}; {mean_time}; {median_time}; {max_time}\n")


### This is code to correct wrong json entries in the aiConceptPrefLabel from Europeana source.

In [ ]:
# =============================================================================
# STEP: Correct stored concept-classification JSON
# Repairs malformed aiConceptPrefLabel JSON for Objaverse records.
# -> REPORT-corrected_concept-data-objaverse-db3.json | DB: objaverse.sqlite32.db
# =============================================================================

#This is code to correct wrong json entries in the aiLabel field from Europeana source in the Objaverse database.
import sqlite3
import json
import re
import math
from geopy.geocoders import GoogleV3

google_key = os.environ["GOOGLE_MAPS_API_KEY"]
DATABASE = 'Tutorials/objaverse.sqlite32.db'

def main():
    # Verbindung zur SQLite-Datenbank herstellen
    conn = sqlite3.connect(DATABASE)
    cursor = conn.cursor()

    # Alle Einträge aus der Tabelle `object` abrufen
    query = 'SELECT id, aiConceptPrefLabel FROM object WHERE "source" like "%Objaverse%";'
    cursor.execute(query)
    results = cursor.fetchall()

    # Liste für korrigierte Daten
    corrected_records = []

    # Durch alle Einträge iterieren
    for record in results:
        record_id, raw_data = record

        if raw_data:  # Nur Einträge mit Daten verarbeiten
            corrected_data = []
            raw_data = raw_data.strip()  # Entfernen von überflüssigen Leerzeichen
            raw_data = str(raw_data).replace('[', '').replace(']', '')  # Entfernen von Zeilenumbrüchen
            # JSON-Daten korrigieren
            raw_data = raw_data.replace("}''{", '},{') 
            raw_data = raw_data.replace("}'{", '},{') 
            raw_data = raw_data.replace("}None{", '},{') 
            if raw_data.startswith("'{"):
                    raw_data = raw_data[1:]
            if raw_data.startswith('"{'):
                    raw_data = raw_data[1:]
            print(raw_data)
            if not raw_data.startswith("["):  # Prüfen, ob es kein Array ist
                raw_data = raw_data.replace("}{", "},{")  # In ein Array umwandeln
                 # In ein Array umwandeln
            try:
                corrected_data = json.loads(raw_data)  # JSON validieren und laden
            except json.JSONDecodeError:
                # Wenn das JSON ungültig ist, versuchen, es zu reparieren
                corrected_items = []
                for item in raw_data.split('},{'):
                    item = item.strip()
        
                    if "'result': " in item:
                        item=item.split("'result': ")
                        try:
                        # Bereinigung der `result`-Zeichenkette
                            item[1] = item[1].replace('\n', '').replace('"', '#').replace('{', '').replace('}', '').replace(']', '').replace('Please begin your answer with the name of the place and be as exact as possible.','') 
                            item[1] = re.sub(r'[^a-zA-Z,.\s]', '', item[1])  # Entfernt Sonderzeichen
                            item[1] = item[1].strip()
                            item[1] = item[1].replace('.,', '').replace(', .', '')
                            if 'nTop predictionsn' in item[1]:
                                item[1] = item[1].replace('nTop predictionsn','')
                                item[1] = item[1].replace('.',',')
                                item[1] = re.sub(r'\s*,\s*$', '', item[1])  # Entfernt das letzte Komma
                                item[1] = re.sub(r'\s*,\s*', ', ', item[1])  # Bereinigt Leerzeichen um Kommas
                                item[1] = re.sub(r'\s+', ' ', item[1])  # Reduziert mehrere Leerzeichen auf eines

                            print(item[1])
                            textitem = item[0] + "'result':'" + item[1][:200] + "'}"
                        except Exception as e:
                            print(f"Fehler bei der Verarbeitung von `result`: {e}")
                            textitem = item[0]
                        item=textitem
                    item = item.replace('\'', '"')

                
                    #if item.endswith('"'):
                    #    item = item[:-1]
                    

                    if not item.startswith("{"):
                        item = "{" + item
                    if not item.endswith("}"):
                        item = item + "}"
                    
                    try:
                        corrected_items.append(json.loads(item))
                    except json.JSONDecodeError:
                        print(f"Fehler beim Parsen des Elements bei ID {record_id}: {item}")
                        continue
                corrected_data = corrected_items
                print(f"Corrected Item {record_id}: "+corrected_data.__str__())

                # Korrigierte Daten speichern
                corrected_records.append({
                    "id": record_id,
                    "categorized_data": corrected_data
                })

            # Korrigierte Daten zurückschreiben
            #corrected_json = json.dumps(corrected_data, ensure_ascii=False)
            #update_query = "UPDATE object SET aiConceptPrefLabel = ? WHERE id = ?;"

    # Änderungen speichern und Verbindung schließen
    output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_concept-data-objaverse-db3.json")
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(corrected_records, f, ensure_ascii=False, indent=4)

    print(f"Korrigierte Daten wurden in {output_file} gespeichert.")
    conn.close()

main()


### Export Classifications as CSV

In [ ]:
# =============================================================================
# STEP: Export classifications to CSV
# Extracts concept categories and writes the classification cross-comparison CSV (Objaverse).
# -> REPORT-cross-comparison_concept-data-objaverse-db3.csv
# =============================================================================

import json
from itertools import combinations

# Europeanacategories-Liste
europeanacategories = [
    'Building', 'Archaeological Site', 'Cartoon', 'Ceramics', 'Clothing',
    'Costume Accessories', 'Drawing', 'Map', 'Furniture', 'Textile', 'Food',
    'Glassware', 'Inscription', 'Jewellery', 'Metalwork', 'Machinery', 'Medal',
    'Memorabilia', 'Mineral', 'Musical Instrument', 'Painting', 'Photograph',
    'Postcard', 'Poster', 'Print', 'Sculpture', 'Specimen', 'Tableware', 'Tool',
    'Tapestry', 'Toy', 'Weaponry', 'Woodwork', 'Stamp'
]

file_path = os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_concept-data-objaverse-db3.json")
output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-cross-comparison_concept-data-objaverse-db3.csv")
# Funktion zum Extrahieren und Normalisieren der Kategorien
def extract_categories(result):
    # Entferne Zahlen und Sonderzeichen, behalte nur Kategorien
    categories = [word.strip() for word in result.split(",") if not any(char.isdigit() for char in word)]
    # Filtere nur gültige Kategorien aus europeanacategories
    return set(cat for cat in categories if cat in europeanacategories)

# JSON-Daten laden
with open(file_path, "r") as file:
    data = json.load(file)

with open(output_file, "w", encoding="utf-8") as f:
    f.write("ID, Processor Comparison, Overlap Count\n")
    # Process data
    for entry in data:
        temp_categories=[]
        for item in entry["categorized_data"]:
            if 'result' in item:
                result = item['result']
                if isinstance(result, str):
                    model = item['model']
                    processor = item['processor']
                    item['result'] = result.strip()
                    itemcategories = extract_categories(item['result'])
                if (len(itemcategories) in {0, 5}) and not any(entry.get('processor') == processor + '+' + model for entry in temp_categories):
                    temp_categories.append({
                        'processor': processor + '+' + model,
                        'itemcategories': itemcategories
                    })
                print(temp_categories)

        for entry1, entry2 in combinations(temp_categories, 2):  # Compare all pairs of entries
            # Extract fields to compare (e.g., categorized_data)
            categorized_data1 = entry1.get("itemcategories", [])
            categorized_data2 = entry2.get("itemcategories", [])

            # Perform comparison (example: find common elements)
            common_elements = set(categorized_data1) & set(categorized_data2)

            # Print or store the results
            print(f"{entry['id']},{entry1.get('processor', 'N/A')} vs. {entry2.get('processor', 'N/A')},{len(common_elements)}, Common Elements: {common_elements}")
            f.write(f"{entry['id']},{entry1.get('processor', 'N/A')} vs. {entry2.get('processor', 'N/A')},{len(common_elements)}\n")

print(f"Korrigierte Daten wurden in {output_file} gespeichert.")


### Classification workflow to compare AI classified categories

In [ ]:
# =============================================================================
# STEP: Classification comparison workflow
# Compares AI-classified categories across sources and exports the cross-comparison CSV.
# -> REPORT-cross-comparison_concept-data-objaverse.csv
# =============================================================================

import json
from itertools import combinations

# Europeanacategories-Liste
europeanacategories = [
    'Building', 'Archaeological Site', 'Cartoon', 'Ceramics', 'Clothing',
    'Costume Accessories', 'Drawing', 'Map', 'Furniture', 'Textile', 'Food',
    'Glassware', 'Inscription', 'Jewellery', 'Metalwork', 'Machinery', 'Medal',
    'Memorabilia', 'Mineral', 'Musical Instrument', 'Painting', 'Photograph',
    'Postcard', 'Poster', 'Print', 'Sculpture', 'Specimen', 'Tableware', 'Tool',
    'Tapestry', 'Toy', 'Weaponry', 'Woodwork', 'Stamp'
]

file_path = os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_concept-data-objaverse-db3.json")#<PROJECT_ROOT>/Reports/REPORT-corrected_concept-data.json"
output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-cross-comparison_concept-data-objaverse.csv")

# Funktion zum Extrahieren und Normalisieren der Kategorien
def extract_categories(result):
    # Entferne Zahlen und Sonderzeichen, behalte nur Kategorien
    categories = [word.strip() for word in result.split(",") if not any(char.isdigit() for char in word)]
    # Filtere nur gültige Kategorien aus europeanacategories
    return set(cat for cat in categories if cat in europeanacategories)

# JSON-Daten laden
with open(file_path, "r") as file:
    data = json.load(file)

with open(output_file, "w", encoding="utf-8") as f:
    f.write("ID; Validity; Common Elements; Processors \n")
    # Process data
    for entry in data:
        temp_categories = []
        category_temp = []
        for item in entry.get("categorized_data", []):
            if 'result' in item:
                result = item['result']
                if isinstance(result, str):
                    model = item.get('model', 'N/A')
                    processor = item.get('processor', 'N/A')
                    item['result'] = result.strip()
                    itemcategories = extract_categories(item['result'])
                if (len(itemcategories) in {0, 5}) and not any(
                    temp_entry.get('processor') == processor + '+' + model for temp_entry in temp_categories
                ):
                    temp_categories.append({
                        'processor': processor + '+' + model,
                        'itemcategories': itemcategories
                    })

        # Schleife über Kombinationen
        valid_found = False
        validity = 0
        j=0
        i=0
        processors = ""
        category_temp = ""
        for combination_size in range(5, 0, -1):  # Iterate from 4 to 1
            for entries in combinations(temp_categories, combination_size):
                j=j+1
            for entries in combinations(temp_categories, combination_size):
                categorized_data_list = [entry.get("itemcategories", []) for entry in entries]
                i=i+1
                common_elements = set.intersection(*map(set, categorized_data_list))
                if len(common_elements) > 0 and len(category_temp) > 0:
                    category_temp+=', '    
                category_temp+=', '.join(common_elements)
                
                
                if len(common_elements) > 0:
                    #    " and ".join(entry.get('processor', 'N/A') for entry in entries))
                    validity = combination_size
                    common_elements = ', '.join(common_elements)
                    if len(common_elements) > 0 and len(processors) > 0:
                        processors+=', '
                    processors+= " and ".join(entry.get('processor', 'N/A') for entry in entries)
                    #category_temp = ', '.join(common_elements)
                    valid_found = True
            
                if valid_found and i == j:
                    break  # Breche die äußere Schleife ab, wenn gültige Elemente gefunden wurden
            if valid_found:  # Breche die äußere Schleife ab, wenn gültige Elemente gefunden wurden
                break
        if valid_found:

            print(f"{entry['id']}; Validity: {str(combination_size)}; Common Elements: {category_temp}; Processors: {processors} ")
            f.write(f"{entry['id']}; {str(combination_size)}; {category_temp}; {processors} ")
            f.write("\n")

print(f"Korrigierte Daten wurden in {output_file} gespeichert.")


### Export classification processing times

In [ ]:
# =============================================================================
# STEP: Export classification processing times
# Aggregates classification inference times for the energy report.
# -> REPORT-energy_concept-data.csv
# =============================================================================

import json
from statistics import mean, median

# Europeanacategories-Liste
europeanacategories = [
    'Building', 'Archaeological Site', 'Cartoon', 'Ceramics', 'Clothing',
    'Costume Accessories', 'Drawing', 'Map', 'Furniture', 'Textile', 'Food',
    'Glassware', 'Inscription', 'Jewellery', 'Metalwork', 'Machinery', 'Medal',
    'Memorabilia', 'Mineral', 'Musical Instrument', 'Painting', 'Photograph',
    'Postcard', 'Poster', 'Print', 'Sculpture', 'Specimen', 'Tableware', 'Tool',
    'Tapestry', 'Toy', 'Weaponry', 'Woodwork', 'Stamp'
]

file_path = os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_concept-data.json")
output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-energy_concept-data.csv")

# Funktion zum Extrahieren und Normalisieren der Kategorien
def extract_categories(result):
    categories = [word.strip() for word in result.split(",") if not any(char.isdigit() for char in word)]
    return set(cat for cat in categories if cat in europeanacategories)

# JSON-Daten laden
with open(file_path, "r") as file:
    data = json.load(file)

# Dictionary to store times grouped by processor and model
processor_model_times = {}

# Process data
for entry in data:
    for item in entry.get("categorized_data", []):
        if 'inferencetime' in item:
            time = item['inferencetime']
            processor = item.get('processor', 'N/A')
            model = item.get('model', 'N/A')

            # Group times by processor and model
            if processor not in processor_model_times:
                processor_model_times[processor] = {}
            if model not in processor_model_times[processor]:
                processor_model_times[processor][model] = []
            processor_model_times[processor][model].append(time)

# Write results to the output file
with open(output_file, "w", encoding="utf-8") as f:
    f.write("Processor; Model; Min Time; Max Time; Mean Time; Median Time\n")
    for processor, models in processor_model_times.items():
        for model, times in models.items():
            min_time = str(min(times)).replace('.',',')
            max_time = str(max(times)).replace('.',',')
            mean_time = str(mean(times)).replace('.',',')
            median_time = str(median(times)).replace('.',',')
            f.write(f"{processor}; {model}; {min_time}; {max_time}; {mean_time}; {median_time}\n")


In [ ]:
# =============================================================================
# STEP: Classification cross-comparison (manual)
# Builds the manual concept cross-comparison report.
# -> REPORT-manual_cross-comparison_concept-data.csv
# =============================================================================

# Classification cross-comparison 
import json
from itertools import combinations

# Europeanacategories-Liste
europeanacategories = [
    'Building', 'Archaeological Site', 'Cartoon', 'Ceramics', 'Clothing',
    'Costume Accessories', 'Drawing', 'Map', 'Furniture', 'Textile', 'Food',
    'Glassware', 'Inscription', 'Jewellery', 'Metalwork', 'Machinery', 'Medal',
    'Memorabilia', 'Mineral', 'Musical Instrument', 'Painting', 'Photograph',
    'Postcard', 'Poster', 'Print', 'Sculpture', 'Specimen', 'Tableware', 'Tool',
    'Tapestry', 'Toy', 'Weaponry', 'Woodwork', 'Stamp'
]

manual_classified_categories = os.path.join(PROJECT_ROOT, "Data/Database/25-08-11-around_700_manual_classified_objects.json")
file_path = os.path.join(PROJECT_ROOT, "Reports/REPORT-corrected_concept-data.json")
output_file = os.path.join(PROJECT_ROOT, "Reports/REPORT-manual_cross-comparison_concept-data.csv")
# Funktion zum Extrahieren und Normalisieren der Kategorien
def extract_categories(result):
    # Entferne Zahlen und Sonderzeichen, behalte nur Kategorien
    categories = [word.strip() for word in result.split(",") if not any(char.isdigit() for char in word)]
    # Filtere nur gültige Kategorien aus europeanacategories
    return set(cat for cat in categories if cat in europeanacategories)

# JSON-Daten laden
with open(manual_classified_categories, "r") as file:
    manual_classified_categories = json.load(file)

with open(file_path, "r") as file:
    data = json.load(file)

with open(output_file, "w", encoding="utf-8") as f:
    for entry["objects"] in manual_classified_categories:
        open_id = entry["uid"]
        matching_entry = next((item for item in data if item.get("id") == open_id), None)
        # Überprüfen, ob ein Eintrag gefunden wurde
        if matching_entry:
    

    
    
        
            temp_categories=[]
            for item in entry["categorized_data"]:
                if 'result' in item:
                    result = item['result']
                    if isinstance(result, str):
                        model = item['model']
                        processor = item['processor']
                        item['result'] = result.strip()
                        itemcategories = extract_categories(item['result'])
                    if (len(itemcategories) in {0, 5}) and not any(entry.get('processor') == processor + '+' + model for entry in temp_categories):
                        temp_categories.append({
                            'processor': processor + '+' + model,
                            'itemcategories': itemcategories
                        })

            for entry1, entry2 in combinations(temp_categories, 2):  # Compare all pairs of entries
                # Extract fields to compare (e.g., categorized_data)
                categorized_data1 = entry1.get("itemcategories", [])
                categorized_data2 = entry2.get("itemcategories", [])

                # Perform comparison (example: find common elements)
                common_elements = set(categorized_data1) & set(categorized_data2)

                # Print or store the results
                print(f"{entry["id"]},{entry1.get('processor', 'N/A')} vs. {entry2.get('processor', 'N/A')},{len(common_elements)}") #Overlap Count ; Common Elements: {common_elements}

                
                f.write(f"{entry['id']},{entry1.get('processor', 'N/A')} vs. {entry2.get('processor', 'N/A')},{len(common_elements)}\n")

print(f"Korrigierte Daten wurden in {output_file} gespeichert.")


# 5. Mapping to authority data

## Query Wikidata entries

In [ ]:
# =============================================================================
# STEP: 5. Authority mapping: Wikidata
# Resolves enriched descriptions to Wikidata entities (with Wikipedia fallback lookups) and stores the entity id.
# reads description -> writes edmWikidataId | Service: Wikidata / Wikipedia | DB: objaverse.sqlite3.db
# =============================================================================

import os
import sqlite3
import secrets
import requests
import json
import re
from pathlib import Path
from mlx_lm import load, generate
from string import Template
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from main_functions import create_record, publish_record, upload_files_into_deposition
from utilities import validate_edm_xml, validate_metsmods, validate_zenodo_metadata
import urllib.parse
from rapidfuzz import fuzz, process
import spacy

#Lade das spaCy-Modell (z. B. englisches Modell)
nlp = spacy.load("en_core_web_sm")
model="mlx-community/DeepSeek-R1-4bit"

def find_wikipedia_entry_optimized(query):
    """
    Find the most relevant Wikipedia entry for a given query using the MediaWiki API.
    
    Parameters:
        query (str): The search query.
    
    Returns:
        dict: A dictionary containing the title, snippet, and link to the Wikipedia page.
    """
    if not query or not isinstance(query, str):
        return {"error": "Invalid input query. Please provide a non-empty string."}

    # Define the MediaWiki API endpoint
    mediawiki_api_url = "https://en.wikipedia.org/w/api.php"

    # Set up the query parameters
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
        "utf8": 1,
        "srlimit": 5,  # Fetch top 5 results for better relevance
        "srprop": "snippet|titlesnippet"  # Include snippets for better context
    }

    # Define custom headers with a User-Agent
    headers = {
        "User-Agent": "University of Jena retrieval (https://uni-jena.de; sander.muenster@uni-jena.de)"
    }

    try:
        # Make the API request
        response = requests.get(mediawiki_api_url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to query MediaWiki API. Network error: {str(e)}"}

    try:
        # Parse the response JSON
        search_results = response.json().get("query", {}).get("search", [])
        if not search_results:
            return {"error": "No results found for the given query."}

        # Perform fuzzy matching to find the most relevant result
        best_match = None
        highest_score = 0

        for result in search_results:
            title = result.get("title", "")
            snippet = re.sub(r"<[^>]+>", "", result.get("snippet", ""))  # Remove HTML tags
            page_id = result.get("pageid", None)
            link = f"https://en.wikipedia.org/wiki?curid={page_id}" if page_id else "No link available"

            # Calculate fuzzy match score
            score = fuzz.token_sort_ratio(query, title)
            if score > highest_score:
                highest_score = score
                best_match = {
                    "title": title,
                    "snippet": snippet,
                    "link": link,
                    "score": score
                }

        if not best_match:
            return {"error": "No suitable match found using fuzzy matching."}

        return best_match
    except (ValueError, KeyError) as e:
        return {"error": f"Failed to parse MediaWiki API response: {str(e)}"}
    
def clean_text(text):
    """
    Clean the input text by removing unnecessary characters and information.
    """
    # Remove URLs, special characters, and extra spaces
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def find_wikipedia_entry_free(query):
    """
    Find the Wikipedia entry for a given query using the MediaWiki API.
    
    Parameters:
        query (str): The search query.
    
    Returns:
        dict: A dictionary containing the title, snippet, and link to the Wikipedia page.
    """
    if not query or not isinstance(query, str):
        return {"error": "Invalid input query. Please provide a non-empty string."}

    # Define the MediaWiki API endpoint
    mediawiki_api_url = "https://en.wikipedia.org/w/api.php"

    # Set up the query parameters
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
        "utf8": 1,
        "srlimit": 1  # Limit to the most relevant result
    }

    # Define custom headers with a User-Agent
    headers = {
        "User-Agent": "University of Jena retrieval (https://uni-jena.de; sander.muenster@uni-jena.de)"
    }

    try:
        # Make the API request
        response = requests.get(mediawiki_api_url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to query MediaWiki API. Network error: {str(e)}"}

    try:
        # Parse the response JSON
        search_results = response.json().get("query", {}).get("search", [])
        if not search_results:
            return {"error": "No results found for the given query."}

        # Extract the first result (most relevant)
        first_result = search_results[0]
        title = first_result.get("title", "No title available")
        snippet = re.sub(r"<[^>]+>", "", first_result.get("snippet", "No snippet available"))  # Remove HTML tags
        page_id = first_result.get("pageid", None)
        link = f"https://en.wikipedia.org/wiki?curid={page_id}" if page_id else "No link available"

        return {
            "title": title,
            "snippet": snippet,
            "link": link,
        }
    except (ValueError, KeyError) as e:
        return {"error": f"Failed to parse MediaWiki API response: {str(e)}"}
    
def find_wikipedia_entry_google_api(query, api_key, cx):
    """
    Find the Wikipedia entry for a given query using Google Custom Search API.
    
    Parameters:
        query (str): The search query.
        api_key (str): Your Google API key.
        cx (str): Your Google Custom Search Engine ID.
    
    Returns:
        dict: A dictionary containing the title, snippet, and link to the Wikipedia page.
    """
    if not query or not isinstance(query, str):
        return {"error": "Invalid input query. Please provide a non-empty string."}

    # Define the Google Custom Search API endpoint
    google_api_url = "https://www.googleapis.com/customsearch/v1"

    # Set up the query parameters
    params = {
        "q": f"{query} site:en.wikipedia.org",  # Restrict search to Wikipedia
        "key": api_key,
        "cx": cx,
    }

    try:
        # Make the API request
        response = requests.get(google_api_url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to query Google API. Network error: {str(e)}"}

    try:
        # Parse the response JSON
        results = response.json().get("items", [])
        if not results:
            return {"error": "No results found for the given query."}

        # Extract the first result (most relevant)
        first_result = results[0]
        return {
            "title": first_result.get("title", "No title available"),
            "snippet": first_result.get("snippet", "No snippet available"),
            "link": first_result.get("link", "No link available"),
        }
    except (ValueError, KeyError) as e:
        return {"error": f"Failed to parse Google API response: {str(e)}"}
    
def query_wikidata_robust(text, details=None):
    """
    Query Wikidata for a given text and return the most relevant Wikidata ID and Wikipedia link.
    Uses spaCy to identify the most likely named entity for the query.
    """
    if not text or not isinstance(text, str):
        return {"error": "Invalid input text. Please provide a non-empty string."}

    # Clean the input text
    text = clean_text(text)

    # Use spaCy to extract named entities
    doc = nlp(text)
    named_entities = [ent.text for ent in doc.ents if ent.label_ in {"PERSON", "ORG", "GPE", "LOC", "WORK_OF_ART"}]

    if not named_entities:
        return {"error": "No named entities found in the input text."}

    # Use the most relevant named entity for the query
    main_entity = named_entities[0]

    # Define the Wikidata API endpoint
    wikidata_api_url = "https://www.wikidata.org/w/api.php"

    headers = {
        "User-Agent": "University of Jena retrieval (https://uni-jena.de; sander.muenster@uni-jena.de)"
    }

    search_params = {
        "action": "wbsearchentities",
        "search": main_entity,
        "language": "en",
        "format": "json",
        "limit": 5  # Retrieve multiple results for better matching
    }

    try:
        response = requests.get(wikidata_api_url, params=search_params, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to query Wikidata. Network error: {str(e)}"}

    try:
        search_results = response.json().get("search", [])
        if not search_results:
            return {"error": "No results found for the given entity."}

        # Perform fuzzy matching to find the best match
        candidates = [(result["id"], result["label"], result.get("description", "")) for result in search_results]
        best_match = None
        highest_score = 0

        for candidate_id, candidate_label, candidate_description in candidates:
            combined_text = f"{candidate_label} {candidate_description}"
            score = fuzz.token_sort_ratio(main_entity, combined_text)
            if details:
                score += fuzz.token_sort_ratio(details, combined_text) * 0.5  # Weight details less
            if score > highest_score:
                highest_score = score
                best_match = {
                    "wikidata_id": candidate_id,
                    "label": candidate_label,
                    "description": candidate_description
                }

        if not best_match:
            return {"error": "No suitable match found using fuzzy matching."}

        # Query the Wikipedia link for the best match
        wikipedia_params = {
            "action": "wbgetentities",
            "ids": best_match["wikidata_id"],
            "props": "sitelinks",
            "format": "json"
        }

        try:
            response = requests.get(wikidata_api_url, params=wikipedia_params, headers=headers, timeout=10)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            return {"error": f"Failed to query Wikipedia link. Network error: {str(e)}"}

        try:
            entities = response.json().get("entities", {})
            if best_match["wikidata_id"] not in entities:
                return {"error": "Failed to retrieve entity details from Wikidata."}

            sitelinks = entities[best_match["wikidata_id"]].get("sitelinks", {})
            wikipedia_link = sitelinks.get("enwiki", {}).get("url", "No Wikipedia link available")
            best_match["wikipedia_link"] = wikipedia_link
        except (ValueError, KeyError) as e:
            return {"error": f"Failed to parse Wikipedia link response: {str(e)}"}

        return best_match
    except (ValueError, KeyError) as e:
        return {"error": f"Failed to parse Wikidata search response: {str(e)}"}

if __name__ == "__main__":
    #Open database:
    DATABASE = os.path.join(PROJECT_ROOT, "Tutorials/objaverse.sqlite3.db")
    dbvalueread="description"
    geonames_username = os.environ["GEONAMES_USERNAME"]
    #model, tokenizer = load("Qwen/Qwen3-32B-MLX-4bit")
    model, tokenizer = load("mlx-community/Mistral-Small-24B-Instruct-2501-4bit")
    #model, tokenizer = load("mlx-community/Llama-3.3-70B-Instruct-4bit")
    #processor='Qwen3.0-32 + Spacy en_core_web_sm + Wikibase query'
    #processor='Qwen3.0-32B + Mediawiki + Rapidfuzz + Wikibase query'
    processor='Mistral-Small-24B-Instruct-2501-4bit + Mediawiki + Rapidfuzz + Wikibase query'
    dbvaluewrite="edmWikidataId"
    retrievalquery = ('SELECT * '
                'FROM object '
                'WHERE source LIKE "%Objaverse%" AND '+dbvaluewrite+' IS "" ') #Europeana

    metsmods_template_path = Path("Templates/template_metsmods.xml")
    metsmods_template = Template(metsmods_template_path.read_text(encoding="utf-8"))

    #Europeana request
    j=1
    db = sqlite3.connect(DATABASE)
    cura = db.cursor()
    cura.execute(retrievalquery)
    rows = cura.fetchall()

    # Create a JSON
    # Get column names
    column_names = [description[0] for description in cura.description]
    # Convert rows to a list of dictionaries
    result = []
    for row in rows:
        result.append(dict(zip(column_names, row)))
    # Convert the list of dictionaries to a JSON string
    json_result = json.dumps(result, indent=4)

    for i in result:
        wikidataquery = ''
        try:
            input_text = str(i["name"])[:2048]  # Beschränkung auf 500 Zeichen #aiDescriptionEn
            prompt = (
                "you are a search agent which needs to retrieve the correct id of an object from wikidata. "
                "Please begin your answer with the most likely name of the object described in the text and be as exact as possible. "
                "Please format your answer as plain json. Please provide as key 'Name' with object name and 'Details' with additional details. "
                "If no object information is given return 'None'. Input text is: " + input_text
            )

            # Nachrichten vorbereiten
            messages = [{"role": "user", "content": prompt}]

            # Tokenizer-Aufruf mit optimierten Parametern
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=False
            )

            # Generierung mit reduzierter Token-Anzahl
            objectname = generate(
                model, tokenizer, prompt=prompt, verbose=True, max_tokens=128  # Reduziert von 64 auf 32
            )
            print(objectname)

            objectname = ((objectname.replace("json", "")).replace("```", "")).replace("...", "")
            objectnamejson = json.loads(objectname)
            try:
                wikidataquery = str(objectnamejson["Name"])
                #wikidataquery = str(objectnamejson["Name"] + " , " + objectnamejson["Details"])
                details = str(objectnamejson.get("Details", ""))
            except:
                wikidataquery = ''
                details = ''
        except:
            wikidataquery = ''
            details = ''

        print("Wikidata Query: " + wikidataquery)
        if wikidataquery != 'None':
            # Query Wikipedia entry with robust function
            #result = query_wikidata_robust(wikidataquery, details)
            api_key = os.environ["GOOGLE_API_KEY"]
            cx = os.environ["GOOGLE_CSE_CX"]
            #result = find_wikipedia_entry_google_api(wikidataquery, api_key, cx)
            #result = find_wikipedia_entry_free(wikidataquery)
            print("### Wikipedia Query: "+str(wikidataquery))
            resulta = find_wikipedia_entry_optimized(wikidataquery)
            print("### Wikipedia entry: "+str(resulta))
            
            if ("error" in str(resulta)):
                updatequery='UPDATE object SET '+dbvaluewrite+'="{}" WHERE "uid" = "'+i["uid"]+'"'
            else:
                updatequery='UPDATE object SET '+dbvaluewrite+'="{\'processor\':\''+processor+'\', \'query\': \''+wikidataquery+'\','+str(resulta)+'}" WHERE "uid" = "'+i["uid"]+'"'   
        else:
            updatequery='UPDATE object SET '+dbvaluewrite+'="{}" WHERE "uid" = "'+i["uid"]+'"'
        
        print(updatequery)    
        try:
            db = sqlite3.connect(DATABASE)
            curab = db.cursor()
            curab.execute(updatequery)
            db.commit()
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+i["uid"]+'"'
            curab.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(curab.fetchall()))
            db.close()
        except:
            print(f"#{j}: Failed: "+str(curab.fetchall()))
        j=j+1


## Retrieving OSM IDs

In [ ]:
# =============================================================================
# STEP: Authority mapping: OSM IDs
# Reverse-geocodes coordinates to OpenStreetMap object IDs via Nominatim.
# reads aiDescriptionEn | Service: OSM / Nominatim | DB: objaverse.sqlite3.db
# =============================================================================

#This is code to query the OSM IDs
 
from geopy.geocoders import Nominatim

import os
import time
import requests
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
import secrets

def get_osm_id_for_coordinate(lat, lon):
    """
    Queries the OSM ID for a given latitude and longitude using geopy and Nominatim.

    Args:
        lat (float): The latitude.
        lon (float): The longitude.

    Returns:
        tuple: A tuple containing the OSM type ('node', 'way', or 'relation') and the OSM ID,
               or (None, None) if not found or an error occurs.
    """
    # It's good practice to specify a custom user-agent per Nominatim's usage policy.
    geolocator = Nominatim(user_agent="my-osm-app/1.0")
    
    try:
        # The language='en' parameter is optional, but can help get consistent results.
        location = geolocator.reverse((lat, lon), exactly_one=True, language='en')

        if location and 'osm_id' in location.raw:
            osm_type = location.raw.get('osm_type')
            osm_id = location.raw.get('osm_id')
            return osm_type, osm_id
        else:
            print("Could not find OSM ID for the given coordinates.")
            return None, None

    except Exception as e:
        print(f"An error occurred: {e}")
        return None, None

if __name__ == '__main__':
    DATABASE = 'Tutorials/objaverse.sqlite3.db'
    DBTABLE = 'object'
    dbvalueread="aiDescriptionEn"
    Path(os.path.join(KERAS_DIR, "object")).mkdir(parents=True, exist_ok=True)
    uidlist=[]
    j=0
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    retrievalquery='select edmPlaceLatitude,edmPlaceLongitude,uid from object where edmOsmId is Null AND edmPlaceLongitude is not "" AND edmPlaceLongitude is not Null;'

    for uid in cur.execute(retrievalquery):
            uidlist.append([uid[0],uid[1],uid[2]])
    db.close()

    for i in uidlist:
        j=j+1
        try:
            # Example: Coordinates for the Eiffel Tower in Paris
            latitude = i[0]
            longitude = i[1]

            print(f"Querying OSM ID for coordinates: ({latitude}, {longitude})...")
            osm_type, osm_id = get_osm_id_for_coordinate(latitude, longitude)

            if osm_id:
                print(f"OSM ID: {osm_id}")
                updatequery='UPDATE object SET edmOsmId="'+str(osm_id)+'" WHERE uid = "'+i[2]+'"'
                print(updatequery)
                db = sqlite3.connect(DATABASE)
                cura = db.cursor()
                cura.execute(str(updatequery))
                db.commit()
                checkquery = 'SELECT * FROM "'+DBTABLE+'" WHERE "uid" = "'+i[2]+'"'
                cura.execute(checkquery)
                print(f"#{j}: Successfully updated: "+str(cura.fetchall()))
                db.close()
            else:
                print("\n--- Result ---")
                print("Failed to retrieve OSM information.")     
        except:
            print("Skipped")

   


In [ ]:
# =============================================================================
# STEP: Authority mapping: Wikidata + GeoNames
# Resolves places to Wikidata and GeoNames identifiers.
# reads description -> writes edmWikidataId | Service: Wikidata, GeoNames | DB: objaverse.sqlite3.db
# =============================================================================

import pyeuropeana.apis as apis
import pyeuropeana.utils as utils
import os
import sqlite3
import secrets
import requests
import json
import re
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from huggingface_hub import InferenceClient, login
from mlx_lm import load, generate
from string import Template
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None
from main_functions import create_record, publish_record, upload_files_into_deposition
from utilities import validate_edm_xml, validate_metsmods, validate_zenodo_metadata
import urllib.parse
from rapidfuzz import fuzz, process

def clean_text(text):
    """
    Clean the input text by removing unnecessary characters and information.
    """
    # Remove URLs, special characters, and extra spaces
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def query_geonames_id(fuzzy_name, username, max_results=10):
    """
    Query GeoNames for a fuzzy input name and return the best matching GeoNames ID.
    
    Args:
        fuzzy_name (str): The input name to search for.
        username (str): Your GeoNames API username.
        max_results (int): Maximum number of results to retrieve from GeoNames.
    
    Returns:
        dict: A dictionary containing the best match with GeoNames ID, name, and score.
    """
    if not fuzzy_name or not isinstance(fuzzy_name, str):
        return {"error": "Invalid input name. Please provide a non-empty string."}

    # GeoNames API endpoint
    geonames_api_url = "http://api.geonames.org/searchJSON"

    # API parameters
    params = {
        "q": fuzzy_name,
        "maxRows": max_results,
        "username": username
    }

    try:
        # Query the GeoNames API
        response = requests.get(geonames_api_url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to query GeoNames. Network error: {str(e)}"}

    try:
        # Parse the API response
        results = response.json().get("geonames", [])
        if not results:
            return {"error": "No results found for the given name."}

        # Perform fuzzy matching to find the best match
        candidates = [(result["geonameId"], result["name"], result.get("countryName", "")) for result in results]
        best_match = None
        highest_score = 0

        for geoname_id, name, country in candidates:
            combined_text = f"{name}, {country}"
            score = fuzz.token_sort_ratio(fuzzy_name, combined_text)
            if score > highest_score:
                highest_score = score
                best_match = {
                    "geonameId": geoname_id,
                    "name": name,
                    "country": country,
                    "score": score
                }

        if not best_match:
            return {"error": "No suitable match found using fuzzy matching."}

        return best_match
    except (ValueError, KeyError) as e:
        return {"error": f"Failed to parse GeoNames response: {str(e)}"}

def query_wikidata_robust(text, details=None):
    """
    Query Wikidata for a given text and return the most relevant Wikidata ID and Wikipedia link.
    Performs a robust search by splitting the input text into keywords.
    """
    if not text or not isinstance(text, str):
        return {"error": "Invalid input text. Please provide a non-empty string."}

    # Clean the input text
    text = clean_text(text)

    # Split the text into individual keywords
    keywords = text.split()

    # Define the Wikidata API endpoint
    wikidata_api_url = "https://www.wikidata.org/w/api.php"

    headers = {
        "User-Agent": "University of Jena retrieval (https://uni-jena.de; sander.muenster@uni-jena.de)"
    }

    all_candidates = []

    # Perform a search for each keyword
    for keyword in keywords:
        search_params = {
            "action": "wbsearchentities",
            "search": keyword,
            "language": "en",
            "format": "json",
            "limit": 5  # Retrieve multiple results for better matching
        }

        try:
            response = requests.get(wikidata_api_url, params=search_params, headers=headers, timeout=10)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            return {"error": f"Failed to query Wikidata. Network error: {str(e)}"}

        try:
            search_results = response.json().get("search", [])
            if search_results:
                candidates = [(result["id"], result["label"], result.get("description", "")) for result in search_results]
                all_candidates.extend(candidates)
        except (ValueError, KeyError) as e:
            return {"error": f"Failed to parse Wikidata search response: {str(e)}"}

    if not all_candidates:
        return {"error": "No results found for the given text."}

    # Use fuzzy matching to find the best match
    best_match = None
    highest_score = 0

    for candidate_id, candidate_label, candidate_description in all_candidates:
        # Combine label and description for matching
        combined_text = f"{candidate_label} {candidate_description}"
        score = fuzz.token_sort_ratio(text, combined_text)
        if details:
            score += fuzz.token_sort_ratio(details, combined_text) * 0.5  # Weight details less
        if score > highest_score:
            highest_score = score
            best_match = {
                "wikidata_id": candidate_id,
                "label": candidate_label,
                "description": candidate_description
            }

    if not best_match:
        return {"error": "No suitable match found using fuzzy matching."}

    # Query the Wikipedia link for the best match
    wikipedia_params = {
        "action": "wbgetentities",
        "ids": best_match["wikidata_id"],
        "props": "sitelinks",
        "format": "json"
    }

    try:
        response = requests.get(wikidata_api_url, params=wikipedia_params, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to query Wikipedia link. Network error: {str(e)}"}

    try:
        entities = response.json().get("entities", {})
        if best_match["wikidata_id"] not in entities:
            return {"error": "Failed to retrieve entity details from Wikidata."}

        sitelinks = entities[best_match["wikidata_id"]].get("sitelinks", {})
        wikipedia_link = sitelinks.get("enwiki", {}).get("url", "No Wikipedia link available")
        best_match["wikipedia_link"] = wikipedia_link
    except (ValueError, KeyError) as e:
        return {"error": f"Failed to parse Wikipedia link response: {str(e)}"}

    return best_match

if __name__ == "__main__":
    #Open database:
    DATABASE = os.path.join(PROJECT_ROOT, "Tutorials/objaverse.sqlite3.db")
    dbvalueread="description"
    geonames_username = os.environ["GEONAMES_USERNAME"]
    dbvaluewrite="edmWikidataId"
    retrievalquery = ('SELECT * '
                'FROM object '
                'WHERE '+dbvaluewrite+' IS NULL and source LIKE "%Europeana%" ') #"WHERE zenodo_url IS NOT NULL AND mode IS NOT 1 ")

    model, tokenizer = load("Qwen/Qwen3-32B-MLX-4bit")
    metsmods_template_path = Path("Templates/template_metsmods.xml")
    metsmods_template = Template(metsmods_template_path.read_text(encoding="utf-8"))

    #Europeana request
    j=1
    db = sqlite3.connect(DATABASE)
    cura = db.cursor()
    cura.execute(retrievalquery)
    rows = cura.fetchall()

    # Create a JSON
    # Get column names
    column_names = [description[0] for description in cura.description]
    # Convert rows to a list of dictionaries
    result = []
    for row in rows:
        result.append(dict(zip(column_names, row)))
    # Convert the list of dictionaries to a JSON string
    json_result = json.dumps(result, indent=4)

    for i in result:
        wikidataquery = ''
        try:
            prompt = "you are a search agent which needs to retrieve the correct id of an object from wikidata. Please begin your answer with the most likely name of the object described in the text and be as exact as possible. Please format your answer as plain json. Please provide as key 'Name' with object name and 'Details' with additional details. If no object information is given return 'None'. Input text is: " + str(i["aiDescriptionEn"])
            messages = [{"role": "user", "content": prompt}]
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=False 
            )
            objectname = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=128)
            objectname = ((objectname.replace("json", "")).replace("```", "")).replace("...", "")
            objectnamejson = json.loads(objectname)
            try:
                wikidataquery = str(objectnamejson["Name"])
                details = str(objectnamejson.get("Details", ""))
            except:
                wikidataquery = ''
                details = ''
        except:
            wikidataquery = ''
            details = ''

        print("Wikidata Query: " + wikidataquery)

        # Query Wikipedia entry with robust function
        result = query_wikidata_robust(wikidataquery, details)
        print("### Wikipedia entry: "+str(result))
        if ("label" in str(result)):
            updatequery='UPDATE object SET '+dbvaluewrite+'="'+str(result)+'" WHERE "uid" = "'+i["uid"]+'"'
            print(updatequery)
        else:
            updatequery='UPDATE object SET '+dbvaluewrite+'="{}" WHERE "uid" = "'+i["uid"]+'"'
            db = sqlite3.connect(DATABASE)
            curab = db.cursor()
            #curab.execute(str(updatequery))
            checkquery = 'SELECT * FROM object WHERE "uid" = "'+i["uid"]+'"'
            curab.execute(checkquery)
            print(f"#{j}: Successfully updated: "+str(curab.fetchall()))
            db.close()

        # Query Geonames 
        addressjson=''
        
        query="Please begin your answer with the name of the place and be as exact as possible. Do not use more than 30 words. The geographic location of the object described in this text: " + str(i["aiDescriptionEn"])
        messages = [{"role": "user", "content": query}]
        prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, enable_thinking=False 
        )
        response = generate(model, tokenizer,  prompt=prompt, verbose=True, max_tokens=128)
        print("### Geolocalization text: "+ str(response))
        
        prompt = "Please begin your answer with the name of the place and be as exact as possible. Include nearby landmarks, regions, or any additional geographic context if available. Please format your answer as plain json. Use the following keys: 'Place' for city, street, and region information, and 'SpecificLocation' for additional details such as landmarks, coordinates, or descriptions. If no location information is given, return 'None'. Provide multiple possible matches if applicable. Geographic location of the object described in this text: " + response     
        messages = [{"role": "user", "content": prompt}]
        address = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=128)
        address = ((address.replace("json","")).replace("```","")).replace("...","")
        try:
            addressjson=json.loads(address)
        except:
            addressjson=response
        try:
            location=(str(addressjson["Place"]).replace('\'','')).replace('"','\'').replace('None','')
        except:
            location=addressjson
        print(location)
        if (location != "None"):
            result = query_geonames_id(location, geonames_username)
            print("### Geonames place: "+ str(result))


# 6. Reports

### Text similarity check

In [ ]:
# =============================================================================
# STEP: 6. Text similarity check (single record set)
# Computes semantic similarity between AI and reference descriptions for quality assurance.
# reads name -> writes aiDescriptionShort | -> REPORT-text-similarity.csv | DB: objaverse.sqlite32.db
# =============================================================================

import os
from sentence_transformers import SentenceTransformer, util
import sqlite3
import csv
import json
import re

dbvalueread = "name"
dbvaluewrite = "aiDescriptionShort"
dbvaluecheck = 'source LIKE "%Europeana%"'

DATABASE = 'objaverse.sqlite32.db'

def main():
    OUTPUT_CSV = 'REPORT-text-similarity.csv'
    retrievalquery = (
        'SELECT aiDescriptionShort, aiDescriptionEn '
        'FROM object '
        'WHERE aiDescriptionShort IS NOT NULL AND aiDescriptionEn IS NOT NULL AND source LIKE "%Europeana%"'
    )
    
    print(retrievalquery)

    # Datenbankverbindung herstellen
    with sqlite3.connect(DATABASE) as db:
        cur = db.cursor()

        # Anzahl der bereits verarbeiteten Datensätze
        cur.execute(f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} IS NOT NULL AND {dbvaluecheck};')
        processed_count = cur.fetchone()[0]
        print(f"Number of datasets already processed: {processed_count}")

        # Anzahl der verbleibenden Datensätze
        cur.execute(f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} IS NULL AND {dbvaluecheck};')
        remaining_count = cur.fetchone()[0]
        print(f"Number of datasets to process is: {remaining_count}")

        # Daten abrufen
        uidlist = cur.execute(retrievalquery).fetchall()

    # Vortrainiertes Modell laden (nur einmal)
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Ergebnisse in CSV-Datei schreiben
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['Similarity', 'String_1', 'String_2', 'Model']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for i in uidlist:
            # JSON-Objekte aufteilen
            json_objects = i[0].split('}{')
            try:
                for obj in json_objects:
                        
                    # Text 1 und Text 2 vorbereiten
                    text1 = i[1].replace("'", '"')
                    text1 = text1.split("result")[0] + ('result":"') + text1.split("result")[1][2:].replace('}', '').replace('"', '') + '"' if "result" in text1 else text1
                    text1=f"{text1}}}" if not text1.endswith('}') else text1
                    
                    text2 = obj.replace("'", '"')
                    text2 = text2.split("result")[0] + ('result":"') + text2.split("result")[1][2:].replace('}', '').replace('"', '') + '"' if "result" in text2 else text2
                    text2=f"{text2}}}" if not text2.endswith('}') else text2
                    text2=f"{{{text2}" if not text2.startswith('{') else text2
                    # Modelle extrahieren

                        # Backslashes escapen
                    text1 = re.sub(r'\\(?!["\\/bfnrt])', r'', text1).replace("\\",'')
                    text2 = re.sub(r'\\(?!["\\/bfnrt])', r'', text2).replace("\\",'')

                    models = []

                    json_data1 = json.loads(text2)
                    json_data2 = json.loads(text1)
                    if 'processor' in json_data1:
                        models.append(json_data1['processor'])
                        models.append(json_data1['model']) if 'model' in json_data1 else None
                    

                    # Texte in Vektoren umwandeln
                    embedding1 = model.encode(json_data1['result'], convert_to_tensor=True)
                    embedding2 = model.encode(json_data2['result'], convert_to_tensor=True)

                    # Kosinusähnlichkeit berechnen
                    cosine_similarity = util.pytorch_cos_sim(embedding1, embedding2).item()

                    # Ergebnisse in die CSV schreiben
                    writer.writerow({
                        'Similarity': f"{cosine_similarity:.2f}",
                        'Model': ', '.join(models)
                        #'String_1': json_data1['result'],
                        #'String_2': json_data2['result']       
                    })

                    print(f"Die semantische Ähnlichkeit zwischen den Texten beträgt: {cosine_similarity:.2f}")
            except:
                print('Skipped')

if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# STEP: 6. Text similarity check (all records)
# Batch semantic-similarity QA across all records using sentence-transformers.
# -> REPORT-text-similarity-all.csv | DB: objaverse.sqlite32.db
# =============================================================================

#Simiarity check for all
# 
import os
from sentence_transformers import SentenceTransformer, util
import sqlite3
import csv
import json
import re

dbvalueread = "name"
dbvaluewrite = "aiDescriptionShort"
dbvaluecheck = ''

DATABASE = 'objaverse.sqlite32.db' #Tutorials/

def main():
    OUTPUT_CSV = 'REPORT-text-similarity-all.csv'
    retrievalquery = (
        'SELECT aiDescriptionShort, aiDescriptionEn, name, description '
        'FROM object '
        'WHERE aiDescriptionShort IS NOT NULL AND name IS NOT NULL'
    )
    
    print(retrievalquery)

    # Datenbankverbindung herstellen
    with sqlite3.connect(DATABASE) as db:
        cur = db.cursor()

        # Anzahl der bereits verarbeiteten Datensätze
        cur.execute(f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} IS NOT NULL;')
        processed_count = cur.fetchone()[0]
        print(f"Number of datasets already processed: {processed_count}")

        # Anzahl der verbleibenden Datensätze
        cur.execute(f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} IS NULL;')
        remaining_count = cur.fetchone()[0]
        print(f"Number of datasets to process is: {remaining_count}")

        # Daten abrufen
        uidlist = cur.execute(retrievalquery).fetchall()

    # Vortrainiertes Modell laden (nur einmal)
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Ergebnisse in CSV-Datei schreiben
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['Similarity', 'String_1', 'String_2', 'Model']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        print(f"Anzahl der Elemente in uidlist: {len(uidlist)}")
        for i in uidlist:

            # JSON-Objekte aufteilen
            json_objects = i[0].split('}{')
            print(i[2])
   
            

            try:
                if i[1] != "":
                    json_data2=str(i[2])+': '+str(i[3])
                    embedding2 = model.encode(json_data2, convert_to_tensor=True)
                else:
                    description=i[1]
                    text1 = description.replace("'", '"')
                    text1 = text1.split("result")[0] + ('result":"') + text1.split("result")[1][2:].replace('}', '').replace('"', '') + '"' if "result" in text1 else text1
                    text1=f"{text1}}}" if not text1.endswith('}') else text1
                    text1 = re.sub(r'\\(?!["\\/bfnrt])', r'', text1).replace("\\",'')
                    json_data2 = json.loads(text1)    
                    embedding2 = model.encode(json_data2['result'], convert_to_tensor=True)
                for obj in json_objects:
                        
                    # Text 1 und Text 2 vorbereiten
        
                    #print (text1)

                    text2 = obj.replace("'", '"')
                    text2 = text2.split("result")[0] + ('result":"') + text2.split("result")[1][2:].replace('}', '').replace('"', '') + '"' if "result" in text2 else text2
                    text2=f"{text2}}}" if not text2.endswith('}') else text2
                    text2=f"{{{text2}" if not text2.startswith('{') else text2
                    # Modelle extrahieren

                        # Backslashes escapen
       
                    text2 = re.sub(r'\\(?!["\\/bfnrt])', r'', text2).replace("\\",'')

                    models = []

                    json_data1 = json.loads(text2)
           
                    if 'processor' in json_data1:
                        models.append(json_data1['processor'])
                        models.append(json_data1['model']) if 'model' in json_data1 else None
                    

                    # Texte in Vektoren umwandeln
                    embedding1 = model.encode(json_data1['result'], convert_to_tensor=True)
       

                    # Kosinusähnlichkeit berechnen
                    cosine_similarity = util.pytorch_cos_sim(embedding1, embedding2).item()

                    # Ergebnisse in die CSV schreiben
                    writer.writerow({
                        'Similarity': f"{cosine_similarity:.2f}",
                        'Model': ', '.join(models)
                        #'String_1': json_data1['result'],
                        #'String_2': json_data2['result']       
                    })

                    print(f"Die semantische Ähnlichkeit zwischen den Texten beträgt: {cosine_similarity:.2f}")
            except:
                print('Skipped')

if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# STEP: 6. Similarity via RapidFuzz / TF-IDF / SequenceMatcher
# Lexical similarity comparison using RapidFuzz, TF-IDF cosine and SequenceMatcher.
# -> REPORT-text-similarity-all2.csv | DB: objaverse.sqlite32.db
# =============================================================================

# Use Rapidfuzz, TFIDF and SequenceMatcher for similarity check
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz.fuzz import token_sort_ratio
from difflib import SequenceMatcher
import sqlite3
import csv
import json
import re

dbvalueread = "name"
dbvaluewrite = "aiDescriptionShort"
dbvaluecheck = ''

DATABASE = 'objaverse.sqlite32.db' #Tutorials/
def main():
    OUTPUT_CSV = 'REPORT-text-similarity-all2.csv'
    retrievalquery = (
        'SELECT aiDescriptionShort, aiDescriptionEn, name, description '
        'FROM object '
        'WHERE aiDescriptionShort IS NOT NULL AND name IS NOT NULL'
    )
    
    print(retrievalquery)

    # Datenbankverbindung herstellen
    with sqlite3.connect(DATABASE) as db:
        cur = db.cursor()

        # Anzahl der bereits verarbeiteten Datensätze
        cur.execute(f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} IS NOT NULL;')
        processed_count = cur.fetchone()[0]
        print(f"Number of datasets already processed: {processed_count}")

        # Anzahl der verbleibenden Datensätze
        cur.execute(f'SELECT COUNT(*) FROM object WHERE {dbvalueread} IS NOT NULL AND {dbvaluewrite} IS NULL;')
        remaining_count = cur.fetchone()[0]
        print(f"Number of datasets to process is: {remaining_count}")

        # Daten abrufen
        uidlist = cur.execute(retrievalquery).fetchall()

    # Ergebnisse in CSV-Datei schreiben
    with open("similarity_results.csv", "w", newline="") as csvfile:
        fieldnames = ["RapidFuzz Similarity", "TF-IDF Cosine Similarity", "SequenceMatcher Similarity"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        print(f"Anzahl der Elemente in uidlist: {len(uidlist)}")
        for i in uidlist:
            json_objects = i[0].split('}{')

            # Sicherstellen, dass text1 initialisiert wird
            text1 = ""  # Standardwert für text1
            try:
                json_data2 = {}
                if i[1] != "":
                    json_data2 = str(i[2]) + ': ' + str(i[3])
                else:
                    description = i[1]
                    text1 = description.replace("'", '"')
                    text1 = text1.split("result")[0] + ('result":"') + text1.split("result")[1][2:].replace('}', '').replace('"', '') + '"' if "result" in text1 else text1
                    text1 = f"{text1}}}" if not text1.endswith('}') else text1
                    text1 = re.sub(r'\\(?!["\\/bfnrt])', r'', text1).replace("\\", '')
                    json_data2 = json.loads(text1)
                if 'result' in json_data2:
                    text1=(json_data2['result']) 
                else:
                    text1=""
                print("TEXT1: "+json_data2.replace("\n", "").replace("\r", ""))
                text1=(json_data2.replace("\n", "").replace("\r", ""))
                for obj in json_objects:
                    text2 = obj.replace("'", '"')
                    text2 = text2.split("result")[0] + ('result":"') + text2.split("result")[1][2:].replace('}', '').replace('"', '') + '"' if "result" in text2 else text2
                    text2 = f"{text2}}}" if not text2.endswith('}') else text2
                    text2 = f"{{{text2}" if not text2.startswith('{') else text2
                    text2 = re.sub(r'\\(?!["\\/bfnrt])', r'', text2).replace("\\", '')

                    # Modelle extrahieren
                    models = []
                    json_data1 = json.loads(text2)

                    if 'result' in json_data1:
                        text2=(json_data1['result']) 
                    else:
                        text2=""
                    
                  
                    

                    # --- Option 1: RapidFuzz Similarity ---
                    rapidfuzz_score = token_sort_ratio(text1, text2)

                    # --- Option 2: TF-IDF Cosine Similarity ---
                    corpus = [text1, text2]
                    vectorizer = TfidfVectorizer()
                    tfidf_matrix = vectorizer.fit_transform(corpus)
                    tfidf_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

                    # --- Option 3: SequenceMatcher Similarity ---
                    sequence_matcher_score = SequenceMatcher(None, text1, text2).ratio()

                    # Ergebnisse in die CSV schreiben
                    writer.writerow({
                        "RapidFuzz Similarity": f"{rapidfuzz_score}%",
                        "TF-IDF Cosine Similarity": f"{tfidf_score:.4f}",
                        "SequenceMatcher Similarity": f"{sequence_matcher_score:.4f}"
                    })

            except:
                print('Skipped')

if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# STEP: 6. Translation-quality similarity check
# Scores translation quality using embeddings; defines JSON-parsing and quality-label helpers.
# DB: objaverse.sqlite32.db
# =============================================================================

#!pip install sentence-transformers
import sqlite3
import csv
import json
import re
from sentence_transformers import SentenceTransformer, util

# Load once outside your loop
semantic_model = SentenceTransformer('all-MiniLM-L6-v2') #paraphrase-multilingual-MiniLM-L12-v2')

def get_translation_quality_label(score: float) -> str:
    if score >= 0.90:
        return "Excellent"
    elif score >= 0.75:
        return "Good"
    elif score >= 0.55:
        return "Acceptable"
    elif score >= 0.35:
        return "Poor"
    else:
        return "Very Poor / Unrelated"

def clean_and_parse_json(raw: str) -> dict:
    """Attempt to clean and parse a potentially malformed JSON string."""
    text = raw.replace("'", '"').replace("\n", "").replace("\r", "")
    text = re.sub(r'\\(?!["\\/bfnrt])', r'', text).replace("\\", '')
    if "result" in text:
        before = text.split("result")[0]
        after = text.split("result")[1][2:].replace('}', '').replace('"', '')
        text = before + 'result":"' + after + '"'
    if not text.endswith('}'):
        text += '}'
    if not text.startswith('{'):
        text = '{' + text
    return json.loads(text)

def extract_result_text(raw: str) -> str:
    """Extract the 'result' field from a JSON-like string, or return the raw string."""
    if not raw:
        return ""
    try:
        parsed = clean_and_parse_json(raw)
        return parsed.get('result', "")
    except (json.JSONDecodeError, Exception):
        # If it's not JSON at all, treat the raw value as plain text
        return raw.strip().replace("\n", " ").replace("\r", "")

DATABASE = 'objaverse.sqlite32.db'

def main():
    OUTPUT_CSV = 'all-MiniLM-L6-v2-semantic_similarity_results.csv'
    retrievalquery = (
        'SELECT aiDescriptionShort, description '
        'FROM object '
        'WHERE description IS NOT NULL AND aiDescriptionShort IS NOT NULL AND "source" LIKE "%Europeana%"'
    )
    print(retrievalquery)

    with sqlite3.connect(DATABASE) as db:
        cur = db.cursor()
        uidlist = cur.execute(retrievalquery).fetchall()

        cur.execute('SELECT COUNT(*) FROM object WHERE name IS NOT NULL AND aiDescriptionShort IS NOT NULL;')
        processed_count = cur.fetchone()[0]
        print(f"Number of datasets already processed: {processed_count}")

        cur.execute('SELECT COUNT(*) FROM object WHERE name IS NOT NULL AND aiDescriptionShort IS NULL;')
        remaining_count = cur.fetchone()[0]
        print(f"Number of datasets to process: {remaining_count}")

    print(f"Number of elements in uidlist: {len(uidlist)}")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["Processor", "Semantic Similarity", "Quality Label"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for row in uidlist:
            ai_description_short = row[0]  # aiDescriptionShort (may contain multiple JSON objects)
            description = row[1]           # description (plain text or JSON)

            # --- Extract text1 from description (column index 1) ---
            text1 = extract_result_text(description)
            if not text1:
                print("Skipped: empty text1")
                continue
            print(f"TEXT1: {text1}")

            # --- Handle multiple concatenated JSON objects in aiDescriptionShort ---
            # Re-split properly: wrap each fragment back into a valid JSON object
            raw_parts = ai_description_short.split('}{')
            for idx, part in enumerate(raw_parts):
                if 'distilbert' in part:
                    processor="distilbert"
                if 'Qwen' in part:
                    processor="qwen"
                else:
                    processor="others"
                # Restore braces stripped by the split
                if len(raw_parts) == 1:
                    fragment = part
                elif idx == 0:
                    fragment = part + '}'
                elif idx == len(raw_parts) - 1:
                    fragment = '{' + part
                else:
                    fragment = '{' + part + '}'
                print(processor)
                try:
                    text2 = extract_result_text(fragment)
                    if not text2:
                        print(f"Skipped fragment {idx}: empty text2")
                        continue
                    print(f"TEXT2: {text2}")

                    embedding1 = semantic_model.encode(text1, convert_to_tensor=True)
                    embedding2 = semantic_model.encode(text2, convert_to_tensor=True)
                    semantic_score = util.cos_sim(embedding1, embedding2).item()
                    quality_label = get_translation_quality_label(semantic_score)
                    processor=processor if processor else "unknown"

                    writer.writerow({
                        "Processor": processor,
                        "Semantic Similarity": f"{semantic_score:.4f}",
                        "Quality Label": quality_label,
                    })

                except Exception as e:
                    print(f"Skipped fragment {idx}: {e}")

if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# STEP: 6. Translation-quality similarity check (extended)
# Extended variant of the translation-quality scoring with additional handling.
# DB: objaverse.sqlite32.db
# =============================================================================

#!pip install sentence-transformers
import sqlite3
import csv
import json
import re
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz.fuzz import token_sort_ratio
from difflib import SequenceMatcher

# Load once outside your loop
semantic_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def get_translation_quality_label(score: float) -> str:
    if score >= 0.90:
        return "Excellent"
    elif score >= 0.75:
        return "Good"
    elif score >= 0.55:
        return "Acceptable"
    elif score >= 0.35:
        return "Poor"
    else:
        return "Very Poor / Unrelated"

def clean_and_parse_json(raw: str) -> dict:
    """Attempt to clean and parse a potentially malformed JSON string."""
    text = raw.replace("'", '"').replace("\n", "").replace("\r", "")
    text = re.sub(r'\\(?!["\\/bfnrt])', r'', text).replace("\\", '')
    if "result" in text:
        before = text.split("result")[0]
        after = text.split("result")[1][2:].replace('}', '').replace('"', '')
        text = before + 'result":"' + after + '"'
    if not text.endswith('}'):
        text += '}'
    if not text.startswith('{'):
        text = '{' + text
    return json.loads(text)

def extract_result_text(raw: str) -> str:
    """Extract the 'result' field from a JSON-like string, or return the raw string."""
    if not raw:
        return ""
    try:
        parsed = clean_and_parse_json(raw)
        return parsed.get('result', "")
    except (json.JSONDecodeError, Exception):
        # If it's not JSON at all, treat the raw value as plain text
        return raw.strip().replace("\n", " ").replace("\r", "")

DATABASE = 'objaverse.sqlite32.db'

def main():
    OUTPUT_CSV = 'tfidf-similarity_results.csv'
    retrievalquery = (
        'SELECT aiDescriptionShort, description '
        'FROM object '
        'WHERE description IS NOT NULL AND aiDescriptionShort IS NOT NULL AND "source" LIKE "%Europeana%"'
    )
    print(retrievalquery)

    with sqlite3.connect(DATABASE) as db:
        cur = db.cursor()
        uidlist = cur.execute(retrievalquery).fetchall()

        cur.execute('SELECT COUNT(*) FROM object WHERE name IS NOT NULL AND aiDescriptionShort IS NOT NULL;')
        processed_count = cur.fetchone()[0]
        print(f"Number of datasets already processed: {processed_count}")

        cur.execute('SELECT COUNT(*) FROM object WHERE name IS NOT NULL AND aiDescriptionShort IS NULL;')
        remaining_count = cur.fetchone()[0]
        print(f"Number of datasets to process: {remaining_count}")

    print(f"Number of elements in uidlist: {len(uidlist)}")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["RapidFuzz Similarity", "TF-IDF Cosine Similarity", "SequenceMatcher Similarity", "Processor"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for row in uidlist:
            ai_description_short = row[0]  # aiDescriptionShort (may contain multiple JSON objects)
            description = row[1]           # description (plain text or JSON)

            # --- Extract text1 from description (column index 1) ---
            text1 = extract_result_text(description)
            if not text1:
                print("Skipped: empty text1")
                continue
            print(f"TEXT1: {text1}")

            # --- Handle multiple concatenated JSON objects in aiDescriptionShort ---
            # Re-split properly: wrap each fragment back into a valid JSON object
            raw_parts = ai_description_short.split('}{')
            for idx, part in enumerate(raw_parts):
                if 'distilbert' in part:
                    processor="distilbert"
                if 'Qwen' in part:
                    processor="qwen"
                else:
                    processor="others"
                # Restore braces stripped by the split
                if len(raw_parts) == 1:
                    fragment = part
                elif idx == 0:
                    fragment = part + '}'
                elif idx == len(raw_parts) - 1:
                    fragment = '{' + part
                else:
                    fragment = '{' + part + '}'
                print(processor)
                try:
                    text2 = extract_result_text(fragment)
                    if not text2:
                        print(f"Skipped fragment {idx}: empty text2")
                        continue
                    print(f"TEXT2: {text2}")

                    
                    # --- Option 1: RapidFuzz Similarity ---
                    rapidfuzz_score = token_sort_ratio(text1, text2)
                    rapidfuzz_score = rapidfuzz_score/100
                    # --- Option 2: TF-IDF Cosine Similarity ---
                    corpus = [text1, text2]
                    vectorizer = TfidfVectorizer()
                    tfidf_matrix = vectorizer.fit_transform(corpus)
                    tfidf_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
            
                    # --- Option 3: SequenceMatcher Similarity ---
                    sequence_matcher_score = SequenceMatcher(None, text1, text2).ratio()

                    # Ergebnisse in die CSV schreiben
                    writer.writerow({
                        "RapidFuzz Similarity": f"{rapidfuzz_score}",
                        "TF-IDF Cosine Similarity": f"{tfidf_score:.4f}",
                        "SequenceMatcher Similarity": f"{sequence_matcher_score:.4f}",
                        "Processor": processor
                    })

                except Exception as e:
                    print(f"Skipped fragment {idx}: {e}")

if __name__ == "__main__":
    main()


## 5.1 Identification of Countries

In [ ]:
# =============================================================================
# STEP: 5.1 Report: object counts by country & source
# Counts records per country and source for reporting.
# -> REPORT-country_source_counts-all.csv | DB: objaverse.sqlite32.db
# =============================================================================

import sqlite3
import csv
import os
import time
import re
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

DATABASE = 'Tutorials/objaverse.sqlite32.db'
OUTPUT_CSV = 'REPORT-country_source_counts-all.csv'

europeancountries = [
'"%Albania%" OR edmPlaceLabel LIKE "%Shqipëria%"',
'"%Andorra%"',
'"%Armenia%" OR edmPlaceLabel LIKE "%Հայաստան%"',
'"%Austria%" OR edmPlaceLabel LIKE "%Österreich%"',
'"%Azerbaijan%" OR edmPlaceLabel LIKE "%Azərbaycan%"',
'"%Belarus%" OR edmPlaceLabel LIKE "%Беларусь%"',
'"%Belgium%" OR edmPlaceLabel LIKE "%België%" OR edmPlaceLabel LIKE "%Belgique%" OR edmPlaceLabel LIKE "%Belgien%"',
'"%Bosnia%" OR edmPlaceLabel LIKE "%Босна и Херцеговина%" OR edmPlaceLabel LIKE "%Bosna i Hercegovina%"',
'"%Bulgaria%" OR edmPlaceLabel LIKE "%България%"',
'"%Croatia%" OR edmPlaceLabel LIKE "%Hrvatska%"',
'"%Cyprus%" OR edmPlaceLabel LIKE "%Κύπρος%" OR edmPlaceLabel LIKE "%Kıbrıs%"',
'"%Czech%" OR edmPlaceLabel LIKE "%Česk%"',
'"%Denmark%" OR edmPlaceLabel LIKE "%Danmark%"',
'"%Estonia%" OR edmPlaceLabel LIKE "%Eesti%"',
'"%Finland%" OR edmPlaceLabel LIKE "%Suomi%"',
'"%Macedonia%" OR edmPlaceLabel LIKE "%Северна Македонија%"',
'"%France%" OR edmPlaceLabel LIKE "%fran%"',
'"%Georgia%" OR edmPlaceLabel LIKE "%საქართველო%"',
'"%Germany%" OR edmPlaceLabel LIKE "%Deutschland%"',
'"%Greece%" OR edmPlaceLabel LIKE "%Ελλάς%"',
'"%Hungary%" OR edmPlaceLabel LIKE "%Magyarország%"',
'"%Iceland%" OR edmPlaceLabel LIKE "%Ísland%"',
'"%Ireland%" OR edmPlaceLabel LIKE "%Éire%"',
'"%Italy%" OR edmPlaceLabel LIKE "%Italia%"',
'"%Kosovo%" OR edmPlaceLabel LIKE "%Kosova%"',
'"%Latvia%" OR edmPlaceLabel LIKE "%Latvija%"',
'"%Liechtenstein%"',
'"%Lithuania%" OR edmPlaceLabel LIKE "%Lietuva%"',
'"%Luxembourg%" OR edmPlaceLabel LIKE "%Lëtzebuerg%"',
'"%Malta%"',
'"%Moldova%"',
'"%Monaco%"',
'"%Montenegro%" OR edmPlaceLabel LIKE "%Crna Gora%"',
'"%Netherlands%" OR edmPlaceLabel LIKE "%Nederland%"',
'"%Norway%" OR edmPlaceLabel LIKE "%Norge%"',
'"%Poland%" OR edmPlaceLabel LIKE "%Polska%"',
'"%Portugal%" OR edmPlaceLabel LIKE "%Portuguesa%"',
'"%Romania%" OR edmPlaceLabel LIKE "%România%"',
'"%Russia%" OR edmPlaceLabel LIKE "%Росси%"',
'"%San Marino%"',
'"%Serbia%" OR edmPlaceLabel LIKE "%Србија%"',
'"%Slovakia%" OR edmPlaceLabel LIKE "%Slovensko%"',
'"%Slovenia%" OR edmPlaceLabel LIKE "%Slovenija%"',
'"%Spain%" OR edmPlaceLabel LIKE "%España%"',
'"%Sweden%" OR edmPlaceLabel LIKE "%SVERIGE%"',
'"%Switzerland%" OR edmPlaceLabel LIKE "%SWISS%"',
'"%Turkey%" OR edmPlaceLabel LIKE "%Türkiye%"',
'"%Ukraine%" OR edmPlaceLabel LIKE "%Україна%"',
'"%United Kingdom%" OR edmPlaceLabel LIKE "%UK%" AND edmPlaceLabel NOT LIKE "%UKR%"'
]

def count_entries_by_source_and_country():
    # Connect to the database
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()

    # Prepare the results list
    results = []

    # Iterate through each country
    for country in europeancountries:
        # Query to count entries grouped by source for the current country
        query = f'''
        SELECT source, COUNT(*) 
        FROM object 
        WHERE edmPlaceLabel LIKE {country} 
        GROUP BY source;
        '''
        cur.execute(query)
        rows = cur.fetchall()

        # Append the results to the list
        for row in rows:
            results.append({
                'Country': country.split('"%')[1].split('%"')[0],  # Extract country name for better readability
                'Source': row[0],
                'Count': row[1]
            })

    # Close the database connection
    db.close()

    # Write the results to a CSV file
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['Country', 'Source', 'Count']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    print(f"Results exported to {OUTPUT_CSV}")

# Run the function
count_entries_by_source_and_country()


## 5.2 Categories

In [ ]:
# =============================================================================
# STEP: 5.2 Report: object counts by category & source
# Counts records per concept category and source.
# -> REPORT-categories_source_counts.csv | DB: objaverse.sqlite32.db
# =============================================================================

import os
import time
import spacy
import re
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

DATABASE = 'Tutorials/objaverse.sqlite32.db'
OUTPUT_CSV = 'REPORT-categories_source_counts.csv'

europeanacategories=[
'Building',
'Archaeological Site',
'Cartoon',
'Ceramics',
'Clothing ',
'Costume Accessories',
'Drawing',
'Map',
'Furniture',
'Textile',
'Food',
'Glassware',
'Inscription',
'Jewellery',
'Metalwork',
'Machinery',
'Medal ',
'Memorabilia',
'Mineral',
'Musical Instrument',
'Painting',
'Photograph',
'Postcard',
'Poster',
'Print',
'Sculpture',
'Specimen',
'Tableware',
'Tool',
'Tapestry',
'Toy',
'Weaponry',
'Woodwork',
'Stamp'
]

def count_entries_by_source_and_category():
    # Connect to the database
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()

    # Prepare the results list
    results = []

    # Iterate through each country
    for category in europeanacategories:
        # Query to count entries grouped by source for the current country
        query = f'''
        SELECT source, COUNT(*) 
        FROM object 
        WHERE aiDescriptionEn LIKE "%{category}%" 
        GROUP BY source;
        '''
        cur.execute(query)
        rows = cur.fetchall()

        # Append the results to the list
        for row in rows:
            results.append({
                'Category': category, 
                'Source': row[0],
                'Count': row[1]
            })

    # Close the database connection
    db.close()

    # Write the results to a CSV file
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['Category', 'Source', 'Count']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    print(f"Results exported to {OUTPUT_CSV}")

# Run the function
count_entries_by_source_and_category()


## 5.3 Technology

In [ ]:
# =============================================================================
# STEP: 5.3 Report: technology / material identification
# Uses spaCy NER over English descriptions to extract technology / material mentions.
# reads aiDescriptionEn | Model: spaCy NER | DB: objaverse.sqlite3.db
# =============================================================================

import os
import time
import spacy
import re
import sqlite3
from pathlib import Path
os.chdir(Path().absolute().parent) if Path().absolute().name == "Tutorials" else None

europeancountries=[
"photogramm", 
"photo gram", 
"photo-gram",
"3Dscan",
"3D scan",
"3D-scan",
"hand model",
"hand-model",
"handmodel",
"laser scan",
"laserscan",
"tomography",
"manually model",
"BIM",
"point clouds",
"voxel",
"neural"

]

DATABASE = 'Tutorials/objaverse.sqlite3.db'
dbvalueread="aiDescriptionEn"
uidlist=[]

for i in europeancountries:   
    db = sqlite3.connect(DATABASE)
    cur = db.cursor()
    cur.execute('select count(*) from object WHERE '+dbvalueread+' LIKE "%'+i+'%";')
    numberOfRows = cur.fetchone()[0]
    print ("Number of datasets for "+i+" is: "+str(numberOfRows))

    #for uid in cur.execute(retrievalquery):
    #    uidlist.append(uid)
    
    #print (uidlist)
